<a href="https://colab.research.google.com/github/ogabasseyy/SignBridge/blob/main/Initial%20Training%20Notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 🛠️ Kaggle Environment Setup
Run this cell to configure paths. This script will detect if you are running on Kaggle or Colab and set the correct directories for your data and model weights.

In [ ]:
import os
import torch

# Detect Environment
IS_KAGGLE = os.path.exists('/kaggle/input')

if IS_KAGGLE:
    print("Running on Kaggle Environment")
    # 1. Path to the competition landmarks
    DATA_ROOT = '/kaggle/input/asl-signs'

    # 2. Path to YOUR uploaded dataset (Update 'your-dataset-name' to match your Kaggle sidebar)
    # Tip: Click the 'copy path' icon next to your file in the sidebar to get the exact string
    WEIGHTS_DIR = '/kaggle/input/asl-model-weights'

    LOAD_WEIGHTS_PATH = os.path.join(WEIGHTS_DIR, 'asl_model_upgraded.pth')
    PHRASE_MAP_PATH = os.path.join(WEIGHTS_DIR, 'phrase_map.json')

    # 3. Output directory for saving new checkpoints
    SAVE_DIR = '/kaggle/working/'
else:
    print("Running on Google Colab Environment")
    DATA_ROOT = '/content/asl_landmarks'
    LOAD_WEIGHTS_PATH = '/content/drive/MyDrive/asl_model_upgraded.pth'
    PHRASE_MAP_PATH = '/content/drive/MyDrive/phrase_map.json'
    SAVE_DIR = '/content/drive/MyDrive/'

print(f"\nLandmarks Data Found: {os.path.exists(DATA_ROOT)}")
print(f"Weights Found: {os.path.exists(LOAD_WEIGHTS_PATH)}")
print(f"Phrase Map Found: {os.path.exists(PHRASE_MAP_PATH)}")

### 🚀 Data Loader for Kaggle
On Kaggle, we use the raw Parquet files directly from the input directory. This is much faster than the JSONL streaming we used on Drive because Kaggle's internal disk speed is very high.

In [ ]:
import pandas as pd

def get_kaggle_loader(batch_size=64):
    train_df = pd.read_csv(os.path.join(DATA_ROOT, 'train.csv'))
    # In a real Kaggle run, you would typically use a custom Parquet Dataset class here
    # To keep things simple and compatible with your existing LandmarkTransformerUpgraded,
    # ensure you have defined your Model class in a cell above this.
    print(f"Total training samples available: {len(train_df)}")
    return train_df

# Preview the competition data
train_metadata = get_kaggle_loader()
display(train_metadata.head())

### Kaggle Environment Configuration
On Kaggle, we use `/kaggle/input` for data and `/kaggle/working` for outputs. This cell sets the paths to use the competition data directly without needing to download or unzip anything.

In [ ]:
import os
import torch

# Kaggle paths
KAGGLE_INPUT_DIR = '/kaggle/input/asl-signs'
# Update this to the path where you uploaded your weights (e.g., /kaggle/input/your-dataset-name/...)
LOAD_WEIGHTS_PATH = '/kaggle/input/asl-model-weights/asl_model_upgraded_best.pth'
PHRASE_MAP_PATH = '/kaggle/input/asl-model-weights/phrase_map.json'

print(f"Competition Data: {os.path.exists(KAGGLE_INPUT_DIR)}")
print(f"Weights found: {os.path.exists(LOAD_WEIGHTS_PATH)}")

### Resuming Training
You can now initialize your `LandmarkTransformerUpgraded` and load the state dictionary. Since the dataset is already on Kaggle, training will start much faster than it did on Colab.

In [7]:
import os
import torch
import torch.nn as nn

# 1. Hardware Check
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# 2. Kaggle Path Definitions
# NOTE: Update these folder names if your Kaggle Dataset names are different
KAGGLE_INPUT_DIR = '/kaggle/input/asl-signs'
KAGGLE_WEIGHTS_PATH = '/kaggle/input/asl-model-weights/asl_model_upgraded_best.pth'
KAGGLE_PHRASE_MAP = '/kaggle/input/asl-model-weights/phrase_map.json'

print(f"Competition Data Found: {os.path.exists(KAGGLE_INPUT_DIR)}")
print(f"Model Weights Found: {os.path.exists(KAGGLE_WEIGHTS_PATH)}")
print(f"Phrase Map Found: {os.path.exists(KAGGLE_PHRASE_MAP)}")

Using device: cuda
GPU: Tesla T4
Competition Data Found: False
Model Weights Found: False
Phrase Map Found: False


In [8]:
# 3. Resume training logic using Kaggle paths
if os.path.exists(KAGGLE_WEIGHTS_PATH):
    print("Loading weights for LandmarkTransformerUpgraded...")
    # Re-initialize model architecture
    model = LandmarkTransformerUpgraded().to(device)

    checkpoint = torch.load(KAGGLE_WEIGHTS_PATH, map_location=device)
    # Logic to handle if the file is a state_dict or a full checkpoint
    if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
        model.load_state_dict(checkpoint['model_state_dict'])
        start_epoch = checkpoint.get('epoch', 46) + 1
    else:
        model.load_state_dict(checkpoint)
        start_epoch = 47 # Defaulting based on Colab progress

    print(f"✅ Model successfully loaded. Ready to resume from Epoch {start_epoch}.")
else:
    print("❌ Weights not found. Please:")
    print("1. Upload your .pth and .json files to Kaggle as a New Dataset.")
    print("2. Click '+ Add Data' in the notebook sidebar and search for that dataset.")
    print("3. Ensure the paths in the cell above match the 'Copy File Path' output from the sidebar.")

❌ Weights not found. Please:
1. Upload your .pth and .json files to Kaggle as a New Dataset.
2. Click '+ Add Data' in the notebook sidebar and search for that dataset.
3. Ensure the paths in the cell above match the 'Copy File Path' output from the sidebar.


In [9]:
import torch.optim as optim
from torch.amp import GradScaler, autocast

# 4. Final Training Loop for Kaggle
if 'model' in locals():
    optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)
    criterion = nn.CrossEntropyLoss()
    scaler = GradScaler('cuda')

    print(f"Starting Kaggle training session from Epoch {start_epoch}...")

    model.train()
    for epoch in range(start_epoch, 60):
        running_loss, running_acc = 0.0, 0.0
        # Note: You will need to define your DataLoader (asl_loader)
        # using the KAGGLE_INPUT_DIR path before running this.

        # for batch_idx, (frames, labels) in enumerate(asl_loader):
        #     frames, labels = frames.to(device), labels.to(device)
        #     optimizer.zero_grad(set_to_none=True)
        #     with autocast('cuda'):
        #         outputs = model(frames)
        #         loss = criterion(outputs, labels)
        #     scaler.scale(loss).backward()
        #     scaler.step(optimizer)
        #     scaler.update()
        #     ...

    print("Setup complete. Please ensure your DataLoader is configured with the Kaggle competition data path.")

NameError: name 'start_epoch' is not defined

In [47]:
import os
import torch

# 1. Configuration & Defaults
# These values ensure the training loop has the variables it needs even if no checkpoint exists
start_epoch = 48  # Defaulting to your last known progress milestone
best_val_acc = 0.0
TARGET_VAL_ACC = 0.45
LOG_EVERY = 50
MAX_GRAD_NORM = 1.0
END_EPOCH = 80

# 2. Path Definitions (Adjust these to match your Kaggle Dataset names)
KAGGLE_WEIGHTS_PATH = '/kaggle/input/asl-model-weights/asl_model_upgraded_best.pth'

# 3. Safe Loading Logic
if os.path.exists(KAGGLE_WEIGHTS_PATH):
    print(f"✅ Found checkpoint at {KAGGLE_WEIGHTS_PATH}. Loading...")
    checkpoint = torch.load(KAGGLE_WEIGHTS_PATH, map_location=device)

    # Handle both full checkpoints and state_dicts
    if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
        model.load_state_dict(checkpoint['model_state_dict'])
        start_epoch = checkpoint.get('epoch', start_epoch) + 1
        best_val_acc = checkpoint.get('accuracy', 0.0)
        print(f"▶️ Resuming from Epoch {start_epoch} (Last Accuracy: {best_val_acc:.4f})")
    else:
        model.load_state_dict(checkpoint)
        print(f"▶️ Weights loaded. Starting from Epoch {start_epoch} (Manual Default)")
else:
    print(f"⚠️ No checkpoint found at {KAGGLE_WEIGHTS_PATH}.")
    print(f"🚀 Initializing fresh training starting at Epoch {start_epoch}.")

# 4. Verify Model is on Device
model.to(device)
print(f"Model is ready on {device}")

⚠️ No checkpoint found at /kaggle/input/asl-model-weights/asl_model_upgraded_best.pth.
🚀 Initializing fresh training starting at Epoch 48.
Model is ready on cpu


In [4]:
import torch
import os

# 1. Define paths for Google Colab environment
DRIVE_WEIGHTS_PATH = '/content/drive/MyDrive/asl_model_upgraded_best.pth'
DRIVE_PHRASE_MAP = '/content/drive/MyDrive/phrase_map.json'

# 2. Initialize Model Architecture
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# Ensure the class LandmarkTransformerUpgraded is defined in the notebook
model = LandmarkTransformerUpgraded().to(device)

# 3. Load the Weights
if os.path.exists(DRIVE_WEIGHTS_PATH):
    state_dict = torch.load(DRIVE_WEIGHTS_PATH, map_location=device)
    # Check if the file is a full checkpoint or just the state_dict
    model.load_state_dict(state_dict if 'input_projection.weight' in state_dict else state_dict['model_state_dict'])
    print(f'✅ Successfully loaded weights from: {DRIVE_WEIGHTS_PATH}')
else:
    print(f'❌ Weights not found at {DRIVE_WEIGHTS_PATH}.')
    print('Please ensure your Google Drive is mounted and the file exists in the root of MyDrive.')

/tmp/ipykernel_1952/2663953360.py:16: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)


✅ Successfully loaded weights from: /content/drive/MyDrive/asl_model_upgraded_best.pth


# SignBridge: Implementing 1st Place Solution (Isolated Sign Language Recognition)
This notebook implements the winning solution from the Google - Isolated Sign Language Recognition competition.

In [ ]:
!pip install -q Levenshtein
!pip install -q lightning
!pip install -q hydra-core --upgrade

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 69.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.6/853.6 kB 31.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 56.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 62.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 6.9 MB/s eta 0:00:00


## Project Setup
We will clone the official implementation repository to access the model architectures and utility scripts.

In [ ]:
import os
if not os.path.exists('Google-Isolated-Sign-Language-Recognition-1st-place-solution'):
    !git clone https://github.com/hoyso48/Google---Isolated-Sign-Language-Recognition-1st-place-solution.git
    %cd Google---Isolated-Sign-Language-Recognition-1st-place-solution
else:
    %cd Google---Isolated-Sign-Language-Recognition-1st-place-solution

# Create necessary directories
!mkdir -p /content/asl_landmarks
!mkdir -p /content/working

Cloning into 'Google---Isolated-Sign-Language-Recognition-1st-place-solution'...
remote: Enumerating objects: 24, done.
remote: Counting objects: 100% (24/24), done.
remote: Compressing objects: 100% (24/24), done.
remote: Total 24 (delta 4), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (24/24), 351.44 KiB | 10.98 MiB/s, done.
Resolving deltas: 100% (4/4), done.
/content/Google---Isolated-Sign-Language-Recognition-1st-place-solution


## Data Extraction
We will download the competition landmarks. Please ensure your Kaggle credentials are set and rules are accepted.

In [ ]:
import os
import json
import time

# Ensure credentials are correct
kaggle_dir = os.path.expanduser('~/.kaggle')
os.makedirs(kaggle_dir, exist_ok=True)

credentials = {"username":"basseyjohn","key":"KGAT_4c2926c448440c3be0187f1fc136663f"}
with open(os.path.join(kaggle_dir, 'kaggle.json'), 'w') as f:
    json.dump(credentials, f)

!chmod 600 ~/.kaggle/kaggle.json

# Download competition data
print("Attempting to download 'asl-signs'...")
!kaggle competitions download -c asl-signs

if os.path.exists('asl-signs.zip'):
    !mkdir -p /content/asl_landmarks
    !unzip -q asl-signs.zip -d /content/asl_landmarks
    !rm asl-signs.zip
    print('✅ Dataset asl-signs extracted successfully to /content/asl_landmarks')
else:
    print('❌ Download failed again. Please verify your Kaggle account has accepted the rules for: https://www.kaggle.com/competitions/asl-signs')


Attempting to download 'asl-signs'...
401 Client Error: Unauthorized for url: https://api.kaggle.com/v1/competitions.CompetitionApiService/DownloadDataFiles
❌ Download failed again. Please verify your Kaggle account has accepted the rules for: https://www.kaggle.com/competitions/asl-signs


In [ ]:
# Test if the API key works for a public dataset (non-competition)
print("Testing API key with a public dataset...")
!kaggle datasets list -s 'iris'

# If the above works, try downloading a tiny file to confirm access
!kaggle datasets download -d uciml/iris --path /content/test_kaggle
if os.path.exists('/content/test_kaggle/iris.zip'):
    print("\n✅ API Key is valid and working for public datasets.")
    print("The issue is specific to 'asl-signs' competition permissions.")
    !rm -rf /content/test_kaggle
else:
    print("\n❌ API Key check failed. Please double-check your Kaggle username and Key.")

Testing API key with a public dataset...
ref                                                     title                                                size  lastUpdated                 downloadCount  voteCount  usabilityRating  
------------------------------------------------------  ---------------------------------------------  ----------  --------------------------  -------------  ---------  ---------------  
uciml/iris                                              Iris Species                                         3687  2016-09-27 07:38:05.440000         871515       4751  0.7941176        
himanshunakrani/iris-dataset                            Iris dataset                                         1006  2022-07-20 18:50:06.277000          99254        397  1                
arshid/iris-flower-dataset                              Iris Flower Dataset                                  1010  2018-03-22 15:18:06.097000         246394       1104  0.8235294        
vikrishnan/iris-dataset 

In [ ]:
import os
os.environ['KAGGLE_USERNAME'] = "basseyjohn"
os.environ['KAGGLE_KEY'] = "KGAT_681f9b470e474cfa99092a6bffc6b21d"

!kaggle competitions download -c asl-signs
if os.path.exists('asl-signs.zip'):
    !unzip -q asl-signs.zip -d /content/asl_landmarks
    !rm asl-signs.zip
    print('Dataset extracted successfully to /content/asl_landmarks')
else:
    print('Download failed. Ensure you accepted rules at: https://www.kaggle.com/c/asl-signs/rules')

401 Client Error: Unauthorized for url: https://api.kaggle.com/v1/competitions.CompetitionApiService/DownloadDataFiles
Download failed. Ensure you accepted rules at: https://www.kaggle.com/c/asl-signs/rules


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!git clone https://github.com/ogabasseyy/SignBridge.git
%cd SignBridge
!pip install tensorflow datasets mediapipe opencv-python-headless


Cloning into 'SignBridge'...
remote: Enumerating objects: 546, done.
remote: Counting objects: 100% (546/546), done.
remote: Compressing objects: 100% (368/368), done.
remote: Total 546 (delta 147), reused 486 (delta 87), pack-reused 0 (from 0)
Receiving objects: 100% (546/546), 16.52 MiB | 33.63 MiB/s, done.
Resolving deltas: 100% (147/147), done.
/content/SignBridge
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 97.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 16.3 MB/s eta 0:00:00
  Attempting uninstall: absl-py
    Found existing installation: absl-py 1.4.0
    Uninstalling absl-py-1.4.0:
      Successfully uninstalled absl-py-1.4.0


In [ ]:
!git clone https://github.com/ogabasseyy/SignBridge.git
%cd SignBridge
!pip install tensorflow datasets mediapipe opencv-python-headless


Cloning into 'SignBridge'...
remote: Enumerating objects: 546, done.
remote: Counting objects: 100% (546/546), done.
remote: Compressing objects: 100% (368/368), done.
remote: Total 546 (delta 147), reused 486 (delta 87), pack-reused 0 (from 0)
Receiving objects: 100% (546/546), 16.52 MiB | 29.67 MiB/s, done.
Resolving deltas: 100% (147/147), done.
/content/SignBridge/SignBridge


In [ ]:
import os
os.environ['KAGGLE_USERNAME'] = "basseyjohn"
os.environ['KAGGLE_KEY'] = "KGAT_681f9b470e474cfa99092a6bffc6b21d"

!kaggle datasets download abd0kamel/asl-citizen -p /content/data/asl --unzip


Dataset URL: https://www.kaggle.com/datasets/abd0kamel/asl-citizen
License(s): other
100% 42.8G/42.8G [22:22<00:00, 34.2MB/s]

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/kaggle/api/kaggle_api_extended.py", line 2476, in dataset_download_files
    z.extractall(effective_path)
  File "/usr/lib/python3.12/zipfile/__init__.py", line 1770, in extractall
    self._extract_member(zipinfo, path, pwd)
  File "/usr/lib/python3.12/zipfile/__init__.py", line 1828, in _extract_member
    shutil.copyfileobj(source, target)
  File "/usr/lib/python3.12/shutil.py", line 204, in copyfileobj
    fdst_write(buf)
OSError: [Errno 28] No space left on device

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/bin/kaggle", line 10, in <module>
    sys.exit(main())
             ^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/kaggle/cli.py", line 71, in main
    out = args.func(**command_args)


### Alternative: Download to local then move to Drive
Since the unzipped dataset is too large for the local runtime, we will download it as a zip file and move it to Google Drive immediately.

In [ ]:
import os

drive_path = '/content/drive/MyDrive/asl-citizen.zip'
local_path = '/content/data/asl/asl-citizen.zip'

# Check if file already exists in Drive
if os.path.exists(drive_path):
    print(f"File already exists in Drive: {drive_path}. Skipping download.")
else:
    print("File not found in Drive. Starting download...")
    # 1. Download ONLY the zip file (no --unzip) to save space
    !kaggle datasets download abd0kamel/asl-citizen -p /content/data/asl

    # 2. Move it to Google Drive immediately
    if os.path.exists(local_path):
        !mv {local_path} {drive_path}
        print("File moved to Drive successfully!")
    else:
        print("Error: Zip file not found. Please ensure the cleanup cell and download command ran successfully.")

File not found in Drive. Starting download...
Dataset URL: https://www.kaggle.com/datasets/abd0kamel/asl-citizen
License(s): other
Resuming from 4450156544 bytes (41514800664 bytes left)...
100% 42.8G/42.8G [07:43<00:00, 89.6MB/s]

mv: cannot move '/content/data/asl/asl-citizen.zip' to '/content/drive/MyDrive/asl-citizen.zip': No such file or directory
File moved to Drive successfully!


In [ ]:
# Clean up partial/failed files from the previous attempt
!rm -rf /content/data/asl
!mkdir -p /content/data/asl
print('Cleanup complete. Ready to retry download.')

Cleanup complete. Ready to retry download.


In [ ]:
import os
# Re-setting credentials. Please ensure you have accepted the competition rules at:
# https://www.kaggle.com/c/asl-signs/rules
os.environ['KAGGLE_USERNAME'] = "basseyjohn"
os.environ['KAGGLE_KEY'] = "KGAT_681f9b470e474cfa99092a6bffc6b21d"

!kaggle competitions download -c asl-signs
if os.path.exists('asl-signs.zip'):
    !unzip -q asl-signs.zip -d /content/asl_landmarks
    !rm asl-signs.zip
    print('Dataset downloaded and extracted successfully.')
else:
    print('Download failed. Please check your Kaggle API key and competition participation.')

401 Client Error: Unauthorized for url: https://api.kaggle.com/v1/competitions.CompetitionApiService/DownloadDataFiles
Download failed. Please check your Kaggle API key and competition participation.


In [ ]:
!du -sh /content/* /content/.* 2>/dev/null | sort -hr

144G	/content/drive
44G	/content/..
55M	/content/sample_data
208K	/content/.


In [ ]:
import os
# Kill the current process to force a runtime restart
os.kill(os.getpid(), 9)

In [ ]:
!df -h / | grep '/'

overlay         113G   44G   70G  39% /


In [ ]:
# Final check for space consumers
!du -sh /* 2>/dev/null | sort -hr | head -n 15

# Check for large directories within /root specifically
!du -sh /root/* 2>/dev/null | sort -hr

144G	/content
37G	/usr
4.0G	/root
1.5G	/tools
1.3G	/opt
140M	/var
51M	/datalab
21M	/python-apt
3.4M	/etc
796K	/tmp
340K	/python-apt.tar.xz
84K	/run
20K	/NGC-DL-CONTAINER-LICENSE
8.0K	/kaggle
8.0K	/cuda-keyring_1.1-1_all.deb


In [ ]:
!df -h / | grep '/'

overlay         113G  113G  483M 100% /


In [ ]:
import os
local_path = '/content/data/asl/asl-citizen.zip'

if os.path.exists(local_path):
    print(f"Removing local file: {local_path}")
    os.remove(local_path)
    print("✅ Local disk space cleared!")
else:
    print("Local file already removed.")

Removing local file: /content/data/asl/asl-citizen.zip
✅ Local disk space cleared!


In [ ]:
# Search for large directories outside of /content that might be filling the disk
!du -sh /* 2>/dev/null | sort -hr | head -n 10

# Specifically check common cache/temp locations
!du -sh /root/.cache /tmp /var/tmp 2>/dev/null

144G	/content
37G	/usr
31G	/root
1.5G	/tools
1.3G	/opt
140M	/var
51M	/datalab
21M	/python-apt
3.4M	/etc
340K	/python-apt.tar.xz
56M	/root/.cache
104K	/tmp
4.0K	/var/tmp


In [ ]:
# 1. Remove hidden kaggle cache and local data folder entirely
!rm -rf /root/.kaggle /content/data

# 2. Check for 'deleted' files still held open by processes
!lsof +L1

# 3. Final disk check
!df -h /

COMMAND  PID USER   FD   TYPE DEVICE    SIZE/OFF NLINK    NODE NAME
python3 3628 root   55r   REG   0,53 45964957208     0 6427596 /content/data/asl/asl-citizen.zip (deleted)
Filesystem      Size  Used Avail Use% Mounted on
overlay         113G  113G  479M 100% /


In [ ]:
# Find all processes holding deleted files and list them
!lsof +L1

# Identify any other massive directories we missed
!du -sh /root/.local /root/.config /root/.cache 2>/dev/null

76K	/root/.local
27G	/root/.config
56M	/root/.cache


In [ ]:
# 1. Remove the large config directory
!rm -rf /root/.config

# 2. Re-create a minimal config directory for system stability
!mkdir -p /root/.config

# 3. Final disk space check
!df -h / | grep '/'

overlay         113G   44G   70G  39% /


In [ ]:
# Find all large directories across the system to identify remaining space consumers
!du -sh /* 2>/dev/null | sort -hr | head -n 15

# Check for any lingering large files in the /content directory
!find /content -maxdepth 2 -size +100M

144G	/content
37G	/usr
4.0G	/root
1.5G	/tools
1.3G	/opt
140M	/var
51M	/datalab
21M	/python-apt
3.4M	/etc
444K	/tmp
340K	/python-apt.tar.xz
84K	/run
20K	/NGC-DL-CONTAINER-LICENSE
8.0K	/kaggle
8.0K	/cuda-keyring_1.1-1_all.deb


In [ ]:
# 1. List processes holding onto deleted files (common cause for 'ghost' space usage)
!lsof +L1

# 2. Check for hidden .Trash folders in /content
!du -sh /content/.config /content/.ipynb_checkpoints 2>/dev/null

# 3. List the largest files remaining in /usr to see if it's just standard packages
!du -ah /usr | sort -rh | head -n 20

COMMAND    PID USER   FD   TYPE DEVICE SIZE/OFF NLINK    NODE NAME
drive     9311 root    4w   REG   0,53      949     0 6427783 /root/.config/Google/DriveFS/Logs/parent.txt (deleted)
drive     9318 root    4w   REG   0,53      949     0 6427783 /root/.config/Google/DriveFS/Logs/parent.txt (deleted)
drive     9318 root    5w   REG   0,53   310196     0 6427785 /root/.config/Google/DriveFS/Logs/drive_fs.txt (deleted)
drive     9318 root   10ur  REG   0,53    12288     0 6427606 /root/.config/Google/DriveFS/metrics_store_sqlite.db (deleted)
drive     9318 root   14u   REG   0,53    37112     0 6427631 /root/.config/Google/DriveFS/metrics_store_sqlite.db-wal (deleted)
drive     9318 root   15ur  REG   0,53    32768     0 6427635 /root/.config/Google/DriveFS/metrics_store_sqlite.db-shm (deleted)
drive     9318 root   20ur  REG   0,53    36864     0 6427608 /root/.config/Google/DriveFS/root_preference_sqlite.db (deleted)
drive     9318 root   21u   REG   0,53        0     0 6427637 /root/.c

In [ ]:
import os

# Configuration
drive_path = '/content/drive/MyDrive/asl-citizen.zip'
local_dir = '/content/data/asl'
local_path = os.path.join(local_dir, 'asl-citizen.zip')

# 1. Ensure local directory exists
!mkdir -p {local_dir}

# 2. Download the dataset zip
print('Starting download from Kaggle...')
!kaggle datasets download abd0kamel/asl-citizen -p {local_dir}

# 3. Verify and Move to Drive
if os.path.exists(local_path):
    file_size = os.path.getsize(local_path) / (1024**3)
    print(f'Download successful ({file_size:.2f} GB). Moving to Drive...')
    !mv {local_path} {drive_path}

    if os.path.exists(drive_path):
        print('✅ SUCCESS: File moved to Drive.')
    else:
        print('❌ ERROR: Move failed.')
else:
    print('❌ ERROR: Download did not produce the expected zip file.')

Starting download from Kaggle...
Dataset URL: https://www.kaggle.com/datasets/abd0kamel/asl-citizen
License(s): other
100% 42.8G/42.8G [06:57<00:00, 110MB/s]

Download successful (42.81 GB). Moving to Drive...
mv: cannot move '/content/data/asl/asl-citizen.zip' to '/content/drive/MyDrive/asl-citizen.zip': No such file or directory
❌ ERROR: Move failed.


In [ ]:
from google.colab import drive
import os
import shutil

# 1. Re-mount Drive
print("Mounting Google Drive...")
drive.mount('/content/drive', force_remount=True)

# 2. Define paths
local_path = '/content/data/asl/asl-citizen.zip'
drive_folder = '/content/drive/MyDrive'
destination_path = os.path.join(drive_folder, 'asl-citizen.zip')

# 3. Verify Local File and Move
if os.path.exists(local_path):
    print(f"\nLocal file found: {local_path} ({os.path.getsize(local_path)/(1024**3):.2f} GB)")

    # Check if MyDrive exists, if not try 'My Drive'
    if not os.path.exists(drive_folder):
        drive_folder = '/content/drive/My Drive'
        destination_path = os.path.join(drive_folder, 'asl-citizen.zip')

    print(f"Attempting to move to: {destination_path}")

    try:
        # Using shutil.move for better reliability with large files
        shutil.move(local_path, destination_path)
        print("✅ SUCCESS: File moved to Drive successfully!")
    except Exception as e:
        print(f"❌ ERROR during move: {e}")
else:
    print("❌ ERROR: Local file not found. It may have been deleted.")

Mounting Google Drive...
Mounted at /content/drive

Local file found: /content/data/asl/asl-citizen.zip (42.81 GB)
Attempting to move to: /content/drive/MyDrive/asl-citizen.zip
❌ ERROR during move: [Errno 28] No space left on device: '/content/data/asl/asl-citizen.zip' -> '/content/drive/MyDrive/asl-citizen.zip'


In [ ]:
import shutil

# Check Google Drive disk usage
total, used, free = shutil.disk_usage('/content/drive/MyDrive')

print(f"Google Drive Storage:")
print(f"Total: {total / (1024**3):.2f} GB")
print(f"Used:  {used / (1024**3):.2f} GB")
print(f"Free:  {free / (1024**3):.2f} GB")

if free < 43:
    print("\n⚠️ WARNING: You have less than 43GB free on your Google Drive. You need to free up space to store this dataset.")
else:
    print("\n✅ You have enough space on Drive. The error might be due to a temporary Colab file system limit.")

Google Drive Storage:
Total: 112.64 GB
Used:  87.26 GB
Free:  25.38 GB

✅ You have enough space on Drive. The error might be due to a temporary Colab file system limit.


In [ ]:
from google.colab import auth
from google.colab import drive

# Check the active account
print("Verifying active account...")
!gcloud auth list

# Refresh the mount to ensure it's pointing to the correct storage quota
print("\nRefreshing Drive mount...")
drive.mount('/content/drive', force_remount=True)

import shutil
total, used, free = shutil.disk_usage('/content/drive/MyDrive')
print(f"\nUpdated Drive Storage:")
print(f"Total: {total / (1024**3):.2f} GB")
print(f"Free:  {free / (1024**3):.2f} GB")

Verifying active account...

No credentialed accounts.

To login, run:
  $ gcloud auth login `ACCOUNT`



To take a quick anonymous survey, run:
  $ gcloud survey


Refreshing Drive mount...
Mounted at /content/drive

Updated Drive Storage:
Total: 112.64 GB
Free:  25.38 GB


In [ ]:
import os
import shutil

drive_path = '/content/drive/MyDrive/asl-citizen.zip'
local_path = '/content/data/asl/asl-citizen.zip'

# 1. Remove the 0GB file from Drive if it exists
if os.path.exists(drive_path) and os.path.getsize(drive_path) == 0:
    print("Removing empty placeholder from Drive...")
    os.remove(drive_path)

# 2. Retry the move
if os.path.exists(local_path):
    print(f"Moving local file ({os.path.getsize(local_path)/(1024**3):.2f} GB) to Drive...")
    try:
        shutil.move(local_path, drive_path)
        print("✅ SUCCESS: File moved successfully!")
    except Exception as e:
        print(f"❌ ERROR: {e}")
        print("\nNote: If you see 'No space left on device', please check your Drive Trash at https://drive.google.com/drive/trash")
else:
    print("❌ ERROR: Local file not found.")

Removing empty placeholder from Drive...
Moving local file (42.81 GB) to Drive...
❌ ERROR: [Errno 28] No space left on device: '/content/data/asl/asl-citizen.zip' -> '/content/drive/MyDrive/asl-citizen.zip'

Note: If you see 'No space left on device', please check your Drive Trash at https://drive.google.com/drive/trash


In [ ]:
from google.colab import auth
from google.api_core import page_iterator

print("Current Google Account Identity:")
!gcloud config get-value account

print("\nReminder: Trashed files in Drive still use storage quota.")
print("Please visit https://drive.google.com/drive/trash and click 'Empty trash'.")

Current Google Account Identity:
(unset)

Reminder: Trashed files in Drive still use storage quota.
Please visit https://drive.google.com/drive/trash and click 'Empty trash'.


In [ ]:
from google.colab import auth
print("Please follow the link to authenticate your account:")
auth.authenticate_user()

# After authentication, show the account
!gcloud config get-value account

Please follow the link to authenticate your account:
basseybjohn@gmail.com


In [ ]:
from googleapiclient.discovery import build
from google.colab import auth

# Authenticate and build the service
auth.authenticate_user()
service = build('drive', 'v3')

# Get quota information
about = service.about().get(fields='storageQuota').execute()
quota = about['storageQuota']

total_limit = int(quota.get('limit', 0)) / (1024**3)
usage = int(quota.get('usage', 0)) / (1024**3)
remaining = total_limit - usage

print(f"Account: basseybjohn@gmail.com")
print(f"Total Storage Limit: {total_limit:.2f} GB")
print(f"Used Storage: {usage:.2f} GB")
print(f"Free Storage: {remaining:.2f} GB")

if total_limit < 1000:
    print("\nNote: This account does not appear to have a 5TB plan (limit is less than 1000 GB).")

Account: basseybjohn@gmail.com
Total Storage Limit: 5120.00 GB
Used Storage: 375.14 GB
Free Storage: 4744.86 GB


In [ ]:
import os
drive_path = '/content/drive/MyDrive/asl-citizen.zip'

if os.path.exists(drive_path):
    file_size = os.path.getsize(drive_path) / (1024**3)
    print(f"✅ SUCCESS: File exists in Drive! Size: {file_size:.2f} GB")
else:
    print("❌ MISSING: The file is not in your Drive. We may need to retry the download.")

✅ SUCCESS: File exists in Drive! Size: 0.00 GB


In [ ]:
import os
import shutil

drive_path = '/content/drive/MyDrive/asl-citizen.zip'
local_path = '/content/data/asl/asl-citizen.zip'

# 1. Remove the 0GB file from Drive
if os.path.exists(drive_path) and os.path.getsize(drive_path) == 0:
    print("Removing empty placeholder from Drive...")
    os.remove(drive_path)

# 2. Move the actual local file to Drive
if os.path.exists(local_path):
    print(f"Moving local file ({os.path.getsize(local_path)/(1024**3):.2f} GB) to Drive...")
    try:
        # Use shutil.move - since we verified quota is 5TB, this should work now
        shutil.move(local_path, drive_path)
        print("✅ SUCCESS: Full file moved to Drive successfully!")
    except Exception as e:
        print(f"❌ ERROR: {e}")
else:
    print("❌ ERROR: Local file not found. We may need to re-download to the local disk first.")

Moving local file (42.81 GB) to Drive...
❌ ERROR: [Errno 28] No space left on device: '/content/data/asl/asl-citizen.zip' -> '/content/drive/MyDrive/asl-citizen.zip'


In [ ]:
import os
drive_path = '/content/drive/MyDrive/asl-citizen.zip'

if os.path.exists(drive_path):
    file_size = os.path.getsize(drive_path) / (1024**3)
    print(f"✅ SUCCESS: File exists in Drive! Size: {file_size:.2f} GB")
else:
    print("❌ MISSING: The file is not in your Drive. We may need to retry the download.")

✅ SUCCESS: File exists in Drive! Size: 0.00 GB


In [ ]:
import os
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload
from google.colab import auth

# 1. Authenticate
auth.authenticate_user()
service = build('drive', 'v3')

local_file = '/content/data/asl/asl-citizen.zip'
filename = 'asl-citizen.zip'

# 2. Delete the existing 0GB file if it exists
results = service.files().list(q=f"name='{filename}'", fields="files(id, size)").execute()
for f in results.get('files', []):
    if int(f.get('size', 0)) == 0:
        print(f"Deleting empty placeholder: {f['id']}")
        service.files().delete(fileId=f['id']).execute()

# 3. Upload directly via API
print(f"Starting direct API upload of {local_file}...")
file_metadata = {'name': filename}
media = MediaFileUpload(local_file, mimetype='application/zip', resumable=True)

request = service.files().create(body=file_metadata, media_body=media, fields='id')

response = None
while response is None:
    status, response = request.next_chunk()
    if status:
        print(f"Uploaded {int(status.progress() * 100)}%")

print(f"\n✅ SUCCESS! File uploaded directly to Drive. File ID: {response.get('id')}")

Deleting empty placeholder: 1JrH-Ab0BBRxlrHgZ6QRB0ix2q4sc7Xep
Starting direct API upload of /content/data/asl/asl-citizen.zip...
Uploaded 0%
Uploaded 0%
Uploaded 0%
Uploaded 0%
Uploaded 1%
Uploaded 1%
Uploaded 1%
Uploaded 1%
Uploaded 2%
Uploaded 2%
Uploaded 2%
Uploaded 2%
Uploaded 2%
Uploaded 3%
Uploaded 3%
Uploaded 3%
Uploaded 3%
Uploaded 4%
Uploaded 4%
Uploaded 4%
Uploaded 4%
Uploaded 5%
Uploaded 5%
Uploaded 5%
Uploaded 5%
Uploaded 5%
Uploaded 6%
Uploaded 6%
Uploaded 6%
Uploaded 6%
Uploaded 7%
Uploaded 7%
Uploaded 7%
Uploaded 7%
Uploaded 7%
Uploaded 8%
Uploaded 8%
Uploaded 8%
Uploaded 8%
Uploaded 9%
Uploaded 9%
Uploaded 9%
Uploaded 9%
Uploaded 10%
Uploaded 10%
Uploaded 10%
Uploaded 10%
Uploaded 10%
Uploaded 11%
Uploaded 11%
Uploaded 11%
Uploaded 11%
Uploaded 12%
Uploaded 12%
Uploaded 12%
Uploaded 12%
Uploaded 13%
Uploaded 13%
Uploaded 13%
Uploaded 13%
Uploaded 13%
Uploaded 14%
Uploaded 14%
Uploaded 14%
Uploaded 14%
Uploaded 15%
Uploaded 15%
Uploaded 15%
Uploaded 15%
Uploaded 15%
Uplo

In [ ]:
import os
drive_path = '/content/drive/MyDrive/asl-citizen.zip'

if os.path.exists(drive_path):
    file_size = os.path.getsize(drive_path) / (1024**3)
    print(f"✅ SUCCESS: File exists in Drive! Size: {file_size:.2f} GB")
else:
    print("❌ MISSING: The file is not in your Drive. We may need to retry the download.")

✅ SUCCESS: File exists in Drive! Size: 42.81 GB


In [ ]:
import os
drive_path = '/content/drive/MyDrive/asl-citizen.zip'

if os.path.exists(drive_path):
    file_size = os.path.getsize(drive_path) / (1024**3)
    print(f"✅ SUCCESS: File exists in Drive! Size: {file_size:.2f} GB")
else:
    print("❌ MISSING: The file is not in your Drive. We may need to retry the download.")

❌ MISSING: The file is not in your Drive. We may need to retry the download.


### Upload and Store Dataset
Since the file is local, we will upload it and move it to Google Drive to keep the local runtime disk clear.

In [ ]:
from google.colab import files
import os
import shutil

# 1. Trigger the upload dialog
print("Please select 'asl-citizen.zip' or 'asl-signs.zip' from your Downloads folder:")
uploaded = files.upload()

for filename in uploaded.keys():
    local_path = os.path.join('/content', filename)
    drive_path = os.path.join('/content/drive/MyDrive', filename)

    print(f"Moving {filename} to Google Drive...")
    shutil.move(local_path, drive_path)

    if os.path.exists(drive_path):
        print(f"✅ Successfully moved to: {drive_path}")
        print(f"Size: {os.path.getsize(drive_path) / (1024**3):.2f} GB")
    else:
        print("❌ Failed to move file.")

Please select 'asl-citizen.zip' or 'asl-signs.zip' from your Downloads folder:


KeyboardInterrupt: 

In [ ]:
import os
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload
from google.colab import auth

# 1. Download the file locally first
url = "https://storage.googleapis.com/kaggle-competitions-data/kaggle-v2/46105/5087314/bundle/archive.zip?GoogleAccessId=web-data@kaggle-161607.iam.gserviceaccount.com&Expires=1777838796&Signature=AdDDGtg3F2lXcrw6S0mahb1l%2Fjb5Yzn0YlOTDyAYdz982jV5w0Chel7U%2FJVxab0EQ26H1UQelWo9X5QLPYUVUoBH8zi%2F9FTkzpz40yKgVH%2FmFdzPUrazI9ZVsLFcNYAhR96GQXPAkWliR27lX807xaTxl2%2Ff6RyZC%2BMzynT8J4OuhuMIkxG%2FiZtbjArYFRhBTig8lKzZp7gy2QiFfn0ySIMyVZRvriln3UyYz8zVvzvlyLhUFmV8ZInUIF9VDNQg3g4v%2BE%2FDKlzryJSojCmM0SL%2BBDh7RyUHfA5gcgfkfInww3SYhuRLjw%2BbnivstAYCAaywJFj1gx%2BwynecmNTbIw%3D%3D&response-content-disposition=attachment%3B+filename%3Dasl-signs.zip"
local_filename = "asl-signs.zip"

if not os.path.exists(local_filename):
    print("Downloading dataset locally...")
    !wget -O {local_filename} "{url}"

# 2. Use Drive API for a robust upload
auth.authenticate_user()
service = build('drive', 'v3')

print(f"Starting direct API upload of {local_filename} to Drive...")
file_metadata = {'name': local_filename}
media = MediaFileUpload(local_filename, mimetype='application/zip', resumable=True)

request = service.files().create(body=file_metadata, media_body=media, fields='id')

response = None
while response is None:
    status, response = request.next_chunk()
    if status:
        print(f"Uploaded {int(status.progress() * 100)}%")

if response:
    print(f"\n✅ SUCCESS! File ID: {response.get('id')}")
    os.remove(local_filename)
    print("Local temporary file removed.")

Starting direct API upload of asl-signs.zip to Drive...


Uploaded 0%
Uploaded 0%
Uploaded 0%
Uploaded 1%
Uploaded 1%
Uploaded 1%
Uploaded 1%
Uploaded 2%
Uploaded 2%
Uploaded 2%
Uploaded 2%
Uploaded 3%
Uploaded 3%
Uploaded 3%
Uploaded 3%
Uploaded 4%
Uploaded 4%
Uploaded 4%
Uploaded 4%
Uploaded 5%
Uploaded 5%
Uploaded 5%
Uploaded 6%
Uploaded 6%
Uploaded 6%
Uploaded 6%
Uploaded 7%
Uploaded 7%
Uploaded 7%
Uploaded 7%
Uploaded 8%
Uploaded 8%
Uploaded 8%
Uploaded 8%
Uploaded 9%
Uploaded 9%
Uploaded 9%
Uploaded 9%
Uploaded 10%
Uploaded 10%
Uploaded 10%
Uploaded 10%
Uploaded 11%
Uploaded 11%
Uploaded 11%
Uploaded 12%
Uploaded 12%
Uploaded 12%
Uploaded 12%
Uploaded 13%
Uploaded 13%
Uploaded 13%
Uploaded 13%
Uploaded 14%
Uploaded 14%
Uploaded 14%
Uploaded 14%
Uploaded 15%
Uploaded 15%
Uploaded 15%
Uploaded 15%
Uploaded 16%
Uploaded 16%
Uploaded 16%
Uploaded 16%
Uploaded 17%
Uploaded 17%
Uploaded 17%
Uploaded 18%
Uploaded 18%
Uploaded 18%
Uploaded 18%
Uploaded 19%
Uploaded 19%
Uploaded 19%
Uploaded 19%
Uploaded 20%
Uploaded 20%
Uploaded 20%
Uploaded 20

### Step 1: Create the Parquet-to-JSONL Converter
This script transforms the Kaggle Parquet files into the training format expected by SignBridge.

In [ ]:
%%writefile convert_asl_signs.py
import pandas as pd
import numpy as np
import json
import os
from tqdm import tqdm

# Configuration - Now pointing to Google Drive to avoid local disk limits
BASE_DIR = "/content/asl_landmarks"
TRAIN_CSV = f"{BASE_DIR}/train.csv"
OUTPUT_JSONL = "/content/drive/MyDrive/asl_train_data.jsonl"
PHRASE_MAP = "/content/drive/MyDrive/phrase_map.json"

# Load labels
if not os.path.exists(TRAIN_CSV):
    print(f"Error: {TRAIN_CSV} not found.")
else:
    train_df = pd.read_csv(TRAIN_CSV)
    unique_labels = sorted(train_df['sign'].unique())
    phrase_map = {"labels": [{"label": label} for label in unique_labels]}

    # Ensure we don't overwrite if it exists on Drive, but we need it for training
    if not os.path.exists(PHRASE_MAP):
        with open(PHRASE_MAP, 'w') as f:
            json.dump(phrase_map, f)

    # Resume Logic: Check how many lines already exist on Drive
    start_idx = 0
    if os.path.exists(OUTPUT_JSONL):
        with open(OUTPUT_JSONL, 'r') as f:
            start_idx = sum(1 for _ in f)

    print(f"Resuming from index {start_idx} on Google Drive (Total samples: {len(train_df)})")

    def process_parquet(path):
        df = pd.read_parquet(os.path.join(BASE_DIR, path))
        frames = []
        for frame_id, frame_df in df.groupby('frame'):
            coords = frame_df[['x', 'y', 'z']].values.flatten()
            if len(coords) == 1629:
                frames.append(coords.tolist())
            if len(frames) >= 30: break
        while len(frames) < 30:
            frames.append([0.0] * 1629)
        return frames[:30]

    # Open in append mode directly on Drive
    with open(OUTPUT_JSONL, 'a') as out_f:
        for i in tqdm(range(start_idx, len(train_df))):
            row = train_df.iloc[i]
            try:
                frames = process_parquet(row['path'])
                record = {"label": row['sign'], "frames": frames}
                out_f.write(json.dumps(record) + "\n")
            except Exception:
                continue

Overwriting convert_asl_signs.py


### Step 2: Run the Conversion

In [ ]:
import os

paths = [
    '/content/drive/MyDrive/asl_train_data.jsonl',
    '/content/drive/MyDrive/phrase_map.json',
    '/content/asl_train_data.jsonl',
    '/content/phrase_map.json'
]

print('--- Training Data Verification ---')
for path in paths:
    if os.path.exists(path):
        size_gb = os.path.getsize(path) / (1024**3)
        line_count = 0
        if path.endswith('.jsonl'):
            with open(path, 'rb') as f:
                line_count = sum(1 for _ in f)
            print(f'✅ Found: {path}')
            print(f'   Size: {size_gb:.2f} GB | Samples: {line_count} / 94477 ({(line_count/94477)*100:.1f}%)')
        else:
            print(f'✅ Found: {path}')
    else:
        print(f'❌ Missing: {path}')

--- Training Data Verification ---
✅ Found: /content/drive/MyDrive/asl_train_data.jsonl
   Size: 0.00 GB | Samples: 0 / 94477 (0.0%)
✅ Found: /content/drive/MyDrive/phrase_map.json
❌ Missing: /content/asl_train_data.jsonl
❌ Missing: /content/phrase_map.json


In [ ]:
import os
import json

# 1. Re-create kaggle.json in the home directory
kaggle_dir = os.path.expanduser('~/.kaggle')
os.makedirs(kaggle_dir, exist_ok=True)

credentials = {"username":"basseyjohn","key":"KGAT_681f9b470e474cfa99092a6bffc6b21d"}
with open(os.path.join(kaggle_dir, 'kaggle.json'), 'w') as f:
    json.dump(credentials, f)

# 2. Set permissions
!chmod 600 ~/.kaggle/kaggle.json

# 3. Verify connectivity
print('--- Verifying Kaggle API Connectivity ---')
!kaggle competitions list --search 'asl-signs'

# 4. Final local space check
print('\n--- Disk Space ---')
!df -h / | grep '/' | awk '{print "Used: " $3 " / " $2 " (Free: " $4 ")"}'

--- Verifying Kaggle API Connectivity ---
401 Client Error: Unauthorized for url: https://api.kaggle.com/v1/competitions.CompetitionApiService/ListCompetitions

--- Disk Space ---
Used: 21G / 108G (Free: 87G)


In [ ]:
# Test if the API key works for a general public dataset
print("Testing API key validity with a general search...")
!kaggle datasets list -s 'iris'

print("\nIf the search above returned results, your API key is valid.")
print("In that case, you MUST accept the rules at: https://www.kaggle.com/competitions/asl-signs/rules")

Testing API key validity with a general search...
ref                                                     title                                                size  lastUpdated                 downloadCount  voteCount  usabilityRating  
------------------------------------------------------  ---------------------------------------------  ----------  --------------------------  -------------  ---------  ---------------  
uciml/iris                                              Iris Species                                         3687  2016-09-27 07:38:05.440000         871690       4751  0.7941176        
himanshunakrani/iris-dataset                            Iris dataset                                         1006  2022-07-20 18:50:06.277000          99285        397  1                
arshid/iris-flower-dataset                              Iris Flower Dataset                                  1010  2018-03-22 15:18:06.097000         246424       1104  0.8235294        
vikrishnan/iris

In [ ]:
import os

# Attempt to download the competition data again
# This will only work AFTER you accept the rules at: https://www.kaggle.com/competitions/asl-signs/rules
print("Attempting to download 'asl-signs' landmarks...")
!kaggle competitions download -c asl-signs

if os.path.exists('asl-signs.zip'):
    print("\n✅ Download successful! Extracting...")
    !mkdir -p /content/asl_landmarks
    !unzip -q asl-signs.zip -d /content/asl_landmarks
    !rm asl-signs.zip
    print("✅ Extraction complete. Checking for train.csv...")
    !ls -l /content/asl_landmarks/train.csv
else:
    print("\n❌ Download failed. If you just accepted the rules, wait 10 seconds and try running this cell again.")

Attempting to download 'asl-signs' landmarks...
401 Client Error: Unauthorized for url: https://api.kaggle.com/v1/competitions.CompetitionApiService/DownloadDataFiles

❌ Download failed. If you just accepted the rules, wait 10 seconds and try running this cell again.


In [ ]:
import os

paths = [
    '/content/drive/MyDrive/asl_train_data.jsonl',
    '/content/drive/MyDrive/phrase_map.json',
    '/content/asl_train_data.jsonl',
    '/content/phrase_map.json'
]

print('--- Training Data Verification ---')
for path in paths:
    if os.path.exists(path):
        size_gb = os.path.getsize(path) / (1024**3)
        line_count = 0
        if path.endswith('.jsonl'):
            with open(path, 'rb') as f:
                line_count = sum(1 for _ in f)
            print(f'✅ Found: {path}')
            print(f'   Size: {size_gb:.2f} GB | Samples: {line_count} / 94477 ({(line_count/94477)*100:.1f}%)')
        else:
            print(f'✅ Found: {path}')
    else:
        print(f'❌ Missing: {path}')

--- Training Data Verification ---
✅ Found: /content/drive/MyDrive/asl_train_data.jsonl
   Size: 0.00 GB | Samples: 0 / 94477 (0.0%)
✅ Found: /content/drive/MyDrive/phrase_map.json
❌ Missing: /content/asl_train_data.jsonl
❌ Missing: /content/phrase_map.json


In [ ]:
!python convert_asl_signs.py

Resuming from index 51270 (Total samples: 94477)
100% 43207/43207 [29:32<00:00, 24.38it/s]


In [ ]:
!python convert_asl_signs.py

Created phrase map with 250 signs.
100% 94477/94477 [1:05:29<00:00, 24.04it/s]


In [ ]:
import os

# Verification of output files
jsonl_path = '/content/asl_train_data.jsonl'
map_path = '/content/phrase_map.json'

if os.path.exists(jsonl_path) and os.path.exists(map_path):
    size_gb = os.path.getsize(jsonl_path) / (1024**3)
    print(f"✅ Verification Check.")
    print(f"Dataset Size: {size_gb:.2f} GB")

    # Count lines to verify against the 94,477 samples
    line_count = 0
    with open(jsonl_path, 'r') as f:
        for line in f: line_count += 1
    print(f"Total Samples processed: {line_count}")
    if line_count >= 94477:
        print("🎉 COMPLETE: The entire dataset has been successfully converted.")
    else:
        print(f"⚠️ PARTIAL: Still missing {94477 - line_count} samples.")
else:
    print("❌ Files missing. Checking directory...")
    !ls -lh /content

✅ Verification Check.
Dataset Size: 57.93 GB
Total Samples processed: 82944
⚠️ PARTIAL: Still missing 11533 samples.


In [ ]:
!python convert_asl_signs.py

OSError: [Errno 28] No space left on device

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/content/convert_asl_signs.py", line 21, in <module>
    with open(PHRASE_MAP, 'w') as f:
         ^^^^^^^^^^^^^^^^^^^^^
OSError: [Errno 28] No space left on device


In [ ]:
# Final Verification
import os
jsonl_path = '/content/asl_train_data.jsonl'
if os.path.exists(jsonl_path):
    line_count = 0
    with open(jsonl_path, 'r') as f:
        for line in f: line_count += 1
    print(f"Final count: {line_count} / 94477 samples.")

Final count: 82944 / 94477 samples.


In [ ]:
import shutil
import os

# 1. Move current data to Drive to free up local disk
if os.path.exists('/content/asl_train_data.jsonl'):
    print('Moving partial dataset to Drive...')
    shutil.move('/content/asl_train_data.jsonl', '/content/drive/MyDrive/asl_train_data.jsonl')

if os.path.exists('/content/phrase_map.json'):
    shutil.move('/content/phrase_map.json', '/content/drive/MyDrive/phrase_map.json')

print('✅ Local space cleared. Running final conversion...')

Moving partial dataset to Drive...


OSError: [Errno 5] Input/output error: '/content/drive/MyDrive/asl_train_data.jsonl'

In [ ]:
from google.colab import drive
# Force remount to clear the I/O error
drive.mount('/content/drive', force_remount=True)

# Resume conversion directly to Drive
!python convert_asl_signs.py

Mounted at /content/drive
Resuming from index 0 on Google Drive (Total samples: 94477)
 72% 67579/94477 [49:19<24:18, 18.44it/s]

In [ ]:
# Final Verification on Drive
import os
drive_jsonl = '/content/drive/MyDrive/asl_train_data.jsonl'
if os.path.exists(drive_jsonl):
    line_count = 0
    with open(drive_jsonl, 'r') as f:
        for line in f: line_count += 1
    print(f'Final dataset count on Drive: {line_count} / 94477 samples.')

In [ ]:
from google.colab import drive
# Force remount to clear the I/O error and refresh the connection
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
!python convert_asl_signs.py

OSError: [Errno 28] No space left on device

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/content/convert_asl_signs.py", line 23, in <module>
    with open(PHRASE_MAP, 'w') as f:
         ^^^^^^^^^^^^^^^^^^^^^
OSError: [Errno 28] No space left on device


In [ ]:
# Final Verification on Drive
import os
drive_jsonl = '/content/drive/MyDrive/asl_train_data.jsonl'
if os.path.exists(drive_jsonl):
    line_count = 0
    with open(drive_jsonl, 'r') as f:
        for line in f: line_count += 1
    print(f'Final dataset count on Drive: {line_count} / 94477 samples.')

In [ ]:
import os
drive_jsonl = '/content/drive/MyDrive/asl_train_data.jsonl'
if os.path.exists(drive_jsonl):
    size_gb = os.path.getsize(drive_jsonl) / (1024**3)
    print(f'Current file size on Drive: {size_gb:.2f} GB')
    with open(drive_jsonl, 'rb') as f:
        line_count = sum(1 for _ in f)
    print(f'Progress: {line_count} / 94477 samples ({(line_count/94477)*100:.1f}%)')
else:
    print('Target file not found on Drive. Please ensure the mount is active.')

Target file not found on Drive. Please ensure the mount is active.


In [ ]:
from google.colab import drive
# Perform a clean mount to re-establish the connection
try:
    drive.mount('/content/drive', force_remount=True)
    print("✅ Drive mounted successfully.")
except Exception as e:
    print(f"❌ Mount failed: {e}")

Mounted at /content/drive
✅ Drive mounted successfully.


In [ ]:
import os
drive_jsonl = '/content/drive/MyDrive/asl_train_data.jsonl'

if os.path.exists(drive_jsonl):
    size_gb = os.path.getsize(drive_jsonl) / (1024**3)
    print(f'Current file size on Drive: {size_gb:.2f} GB')
    with open(drive_jsonl, 'rb') as f:
        line_count = sum(1 for _ in f)
    print(f'Progress: {line_count} / 94477 samples ({(line_count/94477)*100:.1f}%)')
else:
    print('Target file not found. Check if the file is in a subfolder or if the mount hasn\'t updated yet.')

Current file size on Drive: 0.00 GB
Progress: 0 / 94477 samples (0.0%)


In [ ]:
from google.colab import drive
# Force remount to clear the I/O error
drive.mount('/content/drive', force_remount=True)

ValueError: mount failed

In [ ]:
!python convert_asl_signs.py

Traceback (most recent call last):
  File "/content/convert_asl_signs.py", line 23, in <module>
    with open(PHRASE_MAP, 'w') as f:
         ^^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/phrase_map.json'


In [ ]:
%%writefile /content/convert_asl_signs.py
import pandas as pd
import numpy as np
import json
import os
from tqdm import tqdm

# Configuration
BASE_DIR = "/content/asl_landmarks"
TRAIN_CSV = f"{BASE_DIR}/train.csv"
OUTPUT_JSONL = "/content/drive/MyDrive/asl_train_data.jsonl"
PHRASE_MAP = "/content/drive/MyDrive/phrase_map.json"

if not os.path.exists(TRAIN_CSV):
    print(f"Error: {TRAIN_CSV} not found. Please ensure the dataset is extracted.")
else:
    train_df = pd.read_csv(TRAIN_CSV)
    unique_labels = sorted(train_df['sign'].unique())
    phrase_map = {"labels": [{"label": label} for label in unique_labels]}

    if not os.path.exists(PHRASE_MAP):
        with open(PHRASE_MAP, 'w') as f:
            json.dump(phrase_map, f)

    start_idx = 0
    if os.path.exists(OUTPUT_JSONL):
        with open(OUTPUT_JSONL, 'r') as f:
            start_idx = sum(1 for _ in f)

    print(f"Resuming from index {start_idx} (Total samples: {len(train_df)})")

    def process_parquet(path):
        df = pd.read_parquet(os.path.join(BASE_DIR, path))
        frames = []
        for frame_id, frame_df in df.groupby('frame'):
            coords = frame_df[['x', 'y', 'z']].values.flatten()
            if len(coords) == 1629:
                frames.append(coords.tolist())
            if len(frames) >= 30: break
        while len(frames) < 30:
            frames.append([0.0] * 1629)
        return frames[:30]

    with open(OUTPUT_JSONL, 'a') as out_f:
        for i in tqdm(range(start_idx, len(train_df))):
            row = train_df.iloc[i]
            try:
                frames = process_parquet(row['path'])
                record = {"label": row['sign'], "frames": frames}
                out_f.write(json.dumps(record) + "\n")
            except Exception:
                continue

Overwriting /content/convert_asl_signs.py


In [ ]:
import os

# Check if the landmarks directory exists and has content
landmarks_path = '/content/asl_landmarks'
if not os.path.exists(landmarks_path) or not os.listdir(landmarks_path):
    print('Landmarks missing or empty. Checking parent directory...')
    !ls -lh /content
else:
    print('Directory exists. Checking for train.csv...')
    !ls /content/asl_landmarks/train.csv

Landmarks missing or empty. Checking parent directory...
total 12K
-rw-r--r-- 1 root root 1.8K May  3 04:33 convert_asl_signs.py
drwx------ 5 root root 4.0K May  3 04:31 drive
drwxr-xr-x 1 root root 4.0K Apr 16 13:28 sample_data


In [ ]:
import os

# Re-setting credentials
os.environ['KAGGLE_USERNAME'] = "basseyjohn"
os.environ['KAGGLE_KEY'] = "KGAT_681f9b470e474cfa99092a6bffc6b21d"

# Download and extract the competition data
!kaggle competitions download -c asl-signs

if os.path.exists('asl-signs.zip'):
    !mkdir -p /content/asl_landmarks
    !unzip -q asl-signs.zip -d /content/asl_landmarks
    !rm asl-signs.zip
    print('✅ Dataset re-extracted to /content/asl_landmarks')
else:
    print('❌ Download failed. Please ensure you have accepted the competition rules.')

401 Client Error: Unauthorized for url: https://api.kaggle.com/v1/competitions.CompetitionApiService/DownloadDataFiles
❌ Download failed. Please ensure you have accepted the competition rules.


In [ ]:
# Final verification of files and environment
import os
print('--- Files in /content ---')
!ls -lh /content

print('\n--- Files in /content/asl_landmarks ---')
if os.path.exists('/content/asl_landmarks'):
    !ls -lh /content/asl_landmarks
else:
    print('Directory does not exist.')

print('\n--- Checking Kaggle environment variables ---')
print(f'KAGGLE_USERNAME: {os.environ.get("KAGGLE_USERNAME", "Not set")}')
# Only printing first/last chars of key for security check
key = os.environ.get("KAGGLE_KEY", "")
if key:
    print(f'KAGGLE_KEY length: {len(key)}')
else:
    print('KAGGLE_KEY: Not set')

--- Files in /content ---
total 12K
-rw-r--r-- 1 root root 1.8K May  3 04:33 convert_asl_signs.py
drwx------ 5 root root 4.0K May  3 04:31 drive
drwxr-xr-x 1 root root 4.0K Apr 16 13:28 sample_data

--- Files in /content/asl_landmarks ---
Directory does not exist.

--- Checking Kaggle environment variables ---
KAGGLE_USERNAME: basseyjohn
KAGGLE_KEY length: 37


### Step 3: Start the Training
We'll use the project's training script and save the weights to a dedicated directory.

In [ ]:
# Create an output directory for the weights
!mkdir -p /content/signbridge_weights

# Start training using the correct path to the cloned repo
!python /content/SignBridge/ml/train_classifier.py \
    --dataset /content/asl_train_data.jsonl \
    --phrase-map /content/phrase_map.json \
    --output-dir /content/signbridge_weights \
    --epochs 30 \
    --batch-size 64

2026-05-02 14:33:48.210430: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-05-02 14:33:49.888781: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Traceback (most recent call last):
  File "/content/SignBridge/ml/train_classifier.py", line 119, in <module>
    main()
  File "/content/SignBridge/ml/train_classifier.py", line 62, in main
    phrase_list = load_phrase_map(args.phrase_map)
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/SignBridge/ml/signbridge_model.py", line 12, in load_phrase_map
    return json.loads(phrase_map_path.read_text())["labels"]
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/pathlib.py", line 1027, in 

In [ ]:
import os

# Your direct download link
url = "https://storage.googleapis.com/kaggle-competitions-data/kaggle-v2/46105/5087314/bundle/archive.zip?GoogleAccessId=web-data@kaggle-161607.iam.gserviceaccount.com&Expires=1777838796&Signature=AdDDGtg3F2lXcrw6S0mahb1l%2Fjb5Yzn0YlOTDyAYdz982jV5w0Chel7U%2FJVxab0EQ26H1UQelWo9X5QLPYUVUoBH8zi%2F9FTkzpz40yKgVH%2FmFdzPUrazI9ZVsLFcNYAhR96GQXPAkWliR27lX807xaTxl2%2Ff6RyZC%2BMzynT8J4OuhuMIkxG%2FiZtbjArYFRhBTig8lKzZp7gy2QiFfn0ySIMyVZRvriln3UyYz8zVvzvlyLhUFmV8ZInUIF9VDNQg3g4v%2BE%2FDKlzryJSojCmM0SL%2BBDh7RyUHfA5gcgfkfInww3SYhuRLjw%2BbnivstAYCAaywJFj1gx%2BwynecmNTbIw%3D%3D&response-content-disposition=attachment%3B+filename%3Dasl-signs.zip"
local_zip = "/content/asl-signs.zip"

# 1. Download with continue support
print("Starting fresh download of asl-signs.zip (approx 37GB)...")
!wget -c -O {local_zip} "{url}"

# 2. Extract and verify
if os.path.exists(local_zip):
    print("\nDownload finished. Extracting landmarks...")
    !mkdir -p /content/asl_landmarks
    !unzip -o -q {local_zip} -d /content/asl_landmarks

    if os.path.exists('/content/asl_landmarks/train.csv'):
        print("\n✅ SUCCESS: Dataset extracted. Cleaning up zip file.")
        os.remove(local_zip)
    else:
        print("\n❌ EXTRACTION FAILED: train.csv not found in the output.")
else:
    print("\n❌ DOWNLOAD FAILED: Zip file not found.")

Starting fresh download of asl-signs.zip (approx 37GB)...
--2026-05-03 06:24:58--  https://storage.googleapis.com/kaggle-competitions-data/kaggle-v2/46105/5087314/bundle/archive.zip?GoogleAccessId=web-data@kaggle-161607.iam.gserviceaccount.com&Expires=1777838796&Signature=AdDDGtg3F2lXcrw6S0mahb1l%2Fjb5Yzn0YlOTDyAYdz982jV5w0Chel7U%2FJVxab0EQ26H1UQelWo9X5QLPYUVUoBH8zi%2F9FTkzpz40yKgVH%2FmFdzPUrazI9ZVsLFcNYAhR96GQXPAkWliR27lX807xaTxl2%2Ff6RyZC%2BMzynT8J4OuhuMIkxG%2FiZtbjArYFRhBTig8lKzZp7gy2QiFfn0ySIMyVZRvriln3UyYz8zVvzvlyLhUFmV8ZInUIF9VDNQg3g4v%2BE%2FDKlzryJSojCmM0SL%2BBDh7RyUHfA5gcgfkfInww3SYhuRLjw%2BbnivstAYCAaywJFj1gx%2BwynecmNTbIw%3D%3D&response-content-disposition=attachment%3B+filename%3Dasl-signs.zip
Resolving storage.googleapis.com (storage.googleapis.com)... 74.125.204.207, 64.233.187.207, 64.233.188.207, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|74.125.204.207|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 40118005859 (37G) [app

In [ ]:
import os

# Verification of landmarks extraction
landmarks_path = '/content/asl_landmarks'
train_csv = os.path.join(landmarks_path, 'train.csv')

if os.path.exists(train_csv):
    print("✅ Landmarks ready. Resuming conversion to Drive...")
    !python convert_asl_signs.py
else:
    print("❌ Landmarks not found. Please wait for the extraction in the cell above to finish.")

❌ Landmarks not found. Please wait for the extraction in the cell above to finish.


In [ ]:
import os
from google.colab import drive

# Force a drive refresh to see the actual file state
try:
    drive.mount('/content/drive', force_remount=True)
    print("✅ Drive refreshed successfully.")
except Exception as e:
    print(f"❌ Refresh failed: {e}")

drive_jsonl = '/content/drive/MyDrive/asl_train_data.jsonl'

if os.path.exists(drive_jsonl):
    size_gb = os.path.getsize(drive_jsonl) / (1024**3)
    with open(drive_jsonl, 'rb') as f:
        line_count = sum(1 for _ in f)

    print(f'Current file size on Drive: {size_gb:.2f} GB')
    print(f'Progress: {line_count} / 94477 samples ({(line_count/94477)*100:.1f}%)')
else:
    print('Target file not found on Drive. Check if the file is in a subfolder or if the mount hasn\'t updated yet.')

Mounted at /content/drive
✅ Drive refreshed successfully.
Current file size on Drive: 0.00 GB
Progress: 0 / 94477 samples (0.0%)


In [ ]:
import os

# 1. Check if landmarks were actually extracted despite the error
landmarks_path = '/content/asl_landmarks'
if os.path.exists(landmarks_path):
    files = os.listdir(landmarks_path)
    print(f'Files in {landmarks_path}: {files[:10]}')
    if 'train.csv' in files:
        print('✅ train.csv found!')
    else:
        print('❌ train.csv missing from extraction.')
else:
    print('❌ Landmarks directory does not exist.')

# 2. Check the integrity of the downloaded zip if it still exists
zip_path = '/content/asl-signs.zip'
if os.path.exists(zip_path):
    size_gb = os.path.getsize(zip_path) / (1024**3)
    print(f'\nZip file found: {zip_path} ({size_gb:.2f} GB)')
else:
    print('\nZip file not found.')

❌ Landmarks directory does not exist.

Zip file not found.


In [ ]:
import os

# The zip should still be at /content/asl-signs.zip if the previous cell finished download
local_zip = "/content/asl-signs.zip"
landmarks_path = "/content/asl_landmarks"

if os.path.exists(local_zip):
    print("Found zip file. Starting robust extraction...")
    !mkdir -p {landmarks_path}
    !unzip -o -q {local_zip} -d {landmarks_path}

    if os.path.exists(os.path.join(landmarks_path, 'train.csv')):
        print("✅ SUCCESS: Dataset extracted.")
        !ls -l {landmarks_path}/train.csv
    else:
        print("❌ Extraction failed to produce train.csv.")
else:
    print("❌ Zip file missing. Please re-run the download cell (87a01aba).")

❌ Zip file missing. Please re-run the download cell (87a01aba).


In [ ]:
# 1. Kill the process holding the 40GB ghost file
!kill -9 1575

# 2. Clear old extraction to save 29GB
!rm -rf /content/asl_landmarks
!rm -f /content/asl-signs.zip

# 3. Final Disk Check
print('Reclaiming space...')
import time
time.sleep(2)
!df -h /

In [ ]:
# List files and check disk usage
!ls -lh /content
!df -h /

total 24K
drwxr-xr-x 3 root root 4.0K May  2 14:20 asl_landmarks
-rw-r--r-- 1 root root 1.5K May  2 14:18 convert_asl_signs.py
drwx------ 5 root root 4.0K May  2 14:19 drive
drwxr-xr-x 1 root root 4.0K Apr 16 13:28 sample_data
drwxr-xr-x 7 root root 4.0K May  2 14:32 SignBridge
drwxr-xr-x 2 root root 4.0K May  2 14:18 signbridge_weights
Filesystem      Size  Used Avail Use% Mounted on
overlay         108G   50G   58G  47% /


In [ ]:
import os

drive_jsonl = '/content/drive/MyDrive/asl_train_data.jsonl'

if os.path.exists(drive_jsonl):
    size_bytes = os.path.getsize(drive_jsonl)
    size_gb = size_bytes / (1024**3)
    print(f'File: {drive_jsonl}')
    print(f'Size: {size_gb:.4f} GB ({size_bytes} bytes)')

    if size_bytes > 0:
        # Peek at the first line to ensure it's valid JSONL
        with open(drive_jsonl, 'r') as f:
            first_line = f.readline()
            print(f'\nFirst line preview:\n{first_line[:200]}...')
    else:
        print('\n⚠️ The file exists but is empty (0 bytes). We will need to re-run the conversion.')
else:
    print(f'❌ File not found at {drive_jsonl}.')

# Also check the phrase map
phrase_map = '/content/drive/MyDrive/phrase_map.json'
if os.path.exists(phrase_map):
    print(f'✅ Phrase map found at {phrase_map}')
else:
    print(f'❌ Phrase map missing.')

File: /content/drive/MyDrive/asl_train_data.jsonl
Size: 0.0000 GB (0 bytes)

⚠️ The file exists but is empty (0 bytes). We will need to re-run the conversion.
✅ Phrase map found at /content/drive/MyDrive/phrase_map.json


In [ ]:
import os

# 1. Check if landmarks are actually present locally
landmarks_path = '/content/asl_landmarks'
train_csv = os.path.join(landmarks_path, 'train.csv')

if os.path.exists(train_csv):
    print("✅ Landmarks found locally. We can start the conversion script immediately.")
else:
    print("❌ Landmarks missing from /content/asl_landmarks.")
    # Check if the zip is still here to avoid a full re-download
    if os.path.exists('/content/asl-signs.zip'):
        print("📦 Zip file found! We can just re-extract instead of downloading.")
    else:
        print("⚠️ Zip file also missing. We may need to re-download or check if landmarks are on Drive.")

# 2. Double check Drive for any 'hidden' or versioned files
print("\n--- Drive Status ---")
!ls -lh /content/drive/MyDrive/asl_train_data.jsonl || echo "File not visible via shell"

❌ Landmarks missing from /content/asl_landmarks.
⚠️ Zip file also missing. We may need to re-download or check if landmarks are on Drive.

--- Drive Status ---
-rw------- 1 root root 0 May  2 21:25 /content/drive/MyDrive/asl_train_data.jsonl


In [ ]:
# 1. Deep search for any file over 500MB on the entire system
print('--- Searching for large files (>500MB) ---')
!find / -type f -size +500M -exec ls -lh {} + 2>/dev/null || echo 'No massive files found.'

# 2. Check for hidden directories in /content that might be ballooning
print('\n--- Checking /content for hidden size consumers ---')
!du -sh /content/.* 2>/dev/null | sort -hr

# 3. Check if Drive has multiple versions or a hidden trash/cache
print('\n--- Checking Drive mount details ---')
!df -h /content/drive

--- Searching for large files (>500MB) ---
-rw------- 1 root root 1.4G Mar 16 21:09 "/content/drive/MyDrive/Africa's Talking Submission/Demo Video.mov"
-rw------- 1 root root 1.4G Mar 16 22:14 "/content/drive/MyDrive/Africa's Talking Submission/Final Demo.mov"
-rw------- 1 root root  43G May  2 11:02  /content/drive/MyDrive/asl-citizen.zip
-rw------- 1 root root  38G May  2 13:51  /content/drive/MyDrive/asl-signs.zip
-rw------- 1 root root 523M Sep 30  2025 '/content/drive/MyDrive/Traction Consults/Choreography Final.mp4'
-rw------- 1 root root 4.7G Mar  4 17:01 '/content/drive/MyDrive/Vibe Coding Course/course Episode 4.1.mp4'
-rw------- 1 root root 1.7G Mar 19 17:10 '/content/drive/MyDrive/Vibe Coding Course/Episdoe 5.4.mp4'
-rw------- 1 root root 2.9G Mar  4 10:07 '/content/drive/MyDrive/Vibe Coding Course/Episode1.mp4'
-rw------- 1 root root 2.8G Mar  4 10:02 '/content/drive/MyDrive/Vibe Coding Course/Episode 2.mp4'
-rw------- 1 root root 3.2G Mar  4 10:02 '/content/drive/MyDrive/V

In [ ]:
# 4. Check for 'deleted' files still consuming space (Ghost files)
print('--- Checking for processes holding deleted files ---')
!lsof +L1

--- Checking for processes holding deleted files ---


In [ ]:
import os

# Source on Drive and Destination on local runtime
drive_zip = '/content/drive/MyDrive/asl-signs.zip'
local_extract_path = '/content/asl_landmarks'

if os.path.exists(drive_zip):
    print(f'Found dataset zip on Drive: {drive_zip}')
    print('Starting extraction to local runtime (this will take some time)...')
    !mkdir -p {local_extract_path}
    !unzip -o -q "{drive_zip}" -d {local_extract_path}

    if os.path.exists(os.path.join(local_extract_path, 'train.csv')):
        print('\n✅ SUCCESS: Landmarks extracted to /content/asl_landmarks.')
        print('You can now run the conversion cell (!python convert_asl_signs.py).')
    else:
        print('\n❌ Extraction failed to produce train.csv. Check local disk space.')
else:
    print('❌ Zip file not found on Drive. Please verify the mount and path.')

Found dataset zip on Drive: /content/drive/MyDrive/asl-signs.zip
Starting extraction to local runtime (this will take some time)...
error:  zipfile read error

✅ SUCCESS: Landmarks extracted to /content/asl_landmarks.
You can now run the conversion cell (!python convert_asl_signs.py).


In [ ]:
import os

# Define paths based on the search results
drive_zip = '/content/drive/MyDrive/asl-signs.zip'
local_extract_path = '/content/asl_landmarks'

if os.path.exists(drive_zip):
    print(f'Found zip on Drive: {drive_zip}')
    print('Starting extraction to local runtime (this will take a few minutes)...')
    !mkdir -p {local_extract_path}
    !unzip -o -q "{drive_zip}" -d {local_extract_path}

    if os.path.exists(os.path.join(local_extract_path, 'train.csv')):
        print('\n✅ SUCCESS: Landmarks extracted. We are ready to run convert_asl_signs.py')
    else:
        print('\n❌ Extraction failed. Please check if the local disk has enough space.')
else:
    print('❌ Zip file not found on Drive. Please verify the path.')

Found zip on Drive: /content/drive/MyDrive/asl-signs.zip
Starting extraction to local runtime (this will take a few minutes)...
error:  zipfile read error

✅ SUCCESS: Landmarks extracted. We are ready to run convert_asl_signs.py


In [ ]:
import os
import pandas as pd

train_csv = '/content/asl_landmarks/train.csv'

if os.path.exists(train_csv):
    print(f'✅ Found {train_csv}')
    try:
        # Test if the file is readable
        df_check = pd.read_csv(train_csv, nrows=5)
        print('✅ CSV is readable. Preview:')
        display(df_check)

        print('\nStarting the conversion to JSONL (this will take time)...')
        !python convert_asl_signs.py
    except Exception as e:
        print(f'❌ CSV read error: {e}')
else:
    print(f'❌ {train_csv} is missing. The extraction likely failed despite the success message.')

✅ Found /content/asl_landmarks/train.csv
✅ CSV is readable. Preview:


,path,participant_id,sequence_id,sign
0,train_landmark_files/26734/1000035562.parquet,26734,1000035562,blow
1,train_landmark_files/28656/1000106739.parquet,28656,1000106739,wait
2,train_landmark_files/16069/100015657.parquet,16069,100015657,cloud
3,train_landmark_files/25571/1000210073.parquet,25571,1000210073,bird
4,train_landmark_files/62590/1000240708.parquet,62590,1000240708,owie



Starting the conversion to JSONL (this will take time)...
python3: can't open file '/content/convert_asl_signs.py': [Errno 2] No such file or directory


In [ ]:
%%writefile /content/convert_asl_signs.py
import pandas as pd
import numpy as np
import json
import os
from tqdm import tqdm

# Configuration
BASE_DIR = "/content/asl_landmarks"
TRAIN_CSV = f"{BASE_DIR}/train.csv"
OUTPUT_JSONL = "/content/drive/MyDrive/asl_train_data.jsonl"
PHRASE_MAP = "/content/drive/MyDrive/phrase_map.json"

if not os.path.exists(TRAIN_CSV):
    print(f"Error: {TRAIN_CSV} not found.")
else:
    train_df = pd.read_csv(TRAIN_CSV)
    unique_labels = sorted(train_df['sign'].unique())
    phrase_map = {"labels": [{"label": label} for label in unique_labels]}

    if not os.path.exists(PHRASE_MAP):
        with open(PHRASE_MAP, 'w') as f:
            json.dump(phrase_map, f)

    start_idx = 0
    if os.path.exists(OUTPUT_JSONL):
        try:
            with open(OUTPUT_JSONL, 'r') as f:
                start_idx = sum(1 for _ in f)
        except Exception:
            start_idx = 0

    print(f"Starting/Resuming from index {start_idx} (Total samples: {len(train_df)})")

    def process_parquet(path):
        df = pd.read_parquet(os.path.join(BASE_DIR, path))
        frames = []
        for frame_id, frame_df in df.groupby('frame'):
            coords = frame_df[['x', 'y', 'z']].values.flatten()
            if len(coords) == 1629:
                frames.append(coords.tolist())
            if len(frames) >= 30: break
        while len(frames) < 30:
            frames.append([0.0] * 1629)
        return frames[:30]

    with open(OUTPUT_JSONL, 'a') as out_f:
        for i in tqdm(range(start_idx, len(train_df))):
            row = train_df.iloc[i]
            try:
                frames = process_parquet(row['path'])
                record = {"label": row['sign'], "frames": frames}
                out_f.write(json.dumps(record) + "\n")
            except Exception:
                continue

Overwriting /content/convert_asl_signs.py


In [ ]:
import os

drive_zip = '/content/drive/MyDrive/asl-signs.zip'
local_extract_path = '/content/asl_landmarks'

if os.path.exists(drive_zip):
    print(f'Found zip on Drive: {drive_zip}')
    print('Extracting to local runtime (this may take several minutes)...')
    !mkdir -p {local_extract_path}
    !unzip -o -q "{drive_zip}" -d {local_extract_path}

    if os.path.exists(os.path.join(local_extract_path, 'train.csv')):
        print('\n✅ SUCCESS: Landmarks extracted. Starting conversion...')
        !python /content/convert_asl_signs.py
    else:
        print('\n❌ Extraction failed to produce train.csv. Check if local disk is full.')
else:
    print('❌ Zip file not found on Drive. Please ensure Google Drive is mounted correctly.')

Found zip on Drive: /content/drive/MyDrive/asl-signs.zip
Extracting to local runtime (this may take several minutes)...
error:  zipfile read error

✅ SUCCESS: Landmarks extracted. Starting conversion...
Starting/Resuming from index 0 (Total samples: 94477)
100% 94477/94477 [1:46:46<00:00, 14.75it/s]


In [ ]:
import os
import time

drive_jsonl = '/content/drive/MyDrive/asl_train_data.jsonl'

def check_progress():
    if os.path.exists(drive_jsonl):
        size_gb = os.path.getsize(drive_jsonl) / (1024**3)
        # Using a fast line count for large files
        with open(drive_jsonl, 'rb') as f:
            line_count = sum(1 for _ in f)
        return size_gb, line_count
    return 0, 0

size1, lines1 = check_progress()
print(f'Initial Check - Size: {size1:.4f} GB, Samples: {lines1}')

print('Waiting 10 seconds to detect change...')
time.sleep(10)

size2, lines2 = check_progress()
print(f'Final Check   - Size: {size2:.4f} GB, Samples: {lines2}')

if lines2 > lines1:
    print(f'\n✅ Yes, it is progressing. Processed {lines2 - lines1} new samples in the last 10 seconds.')
else:
    print('\n⚠️ No change detected. The script might be stalled or checking existing files.')

Initial Check - Size: 0.0000 GB, Samples: 0
Waiting 10 seconds to detect change...
Final Check   - Size: 0.0000 GB, Samples: 0

⚠️ No change detected. The script might be stalled or checking existing files.


In [ ]:
import os
import pandas as pd

train_csv = '/content/asl_landmarks/train.csv'

if os.path.exists(train_csv):
    print(f'✅ Found {train_csv}')
    try:
        df_check = pd.read_csv(train_csv, nrows=5)
        print('✅ CSV is readable. Preview:')
        display(df_check)
        print('\nNow running conversion...')
        !python convert_asl_signs.py
    except Exception as e:
        print(f'❌ CSV read error: {e}')
else:
    print(f'❌ {train_csv} is missing. The extraction likely failed.')

✅ Found /content/asl_landmarks/train.csv
✅ CSV is readable. Preview:


,path,participant_id,sequence_id,sign
0,train_landmark_files/26734/1000035562.parquet,26734,1000035562,blow
1,train_landmark_files/28656/1000106739.parquet,28656,1000106739,wait
2,train_landmark_files/16069/100015657.parquet,16069,100015657,cloud
3,train_landmark_files/25571/1000210073.parquet,25571,1000210073,bird
4,train_landmark_files/62590/1000240708.parquet,62590,1000240708,owie



Now running conversion...
python3: can't open file '/content/convert_asl_signs.py': [Errno 2] No such file or directory


In [ ]:
%%writefile /content/convert_asl_signs.py
import pandas as pd
import numpy as np
import json
import os
from tqdm import tqdm

# Configuration
BASE_DIR = "/content/asl_landmarks"
TRAIN_CSV = f"{BASE_DIR}/train.csv"
OUTPUT_JSONL = "/content/drive/MyDrive/asl_train_data.jsonl"
PHRASE_MAP = "/content/drive/MyDrive/phrase_map.json"

if not os.path.exists(TRAIN_CSV):
    print(f"Error: {TRAIN_CSV} not found.")
else:
    train_df = pd.read_csv(TRAIN_CSV)
    unique_labels = sorted(train_df['sign'].unique())
    phrase_map = {"labels": [{"label": label} for label in unique_labels]}

    if not os.path.exists(PHRASE_MAP):
        with open(PHRASE_MAP, 'w') as f:
            json.dump(phrase_map, f)

    start_idx = 0
    if os.path.exists(OUTPUT_JSONL):
        try:
            with open(OUTPUT_JSONL, 'r') as f:
                start_idx = sum(1 for _ in f)
        except Exception:
            start_idx = 0

    print(f"Starting/Resuming from index {start_idx} (Total samples: {len(train_df)})")

    def process_parquet(path):
        df = pd.read_parquet(os.path.join(BASE_DIR, path))
        frames = []
        for frame_id, frame_df in df.groupby('frame'):
            coords = frame_df[['x', 'y', 'z']].values.flatten()
            if len(coords) == 1629:
                frames.append(coords.tolist())
            if len(frames) >= 30: break
        while len(frames) < 30:
            frames.append([0.0] * 1629)
        return frames[:30]

    with open(OUTPUT_JSONL, 'a') as out_f:
        for i in tqdm(range(start_idx, len(train_df))):
            row = train_df.iloc[i]
            try:
                frames = process_parquet(row['path'])
                record = {"label": row['sign'], "frames": frames}
                out_f.write(json.dumps(record) + "\n")
            except Exception:
                continue

Overwriting /content/convert_asl_signs.py


In [ ]:
import os
drive_zip = '/content/drive/MyDrive/asl-signs.zip'
local_extract_path = '/content/asl_landmarks'

# 1. Re-extract landmarks if missing
if not os.path.exists(os.path.join(local_extract_path, 'train.csv')):
    print("Extracting landmarks from Drive...")
    !mkdir -p {local_extract_path}
    !unzip -o -q "{drive_zip}" -d {local_extract_path}

# 2. Run conversion
if os.path.exists(os.path.join(local_extract_path, 'train.csv')):
    print("\n✅ Landmarks ready. Resuming conversion...")
    !python /content/convert_asl_signs.py
else:
    print("\n❌ Extraction failed. Please check Drive connection.")


✅ Landmarks ready. Resuming conversion...
Starting/Resuming from index 0 (Total samples: 94477)
100% 94477/94477 [1:48:34<00:00, 14.50it/s]


In [ ]:
import os
from google.colab import drive

# Force a remount to sync Drive file states
try:
    drive.mount('/content/drive', force_remount=True)
    print("\u2705 Drive refreshed successfully.")
except Exception as e:
    print(f"\u274c Refresh failed: {e}")

drive_jsonl = '/content/drive/MyDrive/asl_train_data.jsonl'
phrase_map = '/content/drive/MyDrive/phrase_map.json'

print('\n--- Final Data Verification ---')
if os.path.exists(drive_jsonl):
    size_gb = os.path.getsize(drive_jsonl) / (1024**3)
    with open(drive_jsonl, 'rb') as f:
        line_count = sum(1 for _ in f)

    print(f'\u2705 Dataset JSONL: Found')
    print(f'   Size: {size_gb:.4f} GB')
    print(f'   Samples: {line_count} / 94477')

    if line_count >= 94477:
        print('\n\ud83c\udf89 SUCCESS: The dataset is fully converted and ready for training.')
    else:
        print(f'\n\u26a0\ufe0f WARNING: Only {line_count} samples found.')
else:
    print('\u274c ERROR: JSONL file not found on Drive.')

if os.path.exists(phrase_map):
    print(f'\u2705 Phrase Map: Found')
else:
    print('\u274c ERROR: Phrase map missing.')

Mounted at /content/drive
✅ Drive refreshed successfully.

--- Final Data Verification ---
✅ Dataset JSONL: Found
   Size: 0.0000 GB
   Samples: 0 / 94477

⚠️ WARNING: Only 0 samples found.
✅ Phrase Map: Found


In [ ]:
import os

# Update script to target local disk first for reliability
with open('/content/convert_asl_signs.py', 'r') as f:
    script_content = f.read()

script_content = script_content.replace('/content/drive/MyDrive/asl_train_data.jsonl', '/content/asl_train_data.jsonl')
script_content = script_content.replace('/content/drive/MyDrive/phrase_map.json', '/content/phrase_map.json')

with open('/content/convert_asl_signs.py', 'w') as f:
    f.write(script_content)

print('✅ Conversion script updated to local paths. Starting conversion...')
!python /content/convert_asl_signs.py

# Verify local file and move to Drive
local_jsonl = '/content/asl_train_data.jsonl'
if os.path.exists(local_jsonl) and os.path.getsize(local_jsonl) > 0:
    import shutil
    print('\nMoving final dataset to Google Drive...')
    shutil.move(local_jsonl, '/content/drive/MyDrive/asl_train_data.jsonl')
    shutil.move('/content/phrase_map.json', '/content/drive/MyDrive/phrase_map.json')
    print('✅ SUCCESS: Dataset processed and moved to Drive.')
else:
    print('\n❌ Error: Local conversion failed to produce data.')

✅ Conversion script updated to local paths. Starting conversion...
Starting/Resuming from index 0 (Total samples: 94477)
  0% 182/94477 [00:12<1:49:49, 14.31it/s]
Traceback (most recent call last):
  File "/content/convert_asl_signs.py", line 52, in <module>
    out_f.write(json.dumps(record) + "\n")
                ^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/json/__init__.py", line 231, in dumps
    return _default_encoder.encode(obj)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/json/encoder.py", line 200, in encode
    chunks = self.iterencode(o, _one_shot=True)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/json/encoder.py", line 258, in iterencode
    return _iterencode(o, 0)
           ^^^^^^^^^^^^^^^^^
KeyboardInterrupt
^C

Moving final dataset to Google Drive...
✅ SUCCESS: Dataset processed and moved to Drive.


In [ ]:
# 1. Kill the process holding the 40GB ghost file
!kill -9 1575

# 2. Clear old extraction to save 29GB
!rm -rf /content/asl_landmarks
!rm -f /content/asl-signs.zip

# 3. Final Disk Check
print('Reclaiming space...')
import time
time.sleep(2)
!df -h /

/bin/bash: line 1: kill: (1575) - No such process
Reclaiming space...
Filesystem      Size  Used Avail Use% Mounted on
overlay         108G   21G   87G  20% /


In [ ]:
import os

drive_zip = '/content/drive/MyDrive/asl-signs.zip'
local_extract_path = '/content/asl_landmarks'

if os.path.exists(drive_zip):
    print(f'Found zip on Drive: {drive_zip}')
    print('Extracting landmarks to local runtime...')
    !mkdir -p {local_extract_path}
    !unzip -o -q "{drive_zip}" -d {local_extract_path}

    if os.path.exists(os.path.join(local_extract_path, 'train.csv')):
        print('\n✅ SUCCESS: Landmarks extracted. We are ready to run the conversion.')
    else:
        print('\n❌ Extraction failed to produce train.csv.')
else:
    print('❌ Zip file not found on Drive. Please verify the path.')

Found zip on Drive: /content/drive/MyDrive/asl-signs.zip
Extracting landmarks to local runtime...
error:  zipfile read error

✅ SUCCESS: Landmarks extracted. We are ready to run the conversion.


In [ ]:
import os
import pandas as pd
import numpy as np
import json
from tqdm import tqdm
import shutil

# Configuration
local_extract_path = '/content/asl_landmarks'
train_csv = os.path.join(local_extract_path, 'train.csv')
LOCAL_JSONL = '/content/asl_train_data.jsonl'
LOCAL_MAP = '/content/phrase_map.json'
DRIVE_DEST = '/content/drive/MyDrive/'
CHUNK_SIZE = 50000

def process_parquet(path):
    full_path = os.path.join(local_extract_path, path)
    if not os.path.exists(full_path):
        return None
    try:
        df = pd.read_parquet(full_path)
        frames = []
        for _, frame_df in df.groupby('frame'):
            coords = frame_df[['x', 'y', 'z']].values.flatten()
            if len(coords) == 1629:
                frames.append(coords.tolist())
            if len(frames) >= 30: break
        while len(frames) < 30:
            frames.append([0.0] * 1629)
        return frames[:30]
    except:
        return None

if os.path.exists(train_csv):
    train_df = pd.read_csv(train_csv)

    # Resume logic for local chunks
    start_idx = 0
    if os.path.exists(LOCAL_JSONL):
        with open(LOCAL_JSONL, 'r') as f:
            start_idx = sum(1 for _ in f)

    end_idx = min(start_idx + CHUNK_SIZE, len(train_df))

    if start_idx >= len(train_df):
        print('All samples already processed.')
    else:
        print(f'Processing samples {start_idx} to {end_idx}...')
        with open(LOCAL_JSONL, 'a') as out_f:
            for i in tqdm(range(start_idx, end_idx)):
                row = train_df.iloc[i]
                frames = process_parquet(row['path'])
                if frames:
                    record = {'label': row['sign'], 'frames': frames}
                    out_f.write(json.dumps(record) + '\n')

        print(f'\n✅ Chunk finished. Current local file size: {os.path.getsize(LOCAL_JSONL)/(1024**3):.2f} GB')
        print('Next step: Move this file to Drive to clear space.')
else:
    print('Error: Landmarks missing. Please re-run extraction.')

Processing samples 0 to 50000...


100%|██████████| 50000/50000 [1:03:50<00:00, 13.05it/s]



✅ Chunk finished. Current local file size: 33.08 GB
Next step: Move this file to Drive to clear space.


In [ ]:
import os
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload
from google.colab import auth

# 1. Authenticate to use the Drive API
auth.authenticate_user()
service = build('drive', 'v3')

local_file = '/content/asl_train_data.jsonl'
filename = 'asl_train_data.jsonl'

if os.path.exists(local_file):
    print(f"Starting direct API upload of {local_file} ({os.path.getsize(local_file)/(1024**3):.2f} GB)...")

    # 2. Setup metadata and media upload
    file_metadata = {'name': filename}
    media = MediaFileUpload(local_file, mimetype='application/json', resumable=True)

    # 3. Execute the upload
    request = service.files().create(body=file_metadata, media_body=media, fields='id')

    response = None
    while response is None:
        status, response = request.next_chunk()
        if status:
            print(f"Uploaded {int(status.progress() * 100)}%", end='\r')

    if response:
        print(f"\n✅ SUCCESS! File ID: {response.get('id')}")
        # 4. Remove local file immediately to free space
        os.remove(local_file)
        print("🧹 Local disk space reclaimed!")
else:
    print("❌ ERROR: Local file not found.")

Starting direct API upload of /content/asl_train_data.jsonl (33.08 GB)...

✅ SUCCESS! File ID: 1gCjbSWGQ2PSmnsmrAGm2cV9MZDmmURiT
🧹 Local disk space reclaimed!


In [ ]:
import os
# Reclaiming space: Check what is taking up space and remove local landmarks if necessary
!du -sh /content/*
!rm -rf /content/asl_landmarks
!df -h /

51G	/content/asl_landmarks
217G	/content/drive
55M	/content/sample_data
Filesystem      Size  Used Avail Use% Mounted on
overlay         108G   54G   54G  51% /


In [ ]:
import os

drive_zip = '/content/drive/MyDrive/asl-signs.zip'
local_extract_path = '/content/asl_landmarks'

if os.path.exists(drive_zip):
    print(f'Found zip on Drive: {drive_zip}')
    print('Extracting landmarks to local runtime...')
    os.makedirs(local_extract_path, exist_ok=True)
    !unzip -o -q "{drive_zip}" -d {local_extract_path}

    if os.path.exists(os.path.join(local_extract_path, 'train.csv')):
        print('\n✅ SUCCESS: Landmarks extracted. Now re-run the conversion cell above.')
    else:
        print('\n❌ Extraction failed to produce train.csv. Check if local disk is full.')
else:
    print('❌ Zip file not found on Drive. Please verify the path /content/drive/MyDrive/asl-signs.zip')

Found zip on Drive: /content/drive/MyDrive/asl-signs.zip
Extracting landmarks to local runtime...
error:  zipfile read error

✅ SUCCESS: Landmarks extracted. Now re-run the conversion cell above.


In [ ]:
import os
import pandas as pd
import numpy as np
import json
from tqdm import tqdm

# Configuration
local_extract_path = '/content/asl_landmarks'
train_csv = os.path.join(local_extract_path, 'train.csv')
LOCAL_JSONL = '/content/asl_train_data_part2.jsonl'

def process_parquet(path):
    full_path = os.path.join(local_extract_path, path)
    if not os.path.exists(full_path):
        return None
    try:
        df = pd.read_parquet(full_path)
        frames = []
        for _, frame_df in df.groupby('frame'):
            coords = frame_df[['x', 'y', 'z']].values.flatten()
            if len(coords) == 1629:
                frames.append(coords.tolist())
            if len(frames) >= 30: break
        while len(frames) < 30:
            frames.append([0.0] * 1629)
        return frames[:30]
    except:
        return None

if os.path.exists(train_csv):
    train_df = pd.read_csv(train_csv)
    start_idx = 50000
    end_idx = len(train_df)

    print(f'Processing remaining samples {start_idx} to {end_idx}...')
    with open(LOCAL_JSONL, 'w') as out_f:
        for i in tqdm(range(start_idx, end_idx)):
            row = train_df.iloc[i]
            frames = process_parquet(row['path'])
            if frames:
                record = {'label': row['sign'], 'frames': frames}
                out_f.write(json.dumps(record) + '\n')

    print(f'\n✅ Part 2 finished. Current local file size: {os.path.getsize(LOCAL_JSONL)/(1024**3):.2f} GB')
else:
    print('Error: Landmarks missing. Please re-run extraction cell (723aac3a).')

Error: Landmarks missing. Please re-run extraction cell (723aac3a).


In [ ]:
import os
# Check disk space before running conversion
!df -h /

Filesystem      Size  Used Avail Use% Mounted on
overlay         108G   72G   37G  67% /


In [ ]:
import os

drive_zip = '/content/drive/MyDrive/asl-signs.zip'
local_extract_path = '/content/asl_landmarks'

if os.path.exists(drive_zip):
    print(f'Found zip on Drive: {drive_zip}')
    print('Extracting landmarks to local runtime...')
    os.makedirs(local_extract_path, exist_ok=True)
    # Using -o to overwrite and -q for quiet mode to avoid stdout bloat
    !unzip -o -q "{drive_zip}" -d {local_extract_path}

    if os.path.exists(os.path.join(local_extract_path, 'train.csv')):
        print('\n✅ SUCCESS: Landmarks extracted. Now re-run the conversion cell above.')
    else:
        print('\n❌ Extraction failed to produce train.csv. Check if local disk is full.')
else:
    print('❌ Zip file not found on Drive. Please verify the path /content/drive/MyDrive/asl-signs.zip')

Found zip on Drive: /content/drive/MyDrive/asl-signs.zip
Extracting landmarks to local runtime...


In [ ]:
import os
!df -h /
print('\n--- Current /content directory ---')
!ls -R /content | head -n 20

landmarks_path = '/content/asl_landmarks'
if os.path.exists(landmarks_path):
    print(f'\nContent of {landmarks_path}:')
    !ls -F {landmarks_path}

Filesystem      Size  Used Avail Use% Mounted on
overlay         108G   21G   87G  20% /

--- Current /content directory ---
/content:
drive
sample_data

/content/drive:
MyDrive

/content/drive/MyDrive:
0141691337 (1).gsheet
0141691337.gsheet
15 promax take 1
15 promax take 2
16617142238805028041754937546803.jpg
1752482229827.pdf
1764585440_172580342533719555_subscribers_active.gsheet
1. Attendance & Reliability (The "Backbone")
Goal:... (1).gdoc
1. Attendance & Reliability (The "Backbone")
Goal:... (2).gdoc
1. Attendance & Reliability (The "Backbone")


In [ ]:
import os

drive_zip = '/content/drive/MyDrive/asl-signs.zip'
local_path = '/content/asl_landmarks'

# We already have train.csv, now we need the folders.
# Trying to list the folders inside the zip to extract them incrementally if possible
print("Attempting to list and extract folders individually to avoid read errors...")

# Check how many subdirectories are currently in train_landmark_files
landmarks_dir = os.path.join(local_path, 'train_landmark_files')
os.makedirs(landmarks_dir, exist_ok=True)

# Attempting a more resilient extraction for the first few folders needed for part 2
# Since the full unzip failed, we will try to extract specific chunks.
!unzip -o -q "{drive_zip}" "train_landmark_files/[0-9]*/*" -d "{local_path}" || echo "Partial extraction encountered issues, checking progress..."

subdirs = os.listdir(landmarks_dir)
print(f"\nSubdirectories extracted so far: {len(subdirs)}")
!du -sh {local_path}

Attempting to list and extract folders individually to avoid read errors...
error:  zipfile read error
Partial extraction encountered issues, checking progress...

Subdirectories extracted so far: 20
51G	/content/asl_landmarks


In [ ]:
import pandas as pd
import os

train_csv = '/content/asl_landmarks/train.csv'
landmarks_dir = '/content/asl_landmarks/train_landmark_files'

if os.path.exists(train_csv):
    df = pd.read_csv(train_csv)
    # Focus on Part 2: indices 50,000 to end
    part2_df = df.iloc[50000:]

    # Check how many of these files actually exist on disk
    missing_count = 0
    found_count = 0
    unique_folders_needed = set()

    for path in part2_df['path']:
        full_path = os.path.join('/content/asl_landmarks', path)
        unique_folders_needed.add(path.split('/')[0])
        if os.path.exists(full_path):
            found_count += 1
        else:
            missing_count += 1

    print(f"Part 2 Progress Check:")
    print(f" - Samples found: {found_count}")
    print(f" - Samples missing: {missing_count}")
    print(f" - Unique folders required for Part 2: {len(unique_folders_needed)}")

    # List some missing folders to target them specifically
    if missing_count > 0:
        missing_folders = [f for f in unique_folders_needed if not os.path.exists(os.path.join(landmarks_dir, f))]
        print(f" - Sample of missing folders: {missing_folders[:10]}")
else:
    print("❌ train.csv is missing. Cannot verify progress.")

Part 2 Progress Check:
 - Samples found: 42148
 - Samples missing: 2329
 - Unique folders required for Part 2: 1
 - Sample of missing folders: ['train_landmark_files']


In [ ]:
import os
import pandas as pd
import numpy as np
import json
from tqdm import tqdm

# Configuration
local_extract_path = '/content/asl_landmarks'
train_csv = os.path.join(local_extract_path, 'train.csv')
LOCAL_PART2_JSONL = '/content/asl_train_data_part2.jsonl'

def process_parquet(path):
    full_path = os.path.join(local_extract_path, path)
    if not os.path.exists(full_path):
        return None
    try:
        df = pd.read_parquet(full_path)
        frames = []
        for _, frame_df in df.groupby('frame'):
            coords = frame_df[['x', 'y', 'z']].values.flatten()
            if len(coords) == 1629:
                frames.append(coords.tolist())
            if len(frames) >= 30: break
        while len(frames) < 30:
            frames.append([0.0] * 1629)
        return frames[:30]
    except:
        return None

if os.path.exists(train_csv):
    train_df = pd.read_csv(train_csv)
    # Range for Part 2
    start_idx = 50000
    end_idx = len(train_df)

    print(f'Processing available samples for Part 2 ({start_idx} to {end_idx})...')
    with open(LOCAL_PART2_JSONL, 'w') as out_f:
        for i in tqdm(range(start_idx, end_idx)):
            row = train_df.iloc[i]
            frames = process_parquet(row['path'])
            if frames:
                record = {'label': row['sign'], 'frames': frames}
                out_f.write(json.dumps(record) + '\n')

    size_gb = os.path.getsize(LOCAL_PART2_JSONL) / (1024**3)
    print(f'\n✅ Part 2 conversion finished locally. Size: {size_gb:.2f} GB')
else:
    print('❌ Error: train.csv missing.')

Processing available samples for Part 2 (50000 to 94477)...


100%|██████████| 44477/44477 [49:03<00:00, 15.11it/s]


✅ Part 2 conversion finished locally. Size: 29.36 GB


In [ ]:
import os
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload
from google.colab import auth

# 1. Authenticate
auth.authenticate_user()
service = build('drive', 'v3')

local_file = '/content/asl_train_data_part2.jsonl'
filename = 'asl_train_data_part2.jsonl'

if os.path.exists(local_file):
    print(f"Starting direct API upload of {local_file} ({os.path.getsize(local_file)/(1024**3):.2f} GB)... ")

    # 2. Setup metadata and media upload
    file_metadata = {'name': filename}
    media = MediaFileUpload(local_file, mimetype='application/json', resumable=True)

    # 3. Execute the upload
    request = service.files().create(body=file_metadata, media_body=media, fields='id')

    response = None
    while response is None:
        status, response = request.next_chunk()
        if status:
            print(f"Uploaded {int(status.progress() * 100)}%", end='\r')

    if response:
        print(f"\n✅ SUCCESS! Part 2 uploaded to Drive. File ID: {response.get('id')}")
        # 4. Remove local file to reclaim space
        os.remove(local_file)
        print("🧹 Local disk space reclaimed!")
else:
    print("❌ ERROR: Part 2 local file not found.")

Starting direct API upload of /content/asl_train_data_part2.jsonl (29.36 GB)... 

✅ SUCCESS! Part 2 uploaded to Drive. File ID: 14-Od-zQkak-Ld0RPGmMocRgKTGN4jMOq
🧹 Local disk space reclaimed!


In [ ]:
import os

print('--- Final Runtime Resource Check ---')
!df -h / | grep '/' | awk '{print "Local Disk Available: " $4 " / " $2 " (" $5 " used)"}'

# Verify both parts exist on Drive
parts = ['asl_train_data.jsonl', 'asl_train_data_part2.jsonl', 'phrase_map.json']
print('\n--- Verifying Files on Drive ---')
for p in parts:
    path = os.path.join('/content/drive/MyDrive', p)
    if os.path.exists(path):
        size = os.path.getsize(path) / (1024**3)
        print(f'✅ Found: {p} ({size:.2f} GB)')
    else:
        print(f'❌ Missing: {p}')

--- Final Runtime Resource Check ---
Local Disk Available: 87G / 108G (20% used)

--- Verifying Files on Drive ---
✅ Found: asl_train_data.jsonl (33.08 GB)
✅ Found: asl_train_data_part2.jsonl (29.36 GB)
✅ Found: phrase_map.json (0.00 GB)


In [ ]:
import os
local_part2 = '/content/asl_train_data_part2.jsonl'
if os.path.exists(local_part2):
    size_gb = os.path.getsize(local_part2) / (1024**3)
    with open(local_part2, 'rb') as f:
        line_count = sum(1 for _ in f)
    print(f'Part 2 Local Status:')
    print(f' - Size: {size_gb:.2f} GB')
    print(f' - Samples: {line_count}')
    if line_count >= 44477:
        print('\n✅ SUCCESS: The remaining 44,477 samples are processed.')
    else:
        print(f'\n⚠️ PARTIAL: Found {line_count} samples. Still need {44477 - line_count} more.')
else:
    print('❌ Part 2 file not found locally.')

❌ Part 2 file not found locally.


## Data Loading Pipeline
We will now implement a custom Dataset and DataLoader to handle the large JSONL files stored on Google Drive.

In [ ]:
import torch
from torch.utils.data import IterableDataset, DataLoader
import json
import numpy as np

class ASLStreamingDataset(IterableDataset):
    def __init__(self, file_paths, phrase_map_path):
        self.file_paths = file_paths
        with open(phrase_map_path, 'r') as f:
            data = json.load(f)
            self.label_map = {item['label']: i for i, item in enumerate(data['labels'])}

    def __iter__(self):
        # For parallel loading: split files among workers
        worker_info = torch.utils.data.get_worker_info()
        if worker_info is None:
            files = self.file_paths
        else:
            # Distribute files across workers
            files = [f for i, f in enumerate(self.file_paths) if i % worker_info.num_workers == worker_info.id]

        for file_path in files:
            with open(file_path, 'r') as f:
                for line in f:
                    try:
                        data = json.loads(line)
                        label = self.label_map[data['label']]
                        frames = torch.tensor(data['frames'], dtype=torch.float32)
                        yield frames, label
                    except Exception:
                        continue

drive_files = [
    '/content/drive/MyDrive/asl_train_data.jsonl',
    '/content/drive/MyDrive/asl_train_data_part2.jsonl'
]
phrase_map_path = '/content/drive/MyDrive/phrase_map.json'

asl_dataset = ASLStreamingDataset(drive_files, phrase_map_path)
# Increased num_workers for parallel CPU data processing
asl_loader = DataLoader(asl_dataset, batch_size=128, num_workers=2, pin_memory=True)

print("Optimized Parallel DataLoader initialized.")

Optimized Parallel DataLoader initialized.


In [ ]:
# Test the loader
print("Testing the data loader with one batch...")
for frames, labels in asl_loader:
    print(f"Batch Shape: {frames.shape}") # Expected: [batch_size, 30, 1629]
    print(f"Labels Shape: {labels.shape}")
    break

Testing the data loader with one batch...
Batch Shape: torch.Size([64, 30, 1629])
Labels Shape: torch.Size([64])


# Model Development
We will define a Transformer-based model to classify the ASL sequences. This architecture is designed to handle the variable dependencies between landmarks over the 30-frame window.

In [ ]:
import torch.nn as nn
import math

class LandmarkTransformer(nn.Module):
    def __init__(self, num_classes=250, input_dim=1629, model_dim=256, num_heads=8, num_layers=4, dropout=0.1):
        super().__init__()

        # Linear projection to model dimension
        self.input_projection = nn.Linear(input_dim, model_dim)

        # Positional Encoding
        self.pos_encoder = nn.Parameter(torch.randn(1, 30, model_dim))

        # Transformer Encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=model_dim,
            nhead=num_heads,
            dim_feedforward=model_dim * 4,
            dropout=dropout,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # Classification Head
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(model_dim, num_classes)

    def forward(self, x):
        # x shape: (batch_size, 30, 1629)
        x = self.input_projection(x)
        x = x + self.pos_encoder

        x = self.transformer(x)

        # Global Average Pooling over time dimension
        x = x.mean(dim=1)

        x = self.dropout(x)
        return self.classifier(x)

# Initialize model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = LandmarkTransformer().to(device)

print(f"Model initialized on {device}.")
print(f"Total Parameters: {sum(p.numel() for p in model.parameters()):,}")

Model initialized on cpu.
Total Parameters: 3,648,250


In [ ]:
# Test the model with a single batch
model.eval()
with torch.no_grad():
    for frames, labels in asl_loader:
        frames = frames.to(device)
        output = model(frames)
        print(f"Model Output Shape: {output.shape}") # Expected: [64, 250]
        break

import torch.optim as optim

# Hyperparameters
learning_rate = 1e-4
epochs = 5

# Loss and Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

print(f"Starting training on {device}...")

model.train()
for epoch in range(epochs):
    running_loss = 0.0
    for batch_idx, (frames, labels) in enumerate(asl_loader):
        frames, labels = frames.to(device), labels.to(device)

        # Forward pass
        outputs = model(frames)
        loss = criterion(outputs, labels)

        # Backward and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        if (batch_idx + 1) % 100 == 0:
            print(f'Epoch [{epoch+1}/{epochs}], Batch [{batch_idx+1}], Loss: {running_loss/100:.4f}')
            running_loss = 0.0

print("Training Complete!")

Model Output Shape: torch.Size([64, 250])
Starting training on cpu...
Epoch [1/5], Batch [100], Loss: nan
Epoch [1/5], Batch [200], Loss: nan
Epoch [1/5], Batch [300], Loss: nan
Epoch [1/5], Batch [400], Loss: nan
Epoch [1/5], Batch [500], Loss: nan
Epoch [1/5], Batch [600], Loss: nan
Epoch [1/5], Batch [700], Loss: nan
Epoch [1/5], Batch [800], Loss: nan
Epoch [1/5], Batch [900], Loss: nan
Epoch [1/5], Batch [1000], Loss: nan
Epoch [1/5], Batch [1100], Loss: nan
Epoch [1/5], Batch [1200], Loss: nan
Epoch [1/5], Batch [1300], Loss: nan
Epoch [1/5], Batch [1400], Loss: nan
Epoch [2/5], Batch [100], Loss: nan
Epoch [2/5], Batch [200], Loss: nan
Epoch [2/5], Batch [300], Loss: nan
Epoch [2/5], Batch [400], Loss: nan
Epoch [2/5], Batch [500], Loss: nan
Epoch [2/5], Batch [600], Loss: nan
Epoch [2/5], Batch [700], Loss: nan
Epoch [2/5], Batch [800], Loss: nan
Epoch [2/5], Batch [900], Loss: nan
Epoch [2/5], Batch [1000], Loss: nan
Epoch [2/5], Batch [1100], Loss: nan
Epoch [2/5], Batch [120

## Model Training
We will now implement the training loop. We'll use `CrossEntropyLoss` for classification and the `Adam` optimizer. Given the streaming nature of our dataset, we will track progress per batch.

In [ ]:
import torch
import torch.nn as nn

# 1. Redefine Architecture (needed after runtime restart)
class LandmarkTransformer(nn.Module):
    def __init__(self, num_classes=250, input_dim=1629, model_dim=256, num_heads=8, num_layers=4, dropout=0.1):
        super().__init__()
        self.input_projection = nn.Linear(input_dim, model_dim)
        self.pos_encoder = nn.Parameter(torch.randn(1, 30, model_dim))
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=model_dim,
            nhead=num_heads,
            dim_feedforward=model_dim * 4,
            dropout=dropout,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(model_dim, num_classes)

    def forward(self, x):
        x = self.input_projection(x)
        x = x + self.pos_encoder
        x = self.transformer(x)
        x = x.mean(dim=1)
        x = self.dropout(x)
        return self.classifier(x)

# 2. Check GPU and Initialize
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = LandmarkTransformer().to(device)

if torch.cuda.is_available():
    print(f'✅ GPU detected: {torch.cuda.get_device_name(0)}')
    print(f'Model successfully moved to {device}')
else:
    print('❌ GPU not detected. Running on CPU.')

✅ GPU detected: Tesla T4
Model successfully moved to cuda


In [ ]:
import torch.optim as optim
import torch.nn as nn
from torch.cuda.amp import GradScaler, autocast

# Hyperparameters
learning_rate = 1e-4
epochs = 5

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# Scaler for Mixed Precision Training
scaler = GradScaler()

print(f"Starting optimized training on {device} using Mixed Precision...")

model.train()
for epoch in range(epochs):
    running_loss = 0.0
    for batch_idx, (frames, labels) in enumerate(asl_loader):
        frames, labels = frames.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        frames = torch.nan_to_num(frames, nan=0.0)

        # Runs the forward pass with autocasting
        with autocast():
            outputs = model(frames)
            loss = criterion(outputs, labels)

        optimizer.zero_grad()

        # Scales loss and calls backward() to create scaled gradients
        scaler.scale(loss).backward()

        # Unscales gradients before clipping
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        # scaler.step() first unscales the gradients of the optimizer's assigned params.
        # If these gradients do not contain infs or NaNs, optimizer.step() is then called.
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()

        if (batch_idx + 1) % 100 == 0:
            print(f'Epoch [{epoch+1}/{epochs}], Batch [{batch_idx+1}], Loss: {running_loss/100:.4f}')
            running_loss = 0.0

print("Training Complete!")

Starting optimized training on cuda using Mixed Precision...


/tmp/ipykernel_2693/4119511318.py:13: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
/tmp/ipykernel_2693/4119511318.py:25: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch [1/5], Batch [100], Loss: 5.5614
Epoch [1/5], Batch [200], Loss: 5.5406


/tmp/ipykernel_2693/4119511318.py:25: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch [1/5], Batch [300], Loss: 5.5301


/tmp/ipykernel_2693/4119511318.py:25: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch [1/5], Batch [400], Loss: 5.4988
Epoch [1/5], Batch [500], Loss: 5.4437
Epoch [1/5], Batch [600], Loss: 5.3958
Epoch [1/5], Batch [700], Loss: 5.3484
Epoch [2/5], Batch [100], Loss: 5.3007
Epoch [2/5], Batch [200], Loss: 5.3177
Epoch [2/5], Batch [300], Loss: 5.2751
Epoch [2/5], Batch [400], Loss: 5.2385
Epoch [2/5], Batch [500], Loss: 5.1879
Epoch [2/5], Batch [600], Loss: 5.1663
Epoch [2/5], Batch [700], Loss: 5.1300
Epoch [3/5], Batch [100], Loss: 5.0753
Epoch [3/5], Batch [200], Loss: 5.0239
Epoch [3/5], Batch [300], Loss: 5.0209
Epoch [3/5], Batch [400], Loss: 4.9752
Epoch [3/5], Batch [500], Loss: 4.9211
Epoch [3/5], Batch [600], Loss: 4.8799
Epoch [3/5], Batch [700], Loss: 4.8277
Epoch [4/5], Batch [100], Loss: 4.8007
Epoch [4/5], Batch [200], Loss: 4.7456
Epoch [4/5], Batch [300], Loss: 4.7380
Epoch [4/5], Batch [400], Loss: 4.6964
Epoch [4/5], Batch [500], Loss: 4.6574
Epoch [4/5], Batch [600], Loss: 4.6027
Epoch [4/5], Batch [700], Loss: 4.5450


## Model Checkpointing and Evaluation
We will define a helper to calculate accuracy and a function to save the model weights to Google Drive whenever the loss improves.

In [ ]:
def calculate_accuracy(outputs, targets):
    _, predicted = torch.max(outputs, 1)
    correct = (predicted == targets).sum().item()
    return correct / targets.size(0)

def save_checkpoint(model, optimizer, epoch, loss, path='/content/drive/MyDrive/asl_model_best.pth'):
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': loss,
    }, path)
    print(f'✅ Checkpoint saved to {path}')

## Advanced Training Loop
Now we integrate the checkpointing and accuracy tracking into a final training loop using the modern `torch.amp` API.

In [ ]:
import torch
import torch.optim as optim
import torch.nn as nn
from torch.utils.data import IterableDataset, DataLoader
import json
import os

# 1. Helpers and Data Loading
def calculate_accuracy(outputs, targets):
    _, predicted = torch.max(outputs, 1)
    correct = (predicted == targets).sum().item()
    return correct / targets.size(0)

def save_checkpoint(model, optimizer, epoch, loss, path='/content/drive/MyDrive/asl_model_best.pth'):
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': loss,
    }, path)
    print(f'✅ Checkpoint saved to {path}')

class ASLStreamingDataset(IterableDataset):
    def __init__(self, file_paths, phrase_map_path):
        self.file_paths = file_paths
        with open(phrase_map_path, 'r') as f:
            data = json.load(f)
            self.label_map = {item['label']: i for i, item in enumerate(data['labels'])}
    def __iter__(self):
        for file_path in self.file_paths:
            if not os.path.exists(file_path): continue
            with open(file_path, 'r') as f:
                for line in f:
                    try:
                        data = json.loads(line)
                        label = self.label_map[data['label']]
                        frames = torch.tensor(data['frames'], dtype=torch.float32)
                        yield frames, label
                    except Exception: continue

# Initialize Data Loader
drive_files = ['/content/drive/MyDrive/asl_train_data.jsonl', '/content/drive/MyDrive/asl_train_data_part2.jsonl']
phrase_map_path = '/content/drive/MyDrive/phrase_map.json'
asl_loader = DataLoader(ASLStreamingDataset(drive_files, phrase_map_path), batch_size=64)

# 2. Model Definition
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class LandmarkTransformer(nn.Module):
    def __init__(self, num_classes=250, input_dim=1629, model_dim=256, num_heads=8, num_layers=4, dropout=0.1):
        super().__init__()
        self.input_projection = nn.Linear(input_dim, model_dim)
        self.pos_encoder = nn.Parameter(torch.randn(1, 30, model_dim))
        encoder_layer = nn.TransformerEncoderLayer(d_model=model_dim, nhead=num_heads, dim_feedforward=model_dim * 4, dropout=dropout, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(model_dim, num_classes)
    def forward(self, x):
        x = self.input_projection(x) + self.pos_encoder
        x = self.transformer(x).mean(dim=1)
        return self.classifier(self.dropout(x))

model = LandmarkTransformer().to(device)

# 3. Training Loop
learning_rate = 1e-4
epochs = 5
best_loss = float('inf')
use_amp = torch.cuda.is_available()
scaler = torch.amp.GradScaler('cuda', enabled=use_amp)
optimizer = optim.Adam(model.parameters(), lr=learning_rate)
criterion = nn.CrossEntropyLoss()

print(f"Starting training on {device} (AMP: {use_amp})...")

model.train()
for epoch in range(epochs):
    running_loss, running_acc = 0.0, 0.0
    for batch_idx, (frames, labels) in enumerate(asl_loader):
        frames, labels = frames.to(device), labels.to(device)
        frames = torch.nan_to_num(frames, nan=0.0)
        optimizer.zero_grad()

        device_type = 'cuda' if use_amp else 'cpu'
        with torch.amp.autocast(device_type, enabled=use_amp):
            outputs = model(frames)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()
        running_acc += calculate_accuracy(outputs, labels)

        if (batch_idx + 1) % 100 == 0:
            avg_loss, avg_acc = running_loss / 100, running_acc / 100
            print(f'Epoch [{epoch+1}/{epochs}], Step [{batch_idx+1}], Loss: {avg_loss:.4f}, Acc: {avg_acc:.4f}')
            if avg_loss < best_loss:
                best_loss = avg_loss
                save_checkpoint(model, optimizer, epoch, avg_loss)
            running_loss, running_acc = 0.0, 0.0

print("Training Complete!")

Starting training on cpu (AMP: False)...
Epoch [1/5], Step [100], Loss: 5.5890, Acc: 0.0036
✅ Checkpoint saved to /content/drive/MyDrive/asl_model_best.pth
Epoch [1/5], Step [200], Loss: 5.5694, Acc: 0.0034
✅ Checkpoint saved to /content/drive/MyDrive/asl_model_best.pth
Epoch [1/5], Step [300], Loss: 5.5548, Acc: 0.0047
✅ Checkpoint saved to /content/drive/MyDrive/asl_model_best.pth
Epoch [1/5], Step [400], Loss: 5.5352, Acc: 0.0053
✅ Checkpoint saved to /content/drive/MyDrive/asl_model_best.pth
Epoch [1/5], Step [500], Loss: 5.5344, Acc: 0.0058
✅ Checkpoint saved to /content/drive/MyDrive/asl_model_best.pth
Epoch [1/5], Step [600], Loss: 5.5190, Acc: 0.0073
✅ Checkpoint saved to /content/drive/MyDrive/asl_model_best.pth
Epoch [1/5], Step [700], Loss: 5.4890, Acc: 0.0075
✅ Checkpoint saved to /content/drive/MyDrive/asl_model_best.pth
Epoch [1/5], Step [800], Loss: 5.4604, Acc: 0.0072
✅ Checkpoint saved to /content/drive/MyDrive/asl_model_best.pth
Epoch [1/5], Step [900], Loss: 5.4176, 

In [ ]:
import os
import shutil

# 1. Local path configuration
local_data_dir = '/content/train_data'
os.makedirs(local_data_dir, exist_ok=True)

drive_files = [
    '/content/drive/MyDrive/asl_train_data.jsonl',
    '/content/drive/MyDrive/asl_train_data_part2.jsonl'
]

local_files = []

print("Copying training data from Drive to local SSD for maximum speed...")
for drive_file in drive_files:
    filename = os.path.basename(drive_file)
    dest = os.path.join(local_data_dir, filename)
    if os.path.exists(drive_file):
        if not os.path.exists(dest):
            print(f"Copying {filename}...")
            shutil.copy2(drive_file, dest)
        else:
            print(f"{filename} already exists locally.")
        local_files.append(dest)
    else:
        print(f"❌ Missing on Drive: {drive_file}")

print(f"\n✅ Data ready at: {local_data_dir}")
!df -h /content

Copying training data from Drive to local SSD for maximum speed...
Copying asl_train_data.jsonl...
Copying asl_train_data_part2.jsonl...


OSError: [Errno 28] No space left on device: '/content/drive/MyDrive/asl_train_data_part2.jsonl' -> '/content/train_data/asl_train_data_part2.jsonl'

In [ ]:
import torch
from torch.utils.data import IterableDataset, DataLoader
import json
import os

class ASLHybridDataset(IterableDataset):
    def __init__(self, local_files, drive_files, phrase_map_path):
        self.file_paths = local_files + drive_files
        with open(phrase_map_path, 'r') as f:
            data = json.load(f)
            self.label_map = {item['label']: i for i, item in enumerate(data['labels'])}

    def __iter__(self):
        for file_path in self.file_paths:
            if not os.path.exists(file_path):
                print(f"Skipping missing file: {file_path}")
                continue
            with open(file_path, 'r') as f:
                for line in f:
                    try:
                        data = json.loads(line)
                        label = self.label_map[data['label']]
                        frames = torch.tensor(data['frames'], dtype=torch.float32)
                        yield frames, label
                    except Exception:
                        continue

# Configuration using available local files and remaining drive files
local_files = ['/content/train_data/asl_train_data.jsonl']
drive_files = ['/content/drive/MyDrive/asl_train_data_part2.jsonl']
phrase_map_path = '/content/drive/MyDrive/phrase_map.json'

asl_dataset = ASLHybridDataset(local_files, drive_files, phrase_map_path)
asl_loader = DataLoader(asl_dataset, batch_size=64, num_workers=0) # num_workers=0 to avoid Drive I/O conflicts

print("✅ Hybrid DataLoader (SSD + Drive) initialized.")

✅ Hybrid DataLoader (SSD + Drive) initialized.


In [ ]:
import torch
import torch.optim as optim
import torch.nn as nn
from torch.amp import GradScaler, autocast
import os

# 1. Redefine Architecture
class LandmarkTransformer(nn.Module):
    def __init__(self, num_classes=250, input_dim=1629, model_dim=256, num_heads=8, num_layers=4, dropout=0.1):
        super().__init__()
        self.input_projection = nn.Linear(input_dim, model_dim)
        self.pos_encoder = nn.Parameter(torch.randn(1, 30, model_dim))
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=model_dim,
            nhead=num_heads,
            dim_feedforward=model_dim * 4,
            dropout=dropout,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(model_dim, num_classes)

    def forward(self, x):
        x = self.input_projection(x) + self.pos_encoder
        x = self.transformer(x).mean(dim=1)
        return self.classifier(self.dropout(x))

# 2. Checkpointing Logic
CHECKPOINT_PATH = '/content/drive/MyDrive/asl_model_checkpoint.pth'

def save_checkpoint(model, optimizer, epoch, batch_idx, loss):
    torch.save({
        'epoch': epoch,
        'batch_idx': batch_idx,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': loss,
    }, CHECKPOINT_PATH)
    print(f'\u2705 Checkpoint saved to Drive at Step {batch_idx+1}')

# 3. Initialize
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = LandmarkTransformer().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()
scaler = GradScaler('cuda', enabled=(device.type == 'cuda'))

print(f"Training initialized on {device}. Checkpoints will save to Drive.")

# 4. Training Loop
model.train()
for epoch in range(5):
    running_loss, running_acc = 0.0, 0.0
    for batch_idx, (frames, labels) in enumerate(asl_loader):
        try:
            frames, labels = frames.to(device), labels.to(device)
            frames = torch.nan_to_num(frames, nan=0.0)

            optimizer.zero_grad()
            with autocast('cuda', enabled=(device.type == 'cuda')):
                outputs = model(frames)
                loss = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()

            running_loss += loss.item()
            running_acc += (outputs.argmax(1) == labels).sum().item() / labels.size(0)

            if (batch_idx + 1) % 100 == 0:
                avg_loss = running_loss / 100
                print(f'Epoch [{epoch+1}/5], Step [{batch_idx+1}], Loss: {avg_loss:.4f}, Acc: {running_acc/100:.4f}')
                save_checkpoint(model, optimizer, epoch, batch_idx, avg_loss)
                running_loss, running_acc = 0.0, 0.0

        except OSError as e:
            if 'No space left' in str(e):
                print("\u26a0\ufe0f Disk full error caught. Attempting to clear caches and continue...")
                torch.cuda.empty_cache()
                continue
            else: raise e

print("Training process finished.")

Training initialized on cuda. Checkpoints will save to Drive.
Epoch [1/5], Step [100], Loss: 5.5856, Acc: 0.0044
✅ Checkpoint saved to Drive at Step 100


In [ ]:
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Checking for GPU...')
if torch.cuda.is_available():
    print(f'✅ GPU is available: {torch.cuda.get_device_name(0)}')
    print('Recommendation: You are already set up for GPU training with AMP.')
else:
    print('❌ GPU is NOT available. Running on CPU.')
    print('To enable GPU: Go to Edit -> Notebook settings -> Hardware accelerator -> T4 GPU.')

Checking for GPU...
✅ GPU is available: Tesla T4
Recommendation: You are already set up for GPU training with AMP.


### Speed Optimization Strategies

To maximize training speed in Colab, consider these strategies:

1. **Mixed Precision Training (AMP)**: We are already using `torch.amp`, which speeds up training by using float16 where possible without losing accuracy.
2. **DataLoader Parallelism**: Increase `num_workers` in the `DataLoader` (e.g., to 2 or 4) to allow the CPU to prepare batches while the GPU is computing.
3. **Pin Memory**: Setting `pin_memory=True` in the `DataLoader` speeds up the transfer of data from CPU to GPU.
4. **Model Architecture**: For CPU training, we could reduce the `model_dim` or the number of Transformer layers to decrease the computational load.
5. **I/O Optimization**: Reading from local SSD is faster than streaming from Google Drive. If disk space allows, copying the JSONL files from Drive to `/content/` before starting training can eliminate network latency.

## Resume Training
We will load the `asl_model_best.pth` checkpoint and continue training for more epochs to further improve the classification accuracy.

In [ ]:
# 1. Load Checkpoint
checkpoint_path = '/content/drive/MyDrive/asl_model_best.pth'
additional_epochs = 10

if os.path.exists(checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    start_epoch = checkpoint['epoch'] + 1
    best_loss = checkpoint['loss']
    print(f"\u2705 Loaded checkpoint from epoch {start_epoch-1} (Loss: {best_loss:.4f})")
else:
    print("\u26a0\ufe0f No checkpoint found. Starting fresh.")
    start_epoch = 0
    best_loss = float('inf')

# 2. Continued Training Loop
print(f"Continuing training for {additional_epochs} more epochs...")

model.train()
for epoch in range(start_epoch, start_epoch + additional_epochs):
    running_loss, running_acc = 0.0, 0.0
    for batch_idx, (frames, labels) in enumerate(asl_loader):
        frames, labels = frames.to(device), labels.to(device)
        frames = torch.nan_to_num(frames, nan=0.0)

        optimizer.zero_grad()
        device_type = 'cuda' if use_amp else 'cpu'
        with torch.amp.autocast(device_type, enabled=use_amp):
            outputs = model(frames)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()
        running_acc += calculate_accuracy(outputs, labels)

        if (batch_idx + 1) % 100 == 0:
            avg_loss, avg_acc = running_loss / 100, running_acc / 100
            print(f'Epoch [{epoch+1}/{start_epoch + additional_epochs}], Step [{batch_idx+1}], Loss: {avg_loss:.4f}, Acc: {avg_acc:.4f}')
            if avg_loss < best_loss:
                best_loss = avg_loss
                save_checkpoint(model, optimizer, epoch, avg_loss)
            running_loss, running_acc = 0.0, 0.0

print("Extended Training Complete!")

✅ Loaded checkpoint from epoch 4 (Loss: 3.7431)
Continuing training for 10 more epochs...
Epoch [6/15], Step [100], Loss: 3.7283, Acc: 0.1766
✅ Checkpoint saved to /content/drive/MyDrive/asl_model_best.pth
Epoch [6/15], Step [200], Loss: 3.6593, Acc: 0.1920
✅ Checkpoint saved to /content/drive/MyDrive/asl_model_best.pth
Epoch [6/15], Step [300], Loss: 3.6730, Acc: 0.1781
Epoch [6/15], Step [400], Loss: 3.7029, Acc: 0.1762
Epoch [6/15], Step [500], Loss: 3.6795, Acc: 0.1684
Epoch [6/15], Step [600], Loss: 3.6286, Acc: 0.1822
✅ Checkpoint saved to /content/drive/MyDrive/asl_model_best.pth
Epoch [6/15], Step [700], Loss: 3.6242, Acc: 0.1942
✅ Checkpoint saved to /content/drive/MyDrive/asl_model_best.pth
Epoch [6/15], Step [800], Loss: 3.5607, Acc: 0.2047
✅ Checkpoint saved to /content/drive/MyDrive/asl_model_best.pth
Epoch [6/15], Step [900], Loss: 3.5679, Acc: 0.2006
Epoch [6/15], Step [1000], Loss: 3.5734, Acc: 0.2039
Epoch [6/15], Step [1100], Loss: 3.5337, Acc: 0.2027
✅ Checkpoint sav

OSError: [Errno 107] Transport endpoint is not connected

In [ ]:
import os
import torch

# 1. Enable Intel OneDNN Graph API and other CPU performance flags
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '1'
torch.set_num_threads(os.cpu_count())
torch.set_flush_denormal(True)

# 2. Define a 'Lite' version of the Transformer for CPU speed
# Reducing model_dim from 256 to 128 significantly speeds up matrix multiplications on CPU
class LandmarkTransformerLite(nn.Module):
    def __init__(self, num_classes=250, input_dim=1629, model_dim=128, num_heads=4, num_layers=2, dropout=0.1):
        super().__init__()
        self.input_projection = nn.Linear(input_dim, model_dim)
        self.pos_encoder = nn.Parameter(torch.randn(1, 30, model_dim))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=model_dim,
            nhead=num_heads,
            dim_feedforward=model_dim * 2,
            dropout=dropout,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.classifier = nn.Linear(model_dim, num_classes)

    def forward(self, x):
        # Efficient NaN handling on CPU
        x = torch.where(torch.isnan(x), torch.zeros_like(x), x)
        x = self.input_projection(x) + self.pos_encoder
        x = self.transformer(x).mean(dim=1)
        return self.classifier(x)

# 3. Instantiate and summarize speed gains
model_lite = LandmarkTransformerLite().to('cpu')
print(f'CPU-Optimized Model Initialized.')
print(f'Total Parameters: {sum(p.numel() for p in model_lite.parameters()):,}')
print(f'Current CPU Threads: {torch.get_num_threads()}')

CPU-Optimized Model Initialized.
Total Parameters: 509,690
Current CPU Threads: 2


In [ ]:
from google.colab import drive
import torch.optim as optim

# 1. Re-establish Drive connection
try:
    drive.mount('/content/drive', force_remount=True)
    print("✅ Drive reconnected.")
except Exception as e:
    print(f"❌ Failed to reconnect Drive: {e}")

# 2. Setup Training for Lite Model
optimizer_lite = optim.Adam(model_lite.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()
epochs = 5

print(f"Starting training for LandmarkTransformerLite on CPU...")

model_lite.train()
for epoch in range(epochs):
    running_loss, running_acc = 0.0, 0.0
    for batch_idx, (frames, labels) in enumerate(asl_loader):
        frames, labels = frames.to('cpu'), labels.to('cpu')

        optimizer_lite.zero_grad()
        outputs = model_lite(frames)
        loss = criterion(outputs, labels)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_lite.parameters(), max_norm=1.0)
        optimizer_lite.step()

        running_loss += loss.item()
        running_acc += (outputs.argmax(1) == labels).sum().item() / labels.size(0)

        if (batch_idx + 1) % 50 == 0:
            print(f'Epoch [{epoch+1}/{epochs}], Step [{batch_idx+1}], Loss: {running_loss/50:.4f}, Acc: {running_acc/50:.4f}')
            running_loss, running_acc = 0.0, 0.0

# 3. Save the new Lite model
torch.save(model_lite.state_dict(), '/content/drive/MyDrive/asl_model_lite.pth')
print("✅ Lite model saved to Drive.")

In [ ]:
from google.colab import drive
import torch.optim as optim

# 1. Re-establish Drive connection
try:
    drive.mount('/content/drive', force_remount=True)
    print("✅ Drive reconnected.")
except Exception as e:
    print(f"❌ Failed to reconnect Drive: {e}")

# 2. Setup Training for Lite Model
optimizer_lite = optim.Adam(model_lite.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()
epochs = 5

print(f"Starting training for LandmarkTransformerLite on CPU...")

model_lite.train()
for epoch in range(epochs):
    running_loss, running_acc = 0.0, 0.0
    # Using the existing asl_loader
    for batch_idx, (frames, labels) in enumerate(asl_loader):
        frames, labels = frames.to('cpu'), labels.to('cpu')

        optimizer_lite.zero_grad()
        outputs = model_lite(frames)
        loss = criterion(outputs, labels)

        loss.backward()
        # Gradient clipping for stability
        torch.nn.utils.clip_grad_norm_(model_lite.parameters(), max_norm=1.0)
        optimizer_lite.step()

        running_loss += loss.item()
        running_acc += calculate_accuracy(outputs, labels)

        if (batch_idx + 1) % 50 == 0:
            print(f'Epoch [{epoch+1}/{epochs}], Step [{batch_idx+1}], Loss: {running_loss/50:.4f}, Acc: {running_acc/50:.4f}')
            running_loss, running_acc = 0.0, 0.0

# 3. Save the new Lite model
torch.save(model_lite.state_dict(), '/content/drive/MyDrive/asl_model_lite.pth')
print("✅ Lite model saved to Drive.")

In [ ]:
t

### Resuming Training on CPU
This section re-establishes the connection and resumes training from the last saved checkpoint on Google Drive using the CPU.

### Runtime Health Check
If the cells below fail with a 'Transport' error, the runtime connection to Google Drive or the local disk has crashed. Please use **Runtime > Restart session** or **Disconnect and delete runtime** to recover.

In [ ]:
import os
import shutil

def check_runtime():
    print("--- System Status ---")
    !df -h /content

    print("\n--- Drive Connectivity ---")
    drive_path = '/content/drive/MyDrive'
    if os.path.exists(drive_path):
        try:
            files = os.listdir(drive_path)
            print(f"✅ Drive is connected. Found {len(files)} items in MyDrive.")
        except Exception as e:
            print(f"❌ Drive mount is broken: {e}")
    else:
        print("❌ Drive is not mounted.")

    print("\n--- Local Data Check ---")
    local_data = '/content/train_data/asl_train_data.jsonl'
    if os.path.exists(local_data):
        print(f"✅ Local data found: {os.path.getsize(local_data)/(1024**3):.2f} GB")
    else:
        print("❌ Local data file missing. You may need to re-copy from Drive.")

check_runtime()

--- System Status ---
Filesystem      Size  Used Avail Use% Mounted on
overlay         108G   21G   87G  20% /

--- Drive Connectivity ---
✅ Drive is connected. Found 935 items in MyDrive.

--- Local Data Check ---
❌ Local data file missing. You may need to re-copy from Drive.


In [ ]:
import os

# Verification of the data files being used for training
drive_jsonl = '/content/drive/MyDrive/asl_train_data.jsonl'
drive_jsonl_part2 = '/content/drive/MyDrive/asl_train_data_part2.jsonl'
local_jsonl = '/content/train_data/asl_train_data.jsonl'

files_to_check = [drive_jsonl, drive_jsonl_part2, local_jsonl]

print('--- Data Integrity Check ---')
for f_path in files_to_check:
    if os.path.exists(f_path):
        size_gb = os.path.getsize(f_path) / (1024**3)
        # Quick sample count
        with open(f_path, 'rb') as f:
            count = sum(1 for _ in f)
        print(f'\u2705 {f_path}')
        print(f'   Size: {size_gb:.2f} GB | Samples: {count}')
    else:
        print(f'\u274c {f_path} - Missing')

# Check the model state to see if it has non-zero weights
print('\n--- Model State Check ---')
params = [p for p in model.parameters()]
if len(params) > 0:
    weight_sum = params[0].sum().item()
    print(f'Model Weight Checksum (Sum of first layer): {weight_sum:.4f}')
    if weight_sum != 0:
        print('\u2705 Model weights are initialized and non-zero.')

In [ ]:
from google.colab import drive
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import os

# 1. Re-mount Drive
try:
    drive.mount('/content/drive', force_remount=True)
    print("✅ Drive reconnected.")
except Exception as e:
    print(f"❌ Failed to reconnect Drive: {e}")

# 2. Define Architecture (LandmarkTransformer)
class LandmarkTransformer(nn.Module):
    def __init__(self, num_classes=250, input_dim=1629, model_dim=256, num_heads=8, num_layers=4, dropout=0.1):
        super().__init__()
        self.input_projection = nn.Linear(input_dim, model_dim)
        self.pos_encoder = nn.Parameter(torch.randn(1, 30, model_dim))
        encoder_layer = nn.TransformerEncoderLayer(d_model=model_dim, nhead=num_heads, dim_feedforward=model_dim * 4, dropout=dropout, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(model_dim, num_classes)
    def forward(self, x):
        x = self.input_projection(x) + self.pos_encoder
        x = self.transformer(x).mean(dim=1)
        return self.classifier(self.dropout(x))

# 3. Setup CPU Device and Model
device = torch.device('cpu')
model = LandmarkTransformer().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()

# 4. Load the last checkpoint
checkpoint_path = '/content/drive/MyDrive/asl_model_best.pth'
if os.path.exists(checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    start_epoch = checkpoint['epoch']
    print(f"✅ Resuming from checkpoint (Epoch {start_epoch}, Loss: {checkpoint['loss']:.4f})")
else:
    print("⚠️ No checkpoint found on Drive. Starting from scratch.")
    start_epoch = 0

Mounted at /content/drive
✅ Drive reconnected.
✅ Resuming from checkpoint (Epoch 8, Loss: 2.9090)


In [ ]:
import torch
from torch.utils.data import IterableDataset, DataLoader
import json
import os

# Redefining the Dataset class to ensure it's in scope
class ASLHybridDataset(IterableDataset):
    def __init__(self, local_files, drive_files, phrase_map_path):
        self.file_paths = local_files + drive_files
        with open(phrase_map_path, 'r') as f:
            data = json.load(f)
            self.label_map = {item['label']: i for i, item in enumerate(data['labels'])}

    def __iter__(self):
        for file_path in self.file_paths:
            if not os.path.exists(file_path):
                continue
            with open(file_path, 'r') as f:
                for line in f:
                    try:
                        data = json.loads(line)
                        label = self.label_map[data['label']]
                        frames = torch.tensor(data['frames'], dtype=torch.float32)
                        yield frames, label
                    except Exception:
                        continue

# Configuration
local_files = ['/content/train_data/asl_train_data.jsonl']
drive_files = ['/content/drive/MyDrive/asl_train_data_part2.jsonl']
phrase_map_path = '/content/drive/MyDrive/phrase_map.json'

# Initialize Dataset
asl_dataset = ASLHybridDataset(local_files, drive_files, phrase_map_path)

print("Starting CPU training loop...")
model.train()

for epoch in range(start_epoch, start_epoch + 5):
    running_loss, running_acc = 0.0, 0.0
    # Using a batch size of 32 for CPU to avoid memory pressure
    asl_loader_cpu = DataLoader(asl_dataset, batch_size=32, num_workers=0)

    for batch_idx, (frames, labels) in enumerate(asl_loader_cpu):
        frames = torch.nan_to_num(frames, nan=0.0).to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(frames)
        loss = criterion(outputs, labels)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        running_loss += loss.item()
        running_acc += (outputs.argmax(1) == labels).sum().item() / labels.size(0)

        if (batch_idx + 1) % 50 == 0:
            print(f'Epoch [{epoch+1}], Step [{batch_idx+1}], Loss: {running_loss/50:.4f}, Acc: {running_acc/50:.4f}')
            # Save progress every 50 steps
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'loss': running_loss/50,
            }, '/content/drive/MyDrive/asl_model_best.pth')
            running_loss, running_acc = 0.0, 0.0

Starting CPU training loop...
Epoch [9], Step [50], Loss: 2.8811, Acc: 0.3262
Epoch [9], Step [100], Loss: 2.9845, Acc: 0.3069
Epoch [9], Step [150], Loss: 3.0177, Acc: 0.2969
Epoch [9], Step [200], Loss: 2.9515, Acc: 0.3219
Epoch [9], Step [250], Loss: 3.0525, Acc: 0.2944
Epoch [9], Step [300], Loss: 2.9403, Acc: 0.3106
Epoch [9], Step [350], Loss: 2.9869, Acc: 0.3081
Epoch [9], Step [400], Loss: 2.9959, Acc: 0.3069
Epoch [9], Step [450], Loss: 3.0501, Acc: 0.2950
Epoch [9], Step [500], Loss: 2.9922, Acc: 0.2975
Epoch [9], Step [550], Loss: 2.9485, Acc: 0.3150
Epoch [9], Step [600], Loss: 3.0180, Acc: 0.2981
Epoch [9], Step [650], Loss: 2.9610, Acc: 0.3200
Epoch [9], Step [700], Loss: 2.9909, Acc: 0.3013
Epoch [9], Step [750], Loss: 2.8553, Acc: 0.3431
Epoch [9], Step [800], Loss: 2.8963, Acc: 0.3375
Epoch [9], Step [850], Loss: 2.9569, Acc: 0.3169
Epoch [9], Step [900], Loss: 2.9498, Acc: 0.3131
Epoch [9], Step [950], Loss: 2.8709, Acc: 0.3344
Epoch [9], Step [1000], Loss: 2.9363, Ac

In [ ]:
def evaluate_model(model, dataset, criterion, num_batches=10):
    model.eval()
    total_loss = 0.0
    total_acc = 0.0
    eval_loader = DataLoader(dataset, batch_size=32, num_workers=0)

    print(f'Evaluating over {num_batches} batches...')
    with torch.no_grad():
        for i, (frames, labels) in enumerate(eval_loader):
            if i >= num_batches: break

            frames = torch.nan_to_num(frames, nan=0.0).to(device)
            labels = labels.to(device)

            outputs = model(frames)
            loss = criterion(outputs, labels)

            total_loss += loss.item()
            acc = (outputs.argmax(1) == labels).sum().item() / labels.size(0)
            total_acc += acc

    avg_loss = total_loss / num_batches
    avg_acc = total_acc / num_batches
    print(f'\n--- Validation Results ---')
    print(f'Average Loss: {avg_loss:.4f}')
    print(f'Average Accuracy: {avg_acc:.4%}')
    model.train() # Switch back to training mode
    return avg_loss, avg_acc

# Run a quick evaluation
eval_loss, eval_acc = evaluate_model(model, asl_dataset, criterion)

Evaluating over 10 batches...

--- Validation Results ---
Average Loss: 2.3431
Average Accuracy: 45.9375%


In [ ]:
# Continue Training Loop
print(f"Resuming training on {device}...")
model.train()

# We will run for another 5 epochs
for epoch in range(epoch, epoch + 5):
    running_loss, running_acc = 0.0, 0.0
    # Using the existing asl_dataset and a batch size of 32 for CPU stability
    asl_loader_cpu = DataLoader(asl_dataset, batch_size=32, num_workers=0)

    for batch_idx, (frames, labels) in enumerate(asl_loader_cpu):
        frames = torch.nan_to_num(frames, nan=0.0).to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(frames)
        loss = criterion(outputs, labels)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        running_loss += loss.item()
        running_acc += (outputs.argmax(1) == labels).sum().item() / labels.size(0)

        if (batch_idx + 1) % 50 == 0:
            avg_loss, avg_acc = running_loss/50, running_acc/50
            print(f'Epoch [{epoch+1}], Step [{batch_idx+1}], Loss: {avg_loss:.4f}, Acc: {avg_acc:.4f}')

            # Save progress to Drive
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'loss': avg_loss,
            }, '/content/drive/MyDrive/asl_model_best.pth')
            running_loss, running_acc = 0.0, 0.0

print("Training session complete.")

Resuming training on cpu...
Epoch [13], Step [50], Loss: 2.4241, Acc: 0.4200
Epoch [13], Step [100], Loss: 2.4205, Acc: 0.4188
Epoch [13], Step [150], Loss: 2.4880, Acc: 0.4037
Epoch [13], Step [200], Loss: 2.3283, Acc: 0.4462
Epoch [13], Step [250], Loss: 2.4377, Acc: 0.4238
Epoch [13], Step [300], Loss: 2.3430, Acc: 0.4313
Epoch [13], Step [350], Loss: 2.5224, Acc: 0.3887
Epoch [13], Step [400], Loss: 2.3730, Acc: 0.4313
Epoch [13], Step [450], Loss: 2.4117, Acc: 0.4256
Epoch [13], Step [500], Loss: 2.4265, Acc: 0.4194
Epoch [13], Step [550], Loss: 2.3604, Acc: 0.4256
Epoch [13], Step [600], Loss: 2.3105, Acc: 0.4369
Epoch [13], Step [650], Loss: 2.3814, Acc: 0.4200
Epoch [13], Step [700], Loss: 2.3588, Acc: 0.4244
Epoch [13], Step [750], Loss: 2.2602, Acc: 0.4581
Epoch [13], Step [800], Loss: 2.3144, Acc: 0.4419
Epoch [13], Step [850], Loss: 2.3907, Acc: 0.4219
Epoch [13], Step [900], Loss: 2.4304, Acc: 0.4219
Epoch [13], Step [950], Loss: 2.2435, Acc: 0.4569
Epoch [13], Step [1000]

## Upgraded Training Pipeline: Normalization, Motion Features, and Scheduling
This section implements the performance upgrades: nose-relative normalization, velocity/acceleration features, and a learning rate scheduler.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import IterableDataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau
import json
import os

def normalize_to_nose(frames):
    # frames: (seq_len, 1629)
    frames_3d = frames.view(frames.size(0), 543, 3)  # (seq_len, 543, 3)
    # Nose tip is pose landmark 0, which is index 468 in the 543-landmark set
    nose = frames_3d[:, 468:469, :]  # (seq_len, 1, 3)
    frames_3d = frames_3d - nose     # broadcast subtract
    return frames_3d.view(frames.size(0), 1629)

def add_motion_features(frames):
    # Lag-1: velocity
    diff1 = frames[1:] - frames[:-1]
    diff1 = torch.cat([diff1, torch.zeros(1, 1629)], dim=0)
    # Lag-2: acceleration
    diff2 = frames[2:] - frames[:-2]
    diff2 = torch.cat([diff2, torch.zeros(2, 1629)], dim=0)
    # Concatenate: [original | velocity | acceleration]
    return torch.cat([frames, diff1, diff2], dim=-1)

class ASLUpgradedDataset(IterableDataset):
    def __init__(self, local_files, drive_files, phrase_map_path):
        self.file_paths = local_files + drive_files
        with open(phrase_map_path, 'r') as f:
            data = json.load(f)
            self.label_map = {item['label']: i for i, item in enumerate(data['labels'])}

    def __iter__(self):
        for file_path in self.file_paths:
            if not os.path.exists(file_path): continue
            with open(file_path, 'r') as f:
                for line in f:
                    try:
                        data = json.loads(line)
                        label = self.label_map[data['label']]
                        frames = torch.tensor(data['frames'], dtype=torch.float32)

                        # Apply Upgrades
                        frames = normalize_to_nose(frames)
                        frames = add_motion_features(frames)

                        yield frames, label
                    except Exception: continue

In [ ]:
class LandmarkTransformerUpgraded(nn.Module):
    def __init__(self, num_classes=250, input_dim=4887, model_dim=256, num_heads=8, num_layers=4, dropout=0.1):
        super().__init__()
        self.input_projection = nn.Linear(input_dim, model_dim)
        self.pos_encoder = nn.Parameter(torch.randn(1, 30, model_dim))
        encoder_layer = nn.TransformerEncoderLayer(d_model=model_dim, nhead=num_heads, dim_feedforward=model_dim * 4, dropout=dropout, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(model_dim, num_classes)

    def forward(self, x):
        x = self.input_projection(x) + self.pos_encoder
        x = self.transformer(x).mean(dim=1)
        return self.classifier(self.dropout(x))

# Initialize
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = LandmarkTransformerUpgraded().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-4)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)
criterion = nn.CrossEntropyLoss()

# Data Loader
local_files = ['/content/train_data/asl_train_data.jsonl']
drive_files = ['/content/drive/MyDrive/asl_train_data_part2.jsonl']
phrase_map_path = '/content/drive/MyDrive/phrase_map.json'
asl_dataset = ASLUpgradedDataset(local_files, drive_files, phrase_map_path)
asl_loader = DataLoader(asl_dataset, batch_size=32, num_workers=0)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import IterableDataset, DataLoader
import json
import os

# 1. Dataset Definition (Ensures self-containment)
class ASLUpgradedDataset(IterableDataset):
    def __init__(self, local_files, drive_files, phrase_map_path):
        self.file_paths = local_files + drive_files
        with open(phrase_map_path, 'r') as f:
            data = json.load(f)
            self.label_map = {item['label']: i for i, item in enumerate(data['labels'])}

    def __iter__(self):
        for file_path in self.file_paths:
            if not os.path.exists(file_path): continue
            with open(file_path, 'r') as f:
                for line in f:
                    try:
                        data = json.loads(line)
                        label = self.label_map[data['label']]
                        frames = torch.tensor(data['frames'], dtype=torch.float32)
                        # Normalization & Motion Features
                        frames_3d = frames.view(frames.size(0), 543, 3)
                        nose = frames_3d[:, 468:469, :]
                        frames_3d = frames_3d - nose
                        frames = frames_3d.view(frames.size(0), 1629)
                        diff1 = torch.cat([frames[1:] - frames[:-1], torch.zeros(1, 1629)], dim=0)
                        diff2 = torch.cat([frames[2:] - frames[:-2], torch.zeros(2, 1629)], dim=0)
                        yield torch.cat([frames, diff1, diff2], dim=-1), label
                    except Exception: continue

# 2. Architecture
class LandmarkTransformerUpgraded(nn.Module):
    def __init__(self, num_classes=250, input_dim=4887, model_dim=256, num_heads=8, num_layers=4, dropout=0.1):
        super().__init__()
        self.input_projection = nn.Linear(input_dim, model_dim)
        self.pos_encoder = nn.Parameter(torch.randn(1, 30, model_dim))
        encoder_layer = nn.TransformerEncoderLayer(d_model=model_dim, nhead=num_heads, dim_feedforward=model_dim * 4, dropout=dropout, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(model_dim, num_classes)

    def forward(self, x):
        x = self.input_projection(x) + self.pos_encoder
        x = self.transformer(x).mean(dim=1)
        return self.classifier(self.dropout(x))

# 3. Setup Data, Model, and Optimizer
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
local_files = ['/content/train_data/asl_train_data.jsonl']
drive_files = ['/content/drive/MyDrive/asl_train_data_part2.jsonl']
phrase_map_path = '/content/drive/MyDrive/phrase_map.json'
asl_dataset = ASLUpgradedDataset(local_files, drive_files, phrase_map_path)
asl_loader = DataLoader(asl_dataset, batch_size=32, num_workers=0)

model = LandmarkTransformerUpgraded().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-4)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)
criterion = nn.CrossEntropyLoss()

print(f"Starting Upgraded Training on {device} (with Mixed Precision)...")
model.train()
use_amp = torch.cuda.is_available()
scaler = torch.amp.GradScaler('cuda', enabled=use_amp)

for epoch in range(5):
    running_loss, running_acc = 0.0, 0.0
    epoch_loss = 0.0
    for batch_idx, (frames, labels) in enumerate(asl_loader):
        frames, labels = frames.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        frames = torch.nan_to_num(frames, nan=0.0)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda', enabled=use_amp):
            outputs = model(frames)
            loss = criterion(outputs, labels)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item()
        epoch_loss += loss.item()
        running_acc += (outputs.argmax(1) == labels).sum().item() / labels.size(0)
        if (batch_idx + 1) % 100 == 0:
            print(f'Epoch [{epoch+1}], Step [{batch_idx+1}], Loss: {running_loss/100:.4f}, Acc: {running_acc/100:.4f}')
            running_loss, running_acc = 0.0, 0.0
    scheduler.step(epoch_loss / (batch_idx + 1))
    torch.save(model.state_dict(), '/content/drive/MyDrive/asl_model_upgraded.pth')
    print(f'✅ Epoch {epoch+1} complete.')

Starting Upgraded Training on cuda (with Mixed Precision)...
Epoch [1], Step [100], Loss: 5.5853, Acc: 0.0041
Epoch [1], Step [200], Loss: 5.5645, Acc: 0.0059
Epoch [1], Step [300], Loss: 5.5327, Acc: 0.0056
Epoch [1], Step [400], Loss: 5.5088, Acc: 0.0094
Epoch [1], Step [500], Loss: 5.4993, Acc: 0.0078
Epoch [1], Step [600], Loss: 5.4763, Acc: 0.0072
Epoch [1], Step [700], Loss: 5.4766, Acc: 0.0037
Epoch [1], Step [800], Loss: 5.4653, Acc: 0.0088
Epoch [1], Step [900], Loss: 5.4250, Acc: 0.0100
Epoch [1], Step [1000], Loss: 5.3983, Acc: 0.0106
Epoch [1], Step [1100], Loss: 5.3916, Acc: 0.0112
Epoch [1], Step [1200], Loss: 5.3866, Acc: 0.0103
Epoch [1], Step [1300], Loss: 5.3798, Acc: 0.0112
✅ Epoch 1 complete.
Epoch [2], Step [100], Loss: 5.3227, Acc: 0.0150
Epoch [2], Step [200], Loss: 5.3019, Acc: 0.0150
Epoch [2], Step [300], Loss: 5.3110, Acc: 0.0128
Epoch [2], Step [400], Loss: 5.3032, Acc: 0.0172
Epoch [2], Step [500], Loss: 5.3069, Acc: 0.0144
Epoch [2], Step [600], Loss: 5.26

### Extended CPU Training for Upgraded Model
This cell continues training the `LandmarkTransformerUpgraded` model on the CPU until it reaches the target accuracy or epoch limit.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import os
import json

# 1. Configuration
TARGET_ACCURACY = 0.45
MAX_EPOCHS = 30
CHECKPOINT_PATH = '/content/drive/MyDrive/asl_model_upgraded.pth'
BEST_MODEL_PATH = '/content/drive/MyDrive/asl_model_upgraded_best.pth'
PHRASE_MAP_PATH = '/content/drive/MyDrive/phrase_map.json'
LOCAL_FILES = ['/content/train_data/asl_train_data.jsonl']
DRIVE_FILES = ['/content/drive/MyDrive/asl_train_data_part2.jsonl']

# 2. Re-define Data Pipeline
class ASLUpgradedDataset(torch.utils.data.IterableDataset):
    def __init__(self, local_files, drive_files, phrase_map_path):
        self.file_paths = local_files + drive_files
        with open(phrase_map_path, 'r') as f:
            data = json.load(f)
            self.label_map = {item['label']: i for i, item in enumerate(data['labels'])}
    def __iter__(self):
        for file_path in self.file_paths:
            if not os.path.exists(file_path): continue
            with open(file_path, 'r') as f:
                for line in f:
                    try:
                        data = json.loads(line)
                        label = self.label_map[data['label']]
                        frames = torch.tensor(data['frames'], dtype=torch.float32)
                        frames_3d = frames.view(frames.size(0), 543, 3)
                        nose = frames_3d[:, 468:469, :]
                        frames_3d = frames_3d - nose
                        frames = frames_3d.view(frames.size(0), 1629)
                        diff1 = torch.cat([frames[1:] - frames[:-1], torch.zeros(1, 1629)], dim=0)
                        diff2 = torch.cat([frames[2:] - frames[:-2], torch.zeros(2, 1629)], dim=0)
                        yield torch.cat([frames, diff1, diff2], dim=-1), label
                    except Exception: continue

asl_dataset = ASLUpgradedDataset(LOCAL_FILES, DRIVE_FILES, PHRASE_MAP_PATH)
asl_loader = DataLoader(asl_dataset, batch_size=32, num_workers=0)

# 3. Re-define Architecture
class LandmarkTransformerUpgraded(nn.Module):
    def __init__(self, num_classes=250, input_dim=4887, model_dim=256, num_heads=8, num_layers=4, dropout=0.1):
        super().__init__()
        self.input_projection = nn.Linear(input_dim, model_dim)
        self.pos_encoder = nn.Parameter(torch.randn(1, 30, model_dim))
        encoder_layer = nn.TransformerEncoderLayer(d_model=model_dim, nhead=num_heads, dim_feedforward=model_dim * 4, dropout=dropout, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(model_dim, num_classes)
    def forward(self, x):
        x = self.input_projection(x) + self.pos_encoder
        x = self.transformer(x).mean(dim=1)
        return self.classifier(self.dropout(x))

device = torch.device('cpu')
torch.set_num_threads(os.cpu_count())
model = LandmarkTransformerUpgraded().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()

# 4. Load weights and train
if os.path.exists(CHECKPOINT_PATH):
    model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=device))
    print("✅ Loaded upgraded model weights from Drive.")

best_acc = 0.0
current_epoch = 5
stop_training = False

for epoch in range(current_epoch, MAX_EPOCHS):
    if stop_training: break
    model.train()
    running_loss, running_acc = 0.0, 0.0
    for batch_idx, (frames, labels) in enumerate(asl_loader):
        frames, labels = frames.to(device), labels.to(device)
        frames = torch.nan_to_num(frames, nan=0.0)
        optimizer.zero_grad()
        outputs = model(frames)
        loss = criterion(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        acc = (outputs.argmax(1) == labels).sum().item() / labels.size(0)
        running_loss += loss.item()
        running_acc += acc
        if (batch_idx + 1) % 50 == 0:
            avg_loss, avg_acc = running_loss / 50, running_acc / 50
            print(f'Epoch [{epoch+1}/{MAX_EPOCHS}], Step [{batch_idx+1}], Loss: {avg_loss:.4f}, Acc: {avg_acc:.4f}')
            if avg_acc > best_acc:
                best_acc = avg_acc
                torch.save(model.state_dict(), BEST_MODEL_PATH)
                print(f"⭐ New Best Accuracy: {best_acc:.4f}. Saved to Drive.")
            if avg_acc >= TARGET_ACCURACY:
                stop_training = True
                break
            running_loss, running_acc = 0.0, 0.0
    torch.save(model.state_dict(), CHECKPOINT_PATH)
print("Training session ended.")

✅ Loaded upgraded model weights from Drive.
Epoch [6/30], Step [50], Loss: 4.7864, Acc: 0.0675
⭐ New Best Accuracy: 0.0675. Saved to Drive.
Epoch [6/30], Step [100], Loss: 4.7129, Acc: 0.0881
⭐ New Best Accuracy: 0.0881. Saved to Drive.
Epoch [6/30], Step [150], Loss: 4.7226, Acc: 0.0862
Epoch [6/30], Step [200], Loss: 4.6577, Acc: 0.0813
Epoch [6/30], Step [250], Loss: 4.7009, Acc: 0.0825
Epoch [6/30], Step [300], Loss: 4.7529, Acc: 0.0750
Epoch [6/30], Step [350], Loss: 4.7417, Acc: 0.0844
Epoch [6/30], Step [400], Loss: 4.6892, Acc: 0.0838
Epoch [6/30], Step [450], Loss: 4.6802, Acc: 0.0875
Epoch [6/30], Step [500], Loss: 4.7451, Acc: 0.0819
Epoch [6/30], Step [550], Loss: 4.7281, Acc: 0.0869
Epoch [6/30], Step [600], Loss: 4.6550, Acc: 0.0912
⭐ New Best Accuracy: 0.0912. Saved to Drive.
Epoch [6/30], Step [650], Loss: 4.6820, Acc: 0.0762
Epoch [6/30], Step [700], Loss: 4.7001, Acc: 0.0869
Epoch [6/30], Step [750], Loss: 4.7209, Acc: 0.0838
Epoch [6/30], Step [800], Loss: 4.6961, Ac

In [ ]:
import torch
# Check if GPU has become available
if torch.cuda.is_available():
    print(f'✅ GPU is available: {torch.cuda.get_device_name(0)}')
    print('You can now proceed with GPU-accelerated training.')
else:
    print('❌ GPU is still not available. Continuing on CPU...')

❌ GPU is still not available. Continuing on CPU...


### GPU-Accelerated Training (Run if GPU is assigned)
This loop uses `torch.amp` to maximize the Tesla T4 performance.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.amp import GradScaler, autocast

if torch.cuda.is_available():
    device = torch.device('cuda')
    model.to(device)

    # Re-initialize Optimizer for the correct device
    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    scaler = GradScaler('cuda')

    print(f"Resuming training on {torch.cuda.get_device_name(0)}...")

    model.train()
    for epoch in range(current_epoch, MAX_EPOCHS):
        running_loss, running_acc = 0.0, 0.0

        for batch_idx, (frames, labels) in enumerate(asl_loader):
            frames, labels = frames.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            frames = torch.nan_to_num(frames, nan=0.0)

            optimizer.zero_grad()

            with autocast('cuda'):
                outputs = model(frames)
                loss = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()

            acc = (outputs.argmax(1) == labels).sum().item() / labels.size(0)
            running_loss += loss.item()
            running_acc += acc

            if (batch_idx + 1) % 100 == 0:
                avg_loss, avg_acc = running_loss/100, running_acc/100
                print(f'Epoch [{epoch+1}], Step [{batch_idx+1}], Loss: {avg_loss:.4f}, Acc: {avg_acc:.4f}')

                if avg_acc > best_acc:
                    best_acc = avg_acc
                    torch.save(model.state_dict(), BEST_MODEL_PATH)

                if avg_acc >= TARGET_ACCURACY:
                    print(f"🎯 Target {TARGET_ACCURACY} reached on GPU!")
                    break
                running_loss, running_acc = 0.0, 0.0

        torch.save(model.state_dict(), CHECKPOINT_PATH)
else:
    print("GPU not found. Please switch runtime type or use the CPU loop above.")

GPU not found. Please switch runtime type or use the CPU loop above.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import IterableDataset, DataLoader
import os
import shutil
import json

# 1. Dataset Definition
class ASLUpgradedDataset(IterableDataset):
    def __init__(self, local_files, drive_files, phrase_map_path):
        self.file_paths = local_files + drive_files
        with open(phrase_map_path, 'r') as f:
            data = json.load(f)
            self.label_map = {item['label']: i for i, item in enumerate(data['labels'])}

    def __iter__(self):
        for file_path in self.file_paths:
            if not os.path.exists(file_path): continue
            with open(file_path, 'r') as f:
                for line in f:
                    try:
                        data = json.loads(line)
                        label = self.label_map[data['label']]
                        frames = torch.tensor(data['frames'], dtype=torch.float32)
                        # Normalization & Motion Features
                        frames_3d = frames.view(frames.size(0), 543, 3)
                        nose = frames_3d[:, 468:469, :]
                        frames_3d = frames_3d - nose
                        frames = frames_3d.view(frames.size(0), 1629)
                        diff1 = torch.cat([frames[1:] - frames[:-1], torch.zeros(1, 1629)], dim=0)
                        diff2 = torch.cat([frames[2:] - frames[:-2], torch.zeros(2, 1629)], dim=0)
                        yield torch.cat([frames, diff1, diff2], dim=-1), label
                    except Exception: continue

# 2. Architecture Definition
class LandmarkTransformerUpgraded(nn.Module):
    def __init__(self, num_classes=250, input_dim=4887, model_dim=256, num_heads=8, num_layers=4, dropout=0.1):
        super().__init__()
        self.input_projection = nn.Linear(input_dim, model_dim)
        self.pos_encoder = nn.Parameter(torch.randn(1, 30, model_dim))
        encoder_layer = nn.TransformerEncoderLayer(d_model=model_dim, nhead=num_heads, dim_feedforward=model_dim * 4, dropout=dropout, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(model_dim, num_classes)
    def forward(self, x):
        x = self.input_projection(x) + self.pos_encoder
        x = self.transformer(x).mean(dim=1)
        return self.classifier(self.dropout(x))

# 3. Environment Setup & Data Restoration
local_data_path = '/content/train_data/asl_train_data.jsonl'
os.makedirs('/content/train_data', exist_ok=True)
if not os.path.exists(local_data_path):
    print('⌛ Re-copying data from Drive to local SSD...')
    shutil.copy2('/content/drive/MyDrive/asl_train_data.jsonl', local_data_path)
    print('✅ Local data restored.')

# Initialize Data Loader
local_files = [local_data_path]
drive_files = ['/content/drive/MyDrive/asl_train_data_part2.jsonl']
phrase_map_path = '/content/drive/MyDrive/phrase_map.json'
asl_dataset = ASLUpgradedDataset(local_files, drive_files, phrase_map_path)
asl_loader = DataLoader(asl_dataset, batch_size=32, num_workers=0)

# 4. Initialize Model and Load Weights
device = torch.device('cpu')
torch.set_num_threads(os.cpu_count())
model = LandmarkTransformerUpgraded().to(device)

weights_path = '/content/drive/MyDrive/asl_model_upgraded.pth'
if os.path.exists(weights_path):
    model.load_state_dict(torch.load(weights_path, map_location=device))
    print('✅ Model weights loaded from Drive.')

optimizer = optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()

print(f'Starting optimized CPU training (Threads: {os.cpu_count()})...')

# 5. Training Loop
model.train()
for epoch in range(23, 30):
    running_loss, running_acc = 0.0, 0.0
    for batch_idx, (frames, labels) in enumerate(asl_loader):
        frames = torch.nan_to_num(frames, nan=0.0).to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(frames)
        loss = criterion(outputs, labels)

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        acc = (outputs.argmax(1) == labels).sum().item() / labels.size(0)
        running_loss += loss.item()
        running_acc += acc

        if (batch_idx + 1) % 50 == 0:
            avg_loss, avg_acc = running_loss / 50, running_acc / 50
            print(f'Epoch [{epoch+1}/30], Step [{batch_idx+1}], Loss: {avg_loss:.4f}, Acc: {avg_acc:.4f}')

            if avg_acc >= 0.45:
                print(f'✨ Goal Reached! Accuracy: {avg_acc:.4f}')
                torch.save(model.state_dict(), "/content/drive/MyDrive/asl_model_upgraded_best.pth")
                break
            running_loss, running_acc = 0.0, 0.0

    torch.save(model.state_dict(), "/content/drive/MyDrive/asl_model_upgraded.pth")

✅ Model weights loaded from Drive.
Starting optimized CPU training (Threads: 2)...
Epoch [24/30], Step [50], Loss: 4.4301, Acc: 0.1600
Epoch [24/30], Step [100], Loss: 4.3768, Acc: 0.1756
Epoch [24/30], Step [150], Loss: 4.3489, Acc: 0.1725
Epoch [24/30], Step [200], Loss: 4.3377, Acc: 0.1769
Epoch [24/30], Step [250], Loss: 4.3278, Acc: 0.1775
Epoch [24/30], Step [300], Loss: 4.2259, Acc: 0.1844
Epoch [24/30], Step [350], Loss: 4.3147, Acc: 0.1837
Epoch [24/30], Step [400], Loss: 4.2354, Acc: 0.1856
Epoch [24/30], Step [450], Loss: 4.1745, Acc: 0.1844
Epoch [24/30], Step [500], Loss: 4.2468, Acc: 0.1781
Epoch [24/30], Step [550], Loss: 4.3473, Acc: 0.1606
Epoch [24/30], Step [600], Loss: 4.1693, Acc: 0.1938
Epoch [24/30], Step [650], Loss: 4.2825, Acc: 0.1719
Epoch [24/30], Step [700], Loss: 4.2011, Acc: 0.1762
Epoch [24/30], Step [750], Loss: 4.3731, Acc: 0.1544
Epoch [24/30], Step [800], Loss: 4.2434, Acc: 0.1844
Epoch [24/30], Step [850], Loss: 4.2844, Acc: 0.1619
Epoch [24/30], St

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import os

# Continue from Epoch 24 where we left off
device = torch.device('cpu')
torch.set_num_threads(os.cpu_count())

# Restore latest weights for training continuity
weights_path = '/content/drive/MyDrive/asl_model_upgraded.pth'
if os.path.exists(weights_path):
    model.load_state_dict(torch.load(weights_path, map_location=device))
    print(f'✅ Weights re-synced from Drive. Resuming training...')

optimizer = optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()

# Resume Training Loop
model.train()
# Adjusting loop to start from Epoch 24 and continue
for epoch in range(24, 30):
    running_loss, running_acc = 0.0, 0.0
    for batch_idx, (frames, labels) in enumerate(asl_loader):
        # Skip samples already processed in previous partial run of Epoch 24
        if epoch == 24 and batch_idx < 1650:
            continue

        frames = torch.nan_to_num(frames, nan=0.0).to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(frames)
        loss = criterion(outputs, labels)

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        acc = (outputs.argmax(1) == labels).sum().item() / labels.size(0)
        running_loss += loss.item()
        running_acc += acc

        if (batch_idx + 1) % 50 == 0:
            avg_loss, avg_acc = running_loss / 50, running_acc / 50
            print(f'Epoch [{epoch+1}/30], Step [{batch_idx+1}], Loss: {avg_loss:.4f}, Acc: {avg_acc:.4f}')

            # Save to 'best' if we hit the 45% goal
            if avg_acc >= 0.45:
                print(f'✨ Goal Reached! Accuracy: {avg_acc:.4f}. Saving best model...')
                torch.save(model.state_dict(), "/content/drive/MyDrive/asl_model_upgraded_best.pth")
                break
            running_loss, running_acc = 0.0, 0.0

    # Regular checkpoint save after every epoch
    torch.save(model.state_dict(), "/content/drive/MyDrive/asl_model_upgraded.pth")
    print(f'✅ Epoch {epoch+1} saved to Drive.')

NameError: name 'model' is not defined

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import IterableDataset, DataLoader
from torch.amp import GradScaler, autocast
import json
import os

# 1. Architecture & Dataset Definitions
class ASLUpgradedDataset(IterableDataset):
    def __init__(self, file_paths, phrase_map_path, limit=None):
        self.file_paths = file_paths
        self.limit = limit
        with open(phrase_map_path, 'r') as f:
            data = json.load(f)
            self.label_map = {item['label']: i for i, item in enumerate(data['labels'])}
    def __iter__(self):
        count = 0
        for file_path in self.file_paths:
            if not os.path.exists(file_path): continue
            with open(file_path, 'r') as f:
                for line in f:
                    try:
                        if self.limit and count >= self.limit: return
                        data = json.loads(line)
                        label = self.label_map[data['label']]
                        frames = torch.tensor(data['frames'], dtype=torch.float32)
                        frames_3d = frames.view(frames.size(0), 543, 3)
                        nose = frames_3d[:, 468:469, :]
                        frames_3d = frames_3d - nose
                        frames = frames_3d.view(frames.size(0), 1629)
                        diff1 = torch.cat([frames[1:] - frames[:-1], torch.zeros(1, 1629)], dim=0)
                        diff2 = torch.cat([frames[2:] - frames[:-2], torch.zeros(2, 1629)], dim=0)
                        yield torch.cat([frames, diff1, diff2], dim=-1), label
                        count += 1
                    except Exception: continue

class LandmarkTransformerUpgraded(nn.Module):
    def __init__(self, num_classes=250, input_dim=4887, model_dim=256, num_heads=8, num_layers=4, dropout=0.1):
        super().__init__()
        self.input_projection = nn.Linear(input_dim, model_dim)
        self.pos_encoder = nn.Parameter(torch.randn(1, 30, model_dim))
        encoder_layer = nn.TransformerEncoderLayer(d_model=model_dim, nhead=num_heads, dim_feedforward=model_dim * 4, dropout=dropout, batch_first=True, norm_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.classifier = nn.Linear(model_dim, num_classes)
    def forward(self, x):
        x = self.input_projection(x) + self.pos_encoder
        x = self.transformer(x).mean(dim=1)
        return self.classifier(x)

# 2. Setup Device & Model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = LandmarkTransformerUpgraded().to(device)

# Hyperparameters
MAX_EPOCHS = 45
START_EPOCH = 41 # Resuming from heuristic

optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)
criterion = nn.CrossEntropyLoss()
scaler = GradScaler('cuda', enabled=(device.type == 'cuda'))

# Restore latest weights
weights_path = '/content/drive/MyDrive/asl_model_upgraded.pth'
if os.path.exists(weights_path):
    try:
        state = torch.load(weights_path, map_location=device)
        # Handle both state_dict and checkpoint dict
        model.load_state_dict(state if 'input_projection.weight' in state else state['model_state_dict'])
        print(f'✅ Weights loaded. Resuming training on {device}...')
    except Exception as e: print(f'⚠️ Load failed: {e}')

# Data Loader
drive_files = ['/content/drive/MyDrive/asl_train_data.jsonl', '/content/drive/MyDrive/asl_train_data_part2.jsonl']
phrase_map_path = '/content/drive/MyDrive/phrase_map.json'
asl_dataset = ASLUpgradedDataset(drive_files, phrase_map_path)
asl_loader = DataLoader(asl_dataset, batch_size=64, num_workers=2 if device.type == 'cuda' else 0)

# 3. Enhanced Training Loop (with Metadata)
best_acc = 0.0
for epoch in range(START_EPOCH, MAX_EPOCHS):
    model.train()
    running_loss, running_acc = 0.0, 0.0
    for batch_idx, (frames, labels) in enumerate(asl_loader):
        frames, labels = frames.to(device), labels.to(device)
        frames = torch.nan_to_num(frames, nan=0.0)

        optimizer.zero_grad(set_to_none=True)
        with autocast('cuda', enabled=(device.type == 'cuda')):
            outputs = model(frames)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        acc = (outputs.argmax(1) == labels).sum().item() / labels.size(0)
        running_loss += loss.item()
        running_acc += acc

        if (batch_idx + 1) % 100 == 0:
            avg_loss, avg_acc = running_loss/100, running_acc/100
            print(f'Epoch [{epoch+1}], Step [{batch_idx+1}], Loss: {avg_loss:.4f}, Acc: {avg_acc:.4f}')

            checkpoint = {
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'loss': avg_loss,
                'accuracy': avg_acc
            }

            torch.save(checkpoint, weights_path)
            if avg_acc > best_acc:
                best_acc = avg_acc
                torch.save(checkpoint, '/content/drive/MyDrive/asl_model_upgraded_best.pth')

            running_loss, running_acc = 0.0, 0.0
    print(f'Epoch {epoch+1} complete.')

/tmp/ipykernel_13732/767969992.py:44: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)


✅ Weights loaded. Resuming training on cpu...
Epoch [42], Step [100], Loss: 3.6724, Acc: 0.2969
Epoch [42], Step [200], Loss: 3.6442, Acc: 0.2995
Epoch [42], Step [300], Loss: 3.6333, Acc: 0.3031
Epoch [42], Step [400], Loss: 3.6836, Acc: 0.2930
Epoch [42], Step [500], Loss: 3.6805, Acc: 0.2970
Epoch [42], Step [600], Loss: 3.6049, Acc: 0.3127
Epoch [42], Step [700], Loss: 3.6338, Acc: 0.3097
Epoch [42], Step [800], Loss: 3.4907, Acc: 0.3467
Epoch [42], Step [900], Loss: 3.2646, Acc: 0.3914
Epoch [42], Step [1000], Loss: 3.2483, Acc: 0.3952
Epoch [42], Step [1100], Loss: 3.2820, Acc: 0.3911
Epoch [42], Step [1200], Loss: 3.1885, Acc: 0.4047
Epoch [42], Step [1300], Loss: 3.2139, Acc: 0.4081
Epoch [42], Step [1400], Loss: 3.3257, Acc: 0.3886
Epoch 42 complete.
Epoch [43], Step [100], Loss: 3.6106, Acc: 0.3086
Epoch [43], Step [200], Loss: 3.5824, Acc: 0.3178
Epoch [43], Step [300], Loss: 3.5766, Acc: 0.3180


### GPU-Accelerated Training Resumption
We are now utilizing the Tesla T4 GPU with Mixed Precision (FP16) to reach our performance targets faster.

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, IterableDataset
from torch.amp import GradScaler, autocast
import json
import os

# 1. Architecture and Dataset Definitions
class LandmarkTransformerUpgraded(nn.Module):
    def __init__(self, num_classes=250, input_dim=4887, model_dim=256, num_heads=8, num_layers=4, dropout=0.1):
        super().__init__()
        self.input_projection = nn.Linear(input_dim, model_dim)
        self.pos_encoder = nn.Parameter(torch.randn(1, 30, model_dim))
        encoder_layer = nn.TransformerEncoderLayer(d_model=model_dim, nhead=num_heads, dim_feedforward=model_dim * 4, dropout=dropout, batch_first=True, norm_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.classifier = nn.Linear(model_dim, num_classes)
    def forward(self, x):
        x = self.input_projection(x) + self.pos_encoder
        x = self.transformer(x).mean(dim=1)
        return self.classifier(x)

class ASLUpgradedDataset(IterableDataset):
    def __init__(self, file_paths, phrase_map_path):
        self.file_paths = file_paths
        with open(phrase_map_path, 'r') as f:
            data = json.load(f)
            self.label_map = {item['label']: i for i, item in enumerate(data['labels'])}
    def __iter__(self):
        for file_path in self.file_paths:
            if not os.path.exists(file_path): continue
            with open(file_path, 'r') as f:
                for line in f:
                    try:
                        data = json.loads(line)
                        label = self.label_map[data['label']]
                        frames = torch.tensor(data['frames'], dtype=torch.float32)
                        frames_3d = frames.view(frames.size(0), 543, 3)
                        nose = frames_3d[:, 468:469, :]
                        frames_3d = frames_3d - nose
                        frames = frames_3d.view(frames.size(0), 1629)
                        diff1 = torch.cat([frames[1:] - frames[:-1], torch.zeros(1, 1629)], dim=0)
                        diff2 = torch.cat([frames[2:] - frames[:-2], torch.zeros(2, 1629)], dim=0)
                        yield torch.cat([frames, diff1, diff2], dim=-1), label
                    except Exception: continue

# 2. Setup GPU and Hardware
device = torch.device('cuda')
torch.backends.cudnn.benchmark = True

# 3. Initialize Model and Loader
model = LandmarkTransformerUpgraded().to(device)
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)
criterion = nn.CrossEntropyLoss()
scaler = GradScaler('cuda')

drive_files = ['/content/drive/MyDrive/asl_train_data.jsonl', '/content/drive/MyDrive/asl_train_data_part2.jsonl']
phrase_map_path = '/content/drive/MyDrive/phrase_map.json'
asl_dataset = ASLUpgradedDataset(drive_files, phrase_map_path)
asl_loader_gpu = DataLoader(asl_dataset, batch_size=128, num_workers=2, pin_memory=True)

# 4. Load latest state from Drive
weights_path = '/content/drive/MyDrive/asl_model_upgraded.pth'
if os.path.exists(weights_path):
    state = torch.load(weights_path, map_location=device)
    model.load_state_dict(state if 'input_projection.weight' in state else state['model_state_dict'])
    print('✅ Model loaded on GPU. Starting high-speed training...')

# 5. Training Loop
TARGET_ACC = 0.45
model.train()
for epoch in range(44, 60):
    running_loss, running_acc = 0.0, 0.0
    for batch_idx, (frames, labels) in enumerate(asl_loader_gpu):
        frames, labels = frames.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        frames = torch.nan_to_num(frames, nan=0.0)

        optimizer.zero_grad(set_to_none=True)
        with autocast('cuda'):
            outputs = model(frames)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        acc = (outputs.argmax(1) == labels).sum().item() / labels.size(0)
        running_loss += loss.item()
        running_acc += acc

        if (batch_idx + 1) % 100 == 0:
            avg_acc = running_acc / 100
            print(f'Epoch [{epoch+1}], Step [{batch_idx+1}], Loss: {running_loss/100:.4f}, Acc: {avg_acc:.4f}')
            torch.save(model.state_dict(), weights_path)
            if avg_acc >= TARGET_ACC:
                print(f'⌖ Target accuracy {TARGET_ACC} reached!')
                torch.save(model.state_dict(), "/content/drive/MyDrive/asl_model_upgraded_best.pth")
                break
            running_loss, running_acc = 0.0, 0.0
    print(f'Epoch {epoch+1} complete.')

/tmp/ipykernel_1952/2663953360.py:16: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)


✅ Model loaded on GPU. Starting high-speed training...
Epoch [45], Step [100], Loss: 3.2882, Acc: 0.3880
Epoch [45], Step [200], Loss: 3.2282, Acc: 0.4023
Epoch [45], Step [300], Loss: 3.2880, Acc: 0.3883
Epoch [45], Step [400], Loss: 3.4052, Acc: 0.3523
Epoch [45], Step [500], Loss: 3.4020, Acc: 0.3611
Epoch [45], Step [600], Loss: 3.3371, Acc: 0.3680
Epoch [45], Step [700], Loss: 3.5061, Acc: 0.3278
Epoch [45], Step [800], Loss: 3.4212, Acc: 0.3557
Epoch [45], Step [900], Loss: 3.2238, Acc: 0.3957
Epoch [45], Step [1000], Loss: 3.2232, Acc: 0.3941
Epoch [45], Step [1100], Loss: 3.2521, Acc: 0.3888
Epoch [45], Step [1200], Loss: 3.1565, Acc: 0.4127
Epoch [45], Step [1300], Loss: 3.2434, Acc: 0.3880
Epoch [45], Step [1400], Loss: 3.3512, Acc: 0.3717
Epoch 45 complete.
Epoch [46], Step [100], Loss: 3.2711, Acc: 0.3871
Epoch [46], Step [200], Loss: 3.2120, Acc: 0.4048
Epoch [46], Step [300], Loss: 3.2649, Acc: 0.3943
Epoch [46], Step [400], Loss: 3.3549, Acc: 0.3679
Epoch [46], Step [500

KeyboardInterrupt: 

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, IterableDataset
from torch.amp import GradScaler, autocast
import json
import os

# 1. Define Architecture and Dataset Class
class LandmarkTransformerUpgraded(nn.Module):
    def __init__(self, num_classes=250, input_dim=4887, model_dim=256, num_heads=8, num_layers=4, dropout=0.1):
        super().__init__()
        self.input_projection = nn.Linear(input_dim, model_dim)
        self.pos_encoder = nn.Parameter(torch.randn(1, 30, model_dim))
        encoder_layer = nn.TransformerEncoderLayer(d_model=model_dim, nhead=num_heads, dim_feedforward=model_dim * 4, dropout=dropout, batch_first=True, norm_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.classifier = nn.Linear(model_dim, num_classes)
    def forward(self, x):
        x = self.input_projection(x) + self.pos_encoder
        x = self.transformer(x).mean(dim=1)
        return self.classifier(x)

class ASLUpgradedDataset(IterableDataset):
    def __init__(self, file_paths, phrase_map_path):
        self.file_paths = file_paths
        with open(phrase_map_path, 'r') as f:
            data = json.load(f)
            self.label_map = {item['label']: i for i, item in enumerate(data['labels'])}
    def __iter__(self):
        for file_path in self.file_paths:
            if not os.path.exists(file_path): continue
            with open(file_path, 'r') as f:
                for line in f:
                    try:
                        data = json.loads(line)
                        label = self.label_map[data['label']]
                        frames = torch.tensor(data['frames'], dtype=torch.float32)
                        frames_3d = frames.view(frames.size(0), 543, 3)
                        nose = frames_3d[:, 468:469, :]
                        frames_3d = frames_3d - nose
                        frames = frames_3d.view(frames.size(0), 1629)
                        diff1 = torch.cat([frames[1:] - frames[:-1], torch.zeros(1, 1629)], dim=0)
                        diff2 = torch.cat([frames[2:] - frames[:-2], torch.zeros(2, 1629)], dim=0)
                        yield torch.cat([frames, diff1, diff2], dim=-1), label
                    except Exception: continue

# 2. Setup GPU and Hardware
device = torch.device('cuda')
torch.backends.cudnn.benchmark = True

# 3. Initialize Model and Loader
model = LandmarkTransformerUpgraded().to(device)
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)
criterion = nn.CrossEntropyLoss()
scaler = GradScaler('cuda')

drive_files = ['/content/drive/MyDrive/asl_train_data.jsonl', '/content/drive/MyDrive/asl_train_data_part2.jsonl']
phrase_map_path = '/content/drive/MyDrive/phrase_map.json'
asl_dataset = ASLUpgradedDataset(drive_files, phrase_map_path)

# 4. Load latest state from Drive
weights_path = '/content/drive/MyDrive/asl_model_upgraded.pth'
if os.path.exists(weights_path):
    state = torch.load(weights_path, map_location=device)
    model.load_state_dict(state if 'input_projection.weight' in state else state['model_state_dict'])
    print('✅ Model loaded on GPU.')

# 5. Training Loop
TARGET_ACC = 0.45
START_EPOCH = 41

model.train()
for epoch in range(START_EPOCH, 50):
    running_loss, running_acc = 0.0, 0.0
    asl_loader_gpu = DataLoader(asl_dataset, batch_size=128, num_workers=2, pin_memory=True)

    for batch_idx, (frames, labels) in enumerate(asl_loader_gpu):
        frames, labels = frames.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        frames = torch.nan_to_num(frames, nan=0.0)

        optimizer.zero_grad(set_to_none=True)
        with autocast('cuda'):
            outputs = model(frames)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        acc = (outputs.argmax(1) == labels).sum().item() / labels.size(0)
        running_loss += loss.item()
        running_acc += acc

        if (batch_idx + 1) % 100 == 0:
            avg_acc = running_acc / 100
            print(f'Epoch [{epoch+1}], Step [{batch_idx+1}], Loss: {running_loss/100:.4f}, Acc: {avg_acc:.4f}')

            torch.save(model.state_dict(), weights_path)
            if avg_acc >= TARGET_ACC:
                print(f'፠ Target accuracy {TARGET_ACC} reached!')
                torch.save(model.state_dict(), "/content/drive/MyDrive/asl_model_upgraded_best.pth")
                break
            running_loss, running_acc = 0.0, 0.0
    print(f'Epoch {epoch+1} complete.')

/tmp/ipykernel_783/213915568.py:16: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)


✅ Model loaded on GPU.
Epoch [42], Step [100], Loss: 3.3519, Acc: 0.3781
Epoch [42], Step [200], Loss: 3.3621, Acc: 0.3720
Epoch [42], Step [300], Loss: 3.5504, Acc: 0.3225


In [ ]:
def quick_evaluate(model, dataset, num_batches=20):
    model.eval()
    total_acc = 0.0
    loader = DataLoader(dataset, batch_size=64, num_workers=0)
    print(f'Evaluating current model state over {num_batches} batches...')
    with torch.no_grad():
        for i, (frames, labels) in enumerate(loader):
            if i >= num_batches: break
            frames, labels = frames.to(device), labels.to(device)
            outputs = model(frames)
            acc = (outputs.argmax(1) == labels).sum().item() / labels.size(0)
            total_acc += acc
    avg_acc = (total_acc / num_batches) * 100
    print(f'\n--- Current Accuracy: {avg_acc:.2f}% ---')
    return avg_acc

# Run evaluation
current_val_acc = quick_evaluate(model, asl_dataset)

NameError: name 'model' is not defined

In [ ]:
import torch
import os

def detailed_inspect(path):
    if not os.path.exists(path):
        return f'{path} not found.'

    checkpoint = torch.load(path, map_location='cpu')

    if isinstance(checkpoint, dict):
        keys = list(checkpoint.keys())
        # Check if it's just a state_dict or a full checkpoint
        is_state_dict = any('weight' in k or 'bias' in k for k in keys)

        metadata = {k: v for k, v in checkpoint.items() if not isinstance(v, (torch.Tensor, dict))}

        return {
            'path': path,
            'is_state_dict_only': is_state_dict and len(metadata) == 0,
            'metadata': metadata,
            'keys': keys[:5] # Show first few keys
        }
    return 'Not a dictionary.'

upgraded_path = '/content/drive/MyDrive/asl_model_upgraded.pth'
best_upgraded_path = '/content/drive/MyDrive/asl_model_upgraded_best.pth'

print('--- Upgraded Model Analysis ---')
display(detailed_inspect(upgraded_path))

print('\n--- Best Upgraded Model Analysis ---')
display(detailed_inspect(best_upgraded_path))

--- Upgraded Model Analysis ---


{'path': '/content/drive/MyDrive/asl_model_upgraded.pth',
 'is_state_dict_only': True,
 'metadata': {},
 'keys': ['pos_encoder',
  'input_projection.weight',
  'input_projection.bias',
  'transformer.layers.0.self_attn.in_proj_weight',
  'transformer.layers.0.self_attn.in_proj_bias']}


--- Best Upgraded Model Analysis ---


{'path': '/content/drive/MyDrive/asl_model_upgraded_best.pth',
 'is_state_dict_only': True,
 'metadata': {},
 'keys': ['pos_encoder',
  'input_projection.weight',
  'input_projection.bias',
  'transformer.layers.0.self_attn.in_proj_weight',
  'transformer.layers.0.self_attn.in_proj_bias']}

In [ ]:
import torch
import os

def inspect_checkpoint(path, name):
    if os.path.exists(path):
        print(f'--- Inspecting {name} ({path}) ---')
        checkpoint = torch.load(path, map_location='cpu')
        if isinstance(checkpoint, dict):
            # Check for common metadata keys
            found_metadata = False
            for key in ['epoch', 'loss', 'accuracy', 'batch_idx', 'best_loss', 'step']:
                if key in checkpoint:
                    print(f'{key.capitalize()}: {checkpoint[key]}')
                    found_metadata = True
            if not found_metadata:
                print('Note: File contains a dictionary, but no known training metadata keys found.')
                print(f'Keys available: {list(checkpoint.keys())}')
        else:
            print('Note: File is a direct state_dict/tensor and does not contain a metadata dictionary.')
    else:
        print(f'❌ {name} not found at {path}')
    print('\n')

# Paths to check
models = [
    ('/content/drive/MyDrive/asl_model_upgraded.pth', 'Primary Model'),
    ('/content/drive/MyDrive/asl_model_upgraded_best.pth', 'Best Model'),
    ('/content/drive/MyDrive/asl_model_best.pth', 'Legacy Best Model')
]

for path, name in models:
    inspect_checkpoint(path, name)

--- Inspecting Primary Model (/content/drive/MyDrive/asl_model_upgraded.pth) ---
Note: File contains a dictionary, but no known training metadata keys found.
Keys available: ['pos_encoder', 'input_projection.weight', 'input_projection.bias', 'transformer.layers.0.self_attn.in_proj_weight', 'transformer.layers.0.self_attn.in_proj_bias', 'transformer.layers.0.self_attn.out_proj.weight', 'transformer.layers.0.self_attn.out_proj.bias', 'transformer.layers.0.linear1.weight', 'transformer.layers.0.linear1.bias', 'transformer.layers.0.linear2.weight', 'transformer.layers.0.linear2.bias', 'transformer.layers.0.norm1.weight', 'transformer.layers.0.norm1.bias', 'transformer.layers.0.norm2.weight', 'transformer.layers.0.norm2.bias', 'transformer.layers.1.self_attn.in_proj_weight', 'transformer.layers.1.self_attn.in_proj_bias', 'transformer.layers.1.self_attn.out_proj.weight', 'transformer.layers.1.self_attn.out_proj.bias', 'transformer.layers.1.linear1.weight', 'transformer.layers.1.linear1.bias'

In [ ]:
import torch
import os

# Checking the 'best' model which often contains full checkpoint metadata
path_best = '/content/drive/MyDrive/asl_model_upgraded_best.pth'
if os.path.exists(path_best):
    checkpoint = torch.load(path_best, map_location='cpu')
    print('--- Best Model Checkpoint Progress ---')
    if isinstance(checkpoint, dict):
        for key in ['epoch', 'loss', 'accuracy', 'batch_idx']:
            if key in checkpoint:
                print(f'{key.capitalize()}: {checkpoint[key]}')
        if 'epoch' not in checkpoint:
            print('Note: Best model file also contains state_dict (weights) only.')
    else:
        print('Best model file is a direct model state_dict.')
else:
    print('❌ Best checkpoint file not found.')

--- Best Model Checkpoint Progress ---
Note: Best model file also contains state_dict (weights) only.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import IterableDataset, DataLoader
import json
import os

# Set CPU threads to match the runtime for maximum efficiency
torch.set_num_threads(os.cpu_count())

# 1. Architecture & Dataset Definitions
class ASLUpgradedDataset(IterableDataset):
    def __init__(self, file_paths, phrase_map_path):
        self.file_paths = file_paths
        with open(phrase_map_path, 'r') as f:
            data = json.load(f)
            self.label_map = {item['label']: i for i, item in enumerate(data['labels'])}
    def __iter__(self):
        for file_path in self.file_paths:
            if not os.path.exists(file_path): continue
            with open(file_path, 'r') as f:
                for line in f:
                    try:
                        data = json.loads(line)
                        label = self.label_map[data['label']]
                        frames = torch.tensor(data['frames'], dtype=torch.float32)
                        frames_3d = frames.view(frames.size(0), 543, 3)
                        nose = frames_3d[:, 468:469, :]
                        frames_3d = frames_3d - nose
                        frames = frames_3d.view(frames.size(0), 1629)
                        diff1 = torch.cat([frames[1:] - frames[:-1], torch.zeros(1, 1629)], dim=0)
                        diff2 = torch.cat([frames[2:] - frames[:-2], torch.zeros(2, 1629)], dim=0)
                        yield torch.cat([frames, diff1, diff2], dim=-1), label
                    except Exception: continue

class LandmarkTransformerUpgraded(nn.Module):
    def __init__(self, num_classes=250, input_dim=4887, model_dim=256, num_heads=8, num_layers=4, dropout=0.1):
        super().__init__()
        self.input_projection = nn.Linear(input_dim, model_dim)
        self.pos_encoder = nn.Parameter(torch.randn(1, 30, model_dim))
        encoder_layer = nn.TransformerEncoderLayer(d_model=model_dim, nhead=num_heads, dim_feedforward=model_dim * 4, dropout=dropout, batch_first=True, norm_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.classifier = nn.Linear(model_dim, num_classes)
    def forward(self, x):
        x = self.input_projection(x) + self.pos_encoder
        x = self.transformer(x).mean(dim=1)
        return self.classifier(x)

# 2. Setup Device and Model
device = torch.device('cpu')
model = LandmarkTransformerUpgraded().to(device)

# 3. Data Loader Configuration
drive_files = ['/content/drive/MyDrive/asl_train_data.jsonl', '/content/drive/MyDrive/asl_train_data_part2.jsonl']
phrase_map_path = '/content/drive/MyDrive/phrase_map.json'
asl_dataset = ASLUpgradedDataset(drive_files, phrase_map_path)
asl_loader = DataLoader(asl_dataset, batch_size=32, num_workers=0)

# 4. Optimizer and Criterion
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)
criterion = nn.CrossEntropyLoss()

# 5. Load the latest checkpoint
weights_path = '/content/drive/MyDrive/asl_model_upgraded.pth'
start_epoch = 39
start_step = 650 # Resuming from where the kernel left off
best_acc = 0.4061

if os.path.exists(weights_path):
    try:
        state = torch.load(weights_path, map_location=device)
        model.load_state_dict(state if isinstance(state, dict) and 'input_projection.weight' in state else state['model_state_dict'])
        print(f'✅ Weights loaded. Resuming training from Epoch {start_epoch+1}, Step {start_step} on {device}...')
    except Exception as e:
        print(f'☀ Failed to load weights: {e}')

# 6. Optimized CPU Training Loop
model.train()
for epoch in range(start_epoch, 50):
    running_loss, running_acc = 0.0, 0.0
    for batch_idx, (frames, labels) in enumerate(asl_loader):
        # Skip already processed steps in the current epoch
        if epoch == start_epoch and batch_idx < start_step:
            continue

        frames, labels = frames.to(device), labels.to(device)
        frames = torch.nan_to_num(frames, nan=0.0)

        optimizer.zero_grad(set_to_none=True)
        outputs = model(frames)
        loss = criterion(outputs, labels)

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        acc = (outputs.argmax(1) == labels).sum().item() / labels.size(0)
        running_loss += loss.item()
        running_acc += acc

        if (batch_idx + 1) % 50 == 0:
            avg_acc = running_acc / 50
            print(f'Epoch [{epoch+1}], Step [{batch_idx+1}], Loss: {running_loss/50:.4f}, Acc: {avg_acc:.4f}')

            if avg_acc > best_acc:
                best_acc = avg_acc
                torch.save(model.state_dict(), '/content/drive/MyDrive/asl_model_upgraded_best.pth')
                print(f'✨ New Best Accuracy: {best_acc:.4%}')

            if avg_acc >= 0.45:
                print("⁂ Target accuracy reached!")

            running_loss, running_acc = 0.0, 0.0

    torch.save(model.state_dict(), weights_path)
    print(f'Epoch {epoch+1} complete.')

/tmp/ipykernel_8535/3227968901.py:39: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)


✅ Weights loaded. Resuming training from Epoch 40 on cpu...
Epoch [40], Step [50], Loss: 3.7583, Acc: 0.2775
Epoch [40], Step [100], Loss: 3.7429, Acc: 0.2825
Epoch [40], Step [150], Loss: 3.7442, Acc: 0.2836
Epoch [40], Step [200], Loss: 3.7190, Acc: 0.2895
Epoch [40], Step [250], Loss: 3.6860, Acc: 0.2939
Epoch [40], Step [300], Loss: 3.7592, Acc: 0.2750
Epoch [40], Step [350], Loss: 3.7502, Acc: 0.2744
Epoch [40], Step [400], Loss: 3.7921, Acc: 0.2755
Epoch [40], Step [450], Loss: 3.7899, Acc: 0.2683
Epoch [40], Step [500], Loss: 3.7441, Acc: 0.2842
Epoch [40], Step [550], Loss: 3.7122, Acc: 0.2900
Epoch [40], Step [600], Loss: 3.6804, Acc: 0.2984
Epoch [40], Step [650], Loss: 3.7657, Acc: 0.2814
Epoch [40], Step [700], Loss: 3.6725, Acc: 0.3006
Epoch [40], Step [750], Loss: 3.7173, Acc: 0.2920
Epoch [40], Step [800], Loss: 3.4352, Acc: 0.3525
Epoch [40], Step [850], Loss: 3.2704, Acc: 0.3828
Epoch [40], Step [900], Loss: 3.3766, Acc: 0.3688
Epoch [40], Step [950], Loss: 3.3013, Acc

In [ ]:
import torch
import os

# 1. Check GPU Availability
gpu_available = torch.cuda.is_available()
gpu_name = torch.cuda.get_device_name(0) if gpu_available else 'None'
print(f'--- Hardware Check ---')
print(f'GPU Available: {gpu_available}')
if gpu_available: print(f'GPU Type: {gpu_name}')

# 2. Check Training Artifacts on Drive
print('\n--- Training Artifacts Check ---')
paths = {
    'Final Model': '/content/drive/MyDrive/asl_model_upgraded.pth',
    'Best Model': '/content/drive/MyDrive/asl_model_upgraded_best.pth',
    'Data Part 1': '/content/drive/MyDrive/asl_train_data.jsonl',
    'Data Part 2': '/content/drive/MyDrive/asl_train_data_part2.jsonl',
    'Phrase Map': '/content/drive/MyDrive/phrase_map.json'
}

for name, path in paths.items():
    if os.path.exists(path):
        size = os.path.getsize(path) / (1024**3)
        print(f'✅ {name}: Found ({size:.2f} GB)')
    else:
        print(f'❌ {name}: Missing')

# 3. Quick Model Summary (if weights exist)
if os.path.exists(paths['Final Model']):
    checkpoint = torch.load(paths['Final Model'], map_location='cpu')
    # Note: Depending on how it was saved, it might be a state_dict or a full checkpoint
    if isinstance(checkpoint, dict) and 'epoch' in checkpoint:
        print(f'\nLast Session Epoch: {checkpoint["epoch"] + 1}')
        if 'loss' in checkpoint: print(f'Last Loss: {checkpoint["loss"]:.4f}')

--- Hardware Check ---
GPU Available: True
GPU Type: Tesla T4

--- Training Artifacts Check ---
✅ Final Model: Found (0.02 GB)
✅ Best Model: Found (0.02 GB)
✅ Data Part 1: Found (33.08 GB)
✅ Data Part 2: Found (29.36 GB)
✅ Phrase Map: Found (0.00 GB)


In [ ]:
import torch
import os

# Quick validation of the latest best accuracy recorded
weights_path = '/content/drive/MyDrive/asl_model_upgraded_best.pth'
if os.path.exists(weights_path):
    print(f'✅ Best model found at: {weights_path}')
    # We know from logs the last best was ~40.6%, let's confirm the file is healthy
    file_size = os.path.getsize(weights_path) / (1024**2)
    print(f'Model Size: {file_size:.2f} MB')
else:
    print('☀ No best model file found yet on Drive.')

print('\nNext Step: Continue training from Epoch 39 to reach the 45% target.')

✅ Best model found at: /content/drive/MyDrive/asl_model_upgraded_best.pth
Model Size: 17.12 MB

Next Step: Continue training from Epoch 39 to reach the 45% target.


In [ ]:
import torch
import os

# Define the paths for the models we were tracking
final_model_path = '/content/drive/MyDrive/asl_model_upgraded.pth'
best_model_path = '/content/drive/MyDrive/asl_model_upgraded_best.pth'

print('--- GPU Progress Analysis ---')

if os.path.exists(final_model_path):
    # Load state dict to check if it's a full checkpoint or just weights
    data = torch.load(final_model_path, map_location='cpu')

    # If the saving logic was weights only, we check the 'best' model as well
    if os.path.exists(best_model_path):
        best_data = torch.load(best_model_path, map_location='cpu')
        print(f'✅ Best Model Checkpoint exists.')

    # Based on the last run logs, we were in Epoch 32 reaching ~39% accuracy.
    # Let\'s look at the filesystem timestamps to see when they were last updated.
    import datetime
    final_time = datetime.datetime.fromtimestamp(os.path.getmtime(final_model_path))
    best_time = datetime.datetime.fromtimestamp(os.path.getmtime(best_model_path)) if os.path.exists(best_model_path) else 'N/A'

    print(f'Last Checkpoint Update: {final_time}')
    print(f'Best Model Update: {best_time}')

    print('\nSummary based on logs:\n- You successfully transitioned to GPU (Tesla T4).\n- Epoch 31 completed fully.\n- Epoch 32 was in progress, reaching an accuracy of approximately 38.94% at Step 1200.\n- The target was 45% to trigger a new \'best_model\' save.')
else:
    print('❌ No model files found to analyze progress.')

--- GPU Progress Analysis ---
✅ Best Model Checkpoint exists.
Last Checkpoint Update: 2026-05-09 10:09:15
Best Model Update: 2026-05-08 06:46:10

Summary based on logs:
- You successfully transitioned to GPU (Tesla T4).
- Epoch 31 completed fully.
- Epoch 32 was in progress, reaching an accuracy of approximately 38.94% at Step 1200.
- The target was 45% to trigger a new 'best_model' save.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, IterableDataset
import json
import os

# 1. Self-contained Definitions
class LandmarkTransformerUpgraded(nn.Module):
    def __init__(self, num_classes=250, input_dim=4887, model_dim=256, num_heads=8, num_layers=4, dropout=0.1):
        super().__init__()
        self.input_projection = nn.Linear(input_dim, model_dim)
        self.pos_encoder = nn.Parameter(torch.randn(1, 30, model_dim))
        encoder_layer = nn.TransformerEncoderLayer(d_model=model_dim, nhead=num_heads, dim_feedforward=model_dim * 4, dropout=dropout, batch_first=True, norm_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.classifier = nn.Linear(model_dim, num_classes)
    def forward(self, x):
        x = self.input_projection(x) + self.pos_encoder
        x = self.transformer(x).mean(dim=1)
        return self.classifier(x)

class ASLUpgradedDataset(IterableDataset):
    def __init__(self, file_paths, phrase_map_path):
        self.file_paths = file_paths
        with open(phrase_map_path, 'r') as f:
            data = json.load(f)
            self.label_map = {item['label']: i for i, item in enumerate(data['labels'])}
    def __iter__(self):
        for file_path in self.file_paths:
            if not os.path.exists(file_path): continue
            with open(file_path, 'r') as f:
                for line in f:
                    try:
                        data = json.loads(line)
                        label = self.label_map[data['label']]
                        frames = torch.tensor(data['frames'], dtype=torch.float32)
                        frames_3d = frames.view(frames.size(0), 543, 3)
                        nose = frames_3d[:, 468:469, :]
                        frames_3d = frames_3d - nose
                        frames = frames_3d.view(frames.size(0), 1629)
                        diff1 = torch.cat([frames[1:] - frames[:-1], torch.zeros(1, 1629)], dim=0)
                        diff2 = torch.cat([frames[2:] - frames[:-2], torch.zeros(2, 1629)], dim=0)
                        yield torch.cat([frames, diff1, diff2], dim=-1), label
                    except Exception: continue

# 2. Setup CPU Optimized Environment
device = torch.device('cpu')
torch.set_num_threads(os.cpu_count())
model = LandmarkTransformerUpgraded().to(device)
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)
criterion = nn.CrossEntropyLoss()

# 3. Load Checkpoint
weights_path = '/content/drive/MyDrive/asl_model_upgraded.pth'
if os.path.exists(weights_path):
    state = torch.load(weights_path, map_location=device)
    model.load_state_dict(state if 'input_projection.weight' in state else state['model_state_dict'])
    print('✅ Resuming on CPU from last checkpoint.')

# 4. Resume Loop
drive_files = ['/content/drive/MyDrive/asl_train_data.jsonl', '/content/drive/MyDrive/asl_train_data_part2.jsonl']
phrase_map_path = '/content/drive/MyDrive/phrase_map.json'
asl_dataset = ASLUpgradedDataset(drive_files, phrase_map_path)
asl_loader = DataLoader(asl_dataset, batch_size=32, num_workers=0)

model.train()
for epoch in range(41, 50):
    running_loss, running_acc = 0.0, 0.0
    for batch_idx, (frames, labels) in enumerate(asl_loader):
        frames, labels = frames.to(device), labels.to(device)
        frames = torch.nan_to_num(frames, nan=0.0)
        optimizer.zero_grad()
        outputs = model(frames)
        loss = criterion(outputs, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        acc = (outputs.argmax(1) == labels).sum().item() / labels.size(0)
        running_loss += loss.item()
        running_acc += acc
        if (batch_idx + 1) % 50 == 0:
            avg_acc = running_acc / 50
            print(f'Epoch [{epoch+1}], Step [{batch_idx+1}], Loss: {running_loss/50:.4f}, Acc: {avg_acc:.4f}')
            torch.save(model.state_dict(), weights_path)
            if avg_acc >= 0.45:
                torch.save(model.state_dict(), '/content/drive/MyDrive/asl_model_upgraded_best.pth')
                print('🎯 Target reached!')
                break
            running_loss, running_acc = 0.0, 0.0

/tmp/ipykernel_843/943558385.py:15: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)


✅ Resuming on CPU from last checkpoint.
Epoch [42], Step [50], Loss: 3.3322, Acc: 0.3812
Epoch [42], Step [100], Loss: 3.3182, Acc: 0.3794
Epoch [42], Step [150], Loss: 3.2699, Acc: 0.3881
Epoch [42], Step [200], Loss: 3.3468, Acc: 0.3850
Epoch [42], Step [250], Loss: 3.3626, Acc: 0.3750
Epoch [42], Step [300], Loss: 3.2883, Acc: 0.3881
Epoch [42], Step [350], Loss: 3.3336, Acc: 0.3862
Epoch [42], Step [400], Loss: 3.2114, Acc: 0.4056
Epoch [42], Step [450], Loss: 3.3523, Acc: 0.3663
Epoch [42], Step [500], Loss: 3.3909, Acc: 0.3663
Epoch [42], Step [550], Loss: 3.4619, Acc: 0.3531
Epoch [42], Step [600], Loss: 3.4656, Acc: 0.3431
Epoch [42], Step [650], Loss: 3.7581, Acc: 0.2775
Epoch [42], Step [700], Loss: 3.6250, Acc: 0.2956
Epoch [42], Step [750], Loss: 3.7439, Acc: 0.2806
Epoch [42], Step [800], Loss: 3.6919, Acc: 0.2869
Epoch [42], Step [850], Loss: 3.6725, Acc: 0.2988
Epoch [42], Step [900], Loss: 3.7407, Acc: 0.2856
Epoch [42], Step [950], Loss: 3.5669, Acc: 0.3063
Epoch [42],

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, IterableDataset
import json
import os

# 1. Architecture & Dataset Definitions
class LandmarkTransformerUpgraded(nn.Module):
    def __init__(self, num_classes=250, input_dim=4887, model_dim=256, num_heads=8, num_layers=4, dropout=0.1):
        super().__init__()
        self.input_projection = nn.Linear(input_dim, model_dim)
        self.pos_encoder = nn.Parameter(torch.randn(1, 30, model_dim))
        encoder_layer = nn.TransformerEncoderLayer(d_model=model_dim, nhead=num_heads, dim_feedforward=model_dim * 4, dropout=dropout, batch_first=True, norm_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.classifier = nn.Linear(model_dim, num_classes)
    def forward(self, x):
        x = self.input_projection(x) + self.pos_encoder
        x = self.transformer(x).mean(dim=1)
        return self.classifier(x)

class ASLUpgradedDataset(IterableDataset):
    def __init__(self, file_paths, phrase_map_path):
        self.file_paths = file_paths
        with open(phrase_map_path, 'r') as f:
            data = json.load(f)
            self.label_map = {item['label']: i for i, item in enumerate(data['labels'])}
    def __iter__(self):
        for file_path in self.file_paths:
            if not os.path.exists(file_path): continue
            with open(file_path, 'r') as f:
                for line in f:
                    try:
                        data = json.loads(line)
                        label = self.label_map[data['label']]
                        frames = torch.tensor(data['frames'], dtype=torch.float32)
                        frames_3d = frames.view(frames.size(0), 543, 3)
                        nose = frames_3d[:, 468:469, :]
                        frames_3d = frames_3d - nose
                        frames = frames_3d.view(frames.size(0), 1629)
                        diff1 = torch.cat([frames[1:] - frames[:-1], torch.zeros(1, 1629)], dim=0)
                        diff2 = torch.cat([frames[2:] - frames[:-2], torch.zeros(2, 1629)], dim=0)
                        yield torch.cat([frames, diff1, diff2], dim=-1), label
                    except Exception: continue

# 2. Setup Environment
device = torch.device('cpu')
torch.set_num_threads(os.cpu_count())
model = LandmarkTransformerUpgraded().to(device)
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)
criterion = nn.CrossEntropyLoss()

# 3. Load Checkpoint
weights_path = '/content/drive/MyDrive/asl_model_upgraded.pth'
if os.path.exists(weights_path):
    state = torch.load(weights_path, map_location=device)
    model.load_state_dict(state if 'input_projection.weight' in state else state['model_state_dict'])
    print('✅ Resuming on CPU from last checkpoint.')

# 4. Training Loop
drive_files = ['/content/drive/MyDrive/asl_train_data.jsonl', '/content/drive/MyDrive/asl_train_data_part2.jsonl']
phrase_map_path = '/content/drive/MyDrive/phrase_map.json'
asl_dataset = ASLUpgradedDataset(drive_files, phrase_map_path)
asl_loader = DataLoader(asl_dataset, batch_size=32, num_workers=0)

model.train()
print('Starting Training Loop...')
for epoch in range(42, 50):
    running_loss, running_acc = 0.0, 0.0
    for batch_idx, (frames, labels) in enumerate(asl_loader):
        frames, labels = frames.to(device), labels.to(device)
        frames = torch.nan_to_num(frames, nan=0.0)

        optimizer.zero_grad()
        outputs = model(frames)
        loss = criterion(outputs, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        acc = (outputs.argmax(1) == labels).sum().item() / labels.size(0)
        running_loss += loss.item()
        running_acc += acc

        if (batch_idx + 1) % 50 == 0:
            avg_loss = running_loss / 50
            avg_acc = running_acc / 50
            print(f'Epoch [{epoch+1}], Step [{batch_idx+1}], Loss: {avg_loss:.4f}, Acc: {avg_acc:.4f}')
            torch.save(model.state_dict(), weights_path)
            if avg_acc >= 0.45:
                torch.save(model.state_dict(), '/content/drive/MyDrive/asl_model_upgraded_best.pth')
                print('🎯 Target accuracy reached!')
                break
            running_loss, running_acc = 0.0, 0.0
    print(f'Epoch {epoch+1} complete.')

/tmp/ipykernel_4235/3759185167.py:15: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)


✅ Resuming on CPU from last checkpoint.
Epoch [43], Step [50], Loss: 3.4362, Acc: 0.3500
Epoch [43], Step [100], Loss: 3.4295, Acc: 0.3425
Epoch [43], Step [150], Loss: 3.3482, Acc: 0.3638
Epoch [43], Step [200], Loss: 3.4086, Acc: 0.3581
Epoch [43], Step [250], Loss: 3.3982, Acc: 0.3619
Epoch [43], Step [300], Loss: 3.3555, Acc: 0.3650
Epoch [43], Step [350], Loss: 3.3907, Acc: 0.3594
Epoch [43], Step [400], Loss: 3.2587, Acc: 0.3825
Epoch [43], Step [450], Loss: 3.3675, Acc: 0.3656
Epoch [43], Step [500], Loss: 3.4183, Acc: 0.3538
Epoch [43], Step [550], Loss: 3.4731, Acc: 0.3362
Epoch [43], Step [600], Loss: 3.4423, Acc: 0.3531
Epoch [43], Step [650], Loss: 3.6290, Acc: 0.3019
Epoch [43], Step [700], Loss: 3.5072, Acc: 0.3237
Epoch [43], Step [750], Loss: 3.6165, Acc: 0.3044
Epoch [43], Step [800], Loss: 3.5823, Acc: 0.3150
Epoch [43], Step [850], Loss: 3.5681, Acc: 0.3137
Epoch [43], Step [900], Loss: 3.6202, Acc: 0.3144
Epoch [43], Step [950], Loss: 3.4908, Acc: 0.3294
Epoch [43],

In [10]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, IterableDataset
from torch.amp import GradScaler, autocast
import json
import os

# 1. Self-contained Architecture & Dataset
class LandmarkTransformerUpgraded(nn.Module):
    def __init__(self, num_classes=250, input_dim=4887, model_dim=256, num_heads=8, num_layers=4, dropout=0.1):
        super().__init__()
        self.input_projection = nn.Linear(input_dim, model_dim)
        self.pos_encoder = nn.Parameter(torch.randn(1, 30, model_dim))
        encoder_layer = nn.TransformerEncoderLayer(d_model=model_dim, nhead=num_heads, dim_feedforward=model_dim * 4, dropout=dropout, batch_first=True, norm_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.classifier = nn.Linear(model_dim, num_classes)
    def forward(self, x):
        x = self.input_projection(x) + self.pos_encoder
        x = self.transformer(x).mean(dim=1)
        return self.classifier(x)

class ASLUpgradedDataset(IterableDataset):
    def __init__(self, file_paths, phrase_map_path):
        self.file_paths = file_paths
        with open(phrase_map_path, 'r') as f:
            data = json.load(f)
            self.label_map = {item['label']: i for i, item in enumerate(data['labels'])}
    def __iter__(self):
        for file_path in self.file_paths:
            if not os.path.exists(file_path): continue
            with open(file_path, 'r') as f:
                for line in f:
                    try:
                        data = json.loads(line)
                        label = self.label_map[data['label']]
                        frames = torch.tensor(data['frames'], dtype=torch.float32)
                        frames_3d = frames.view(frames.size(0), 543, 3)
                        nose = frames_3d[:, 468:469, :]
                        frames_3d = frames_3d - nose
                        frames = frames_3d.view(frames.size(0), 1629)
                        diff1 = torch.cat([frames[1:] - frames[:-1], torch.zeros(1, 1629)], dim=0)
                        diff2 = torch.cat([frames[2:] - frames[:-2], torch.zeros(2, 1629)], dim=0)
                        yield torch.cat([frames, diff1, diff2], dim=-1), label
                    except Exception: continue

# 2. Setup GPU and Hardware
device = torch.device('cuda')
torch.backends.cudnn.benchmark = True

# 3. Initialize Model, Loader and Optimization tools
model = LandmarkTransformerUpgraded().to(device)
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)
criterion = nn.CrossEntropyLoss()
scaler = GradScaler('cuda')

drive_files = ['/content/drive/MyDrive/asl_train_data.jsonl', '/content/drive/MyDrive/asl_train_data_part2.jsonl']
phrase_map_path = '/content/drive/MyDrive/phrase_map.json'
asl_dataset = ASLUpgradedDataset(drive_files, phrase_map_path)
asl_loader_gpu = DataLoader(asl_dataset, batch_size=128, num_workers=2, pin_memory=True)

# 4. Load latest state
weights_path = '/content/drive/MyDrive/asl_model_upgraded.pth'
if os.path.exists(weights_path):
    state = torch.load(weights_path, map_location=device)
    model.load_state_dict(state if 'input_projection.weight' in state else state['model_state_dict'])
    print('✅ Resuming GPU training from Epoch 47...')

# 5. Training Loop
TARGET_ACC = 0.45
model.train()
for epoch in range(47, 60):
    running_loss, running_acc = 0.0, 0.0
    for batch_idx, (frames, labels) in enumerate(asl_loader_gpu):
        frames, labels = frames.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        frames = torch.nan_to_num(frames, nan=0.0)

        optimizer.zero_grad(set_to_none=True)
        with autocast('cuda'):
            outputs = model(frames)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        acc = (outputs.argmax(1) == labels).sum().item() / labels.size(0)
        running_loss += loss.item()
        running_acc += acc

        if (batch_idx + 1) % 20 == 0:
            avg_loss, avg_acc = running_loss/20, running_acc/20
            print(f'Epoch [{epoch+1}], Step [{batch_idx+1}], Loss: {avg_loss:.4f}, Acc: {avg_acc:.4f}')
            torch.save(model.state_dict(), weights_path)
            if avg_acc >= TARGET_ACC:
                print(f'❉ Target accuracy {TARGET_ACC} reached!')
                torch.save(model.state_dict(), "/content/drive/MyDrive/asl_model_upgraded_best.pth")
                break
            running_loss, running_acc = 0.0, 0.0
    print(f'Epoch {epoch+1} complete.')

/tmp/ipykernel_1952/4174987350.py:16: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)


✅ Resuming GPU training from Epoch 47...
Epoch [48], Step [50], Loss: 3.1372, Acc: 0.4270
Epoch [48], Step [100], Loss: 3.1211, Acc: 0.4302
Epoch [48], Step [150], Loss: 3.1346, Acc: 0.4311
Epoch [48], Step [200], Loss: 3.0696, Acc: 0.4439
Epoch [48], Step [250], Loss: 3.2345, Acc: 0.3955


KeyboardInterrupt: 

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, IterableDataset
from torch.amp import GradScaler, autocast
import json
import os
from google.colab import drive

# 1. Mount Drive
drive.mount('/content/drive')

# 2. Model & Dataset Architecture
class LandmarkTransformerUpgraded(nn.Module):
    def __init__(self, num_classes=250, input_dim=4887, model_dim=256, num_heads=8, num_layers=4, dropout=0.1):
        super().__init__()
        self.input_projection = nn.Linear(input_dim, model_dim)
        self.pos_encoder = nn.Parameter(torch.randn(1, 30, model_dim))
        encoder_layer = nn.TransformerEncoderLayer(d_model=model_dim, nhead=num_heads, dim_feedforward=model_dim * 4, dropout=dropout, batch_first=True, norm_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.classifier = nn.Linear(model_dim, num_classes)
    def forward(self, x):
        x = self.input_projection(x) + self.pos_encoder
        x = self.transformer(x).mean(dim=1)
        return self.classifier(x)

class ASLUpgradedDataset(IterableDataset):
    def __init__(self, file_paths, phrase_map_path):
        self.file_paths = file_paths
        with open(phrase_map_path, 'r') as f:
            data = json.load(f)
            self.label_map = {item['label']: i for i, item in enumerate(data['labels'])}
    def __iter__(self):
        for file_path in self.file_paths:
            if not os.path.exists(file_path): continue
            with open(file_path, 'r') as f:
                for line in f:
                    try:
                        data = json.loads(line)
                        label = self.label_map[data['label']]
                        frames = torch.tensor(data['frames'], dtype=torch.float32)
                        frames_3d = frames.view(frames.size(0), 543, 3)
                        nose = frames_3d[:, 468:469, :]
                        frames_3d = frames_3d - nose
                        frames = frames_3d.view(frames.size(0), 1629)
                        diff1 = torch.cat([frames[1:] - frames[:-1], torch.zeros(1, 1629)], dim=0)
                        diff2 = torch.cat([frames[2:] - frames[:-2], torch.zeros(2, 1629)], dim=0)
                        yield torch.cat([frames, diff1, diff2], dim=-1), label
                    except Exception: continue

# 3. Setup Hardware
device = torch.device('cuda')
model = LandmarkTransformerUpgraded().to(device)
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)
criterion = nn.CrossEntropyLoss()
scaler = GradScaler('cuda')

# 4. Load Progress
weights_path = '/content/drive/MyDrive/asl_model_upgraded.pth'
if os.path.exists(weights_path):
    state = torch.load(weights_path, map_location=device)
    model.load_state_dict(state if 'input_projection.weight' in state else state['model_state_dict'])
    print('✅ Resuming from checkpoint.')

# 5. Resume Training Loop
drive_files = ['/content/drive/MyDrive/asl_train_data.jsonl', '/content/drive/MyDrive/asl_train_data_part2.jsonl']
phrase_map_path = '/content/drive/MyDrive/phrase_map.json'
asl_loader = DataLoader(ASLUpgradedDataset(drive_files, phrase_map_path), batch_size=128, num_workers=2, pin_memory=True)

TARGET_ACC = 0.45
model.train()
for epoch in range(48, 60):
    running_loss, running_acc = 0.0, 0.0
    for batch_idx, (frames, labels) in enumerate(asl_loader):
        frames, labels = frames.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        frames = torch.nan_to_num(frames, nan=0.0)

        optimizer.zero_grad(set_to_none=True)
        with autocast('cuda'):
            outputs = model(frames)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        acc = (outputs.argmax(1) == labels).sum().item() / labels.size(0)
        running_loss += loss.item()
        running_acc += acc

        if (batch_idx + 1) % 20 == 0:
            avg_acc = running_acc / 20
            print(f'Epoch [{epoch+1}], Step [{batch_idx+1}], Loss: {running_loss/20:.4f}, Acc: {avg_acc:.4f}')
            torch.save(model.state_dict(), weights_path)
            if avg_acc >= TARGET_ACC:
                print(f'🎯 Target accuracy {TARGET_ACC} reached!')
                torch.save(model.state_dict(), '/content/drive/MyDrive/asl_model_upgraded_best.pth')
                break
            running_loss, running_acc = 0.0, 0.0
    print(f'Epoch {epoch+1} complete.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Resuming from checkpoint.


/tmp/ipykernel_1952/1952567253.py:20: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)


Epoch [49], Step [20], Loss: 3.0902, Acc: 0.4375


### 📥 Download Weights for Kaggle Upload
Run this cell to download your model weights and phrase map to your local machine so you can upload them to Kaggle.

In [ ]:
from google.colab import files
import os

weights_to_download = [
    '/content/drive/MyDrive/asl_model_upgraded_best.pth',
    '/content/drive/MyDrive/phrase_map.json'
]

for path in weights_to_download:
    if os.path.exists(path):
        print(f'Downloading: {path}')
        files.download(path)
    else:
        print(f'❌ File not found: {path}. Make sure training has saved at least one checkpoint.')

In [1]:
import os
# Let's check exactly what files are available in your Drive right now
print('--- Drive File Check ---')
!ls -lh /content/drive/MyDrive/asl_model_upgraded*
!ls -lh /content/drive/MyDrive/phrase_map.json

--- Drive File Check ---
-rw------- 1 root root 52M May 11 07:58 /content/drive/MyDrive/asl_model_upgraded_best.pth
-rw------- 1 root root 18M May 13 19:25 /content/drive/MyDrive/asl_model_upgraded.pth
-rw------- 1 root root 4.9K May  3 17:08 /content/drive/MyDrive/phrase_map.json


### 📤 Upload Weights to Kaggle via CLI
If the files exist above, we can bundle them and upload them to Kaggle as a new dataset version directly from Colab.

In [34]:
import json
import os
import shutil

# 1. Prepare a temporary folder for the Kaggle Dataset
os.makedirs('/content/kaggle_upload', exist_ok=True)

# 2. Copy ALL relevant model files discovered in the audit
print("Copying model weights to upload directory...")
!cp /content/drive/MyDrive/asl_model_upgraded_best.pth /content/kaggle_upload/
!cp /content/drive/MyDrive/asl_model_upgraded.pth /content/kaggle_upload/
!cp /content/drive/MyDrive/asl_model_best.pth /content/kaggle_upload/
!cp /content/drive/MyDrive/phrase_map.json /content/kaggle_upload/

# 3. Create the dataset metadata
meta = {
  "title": "ASL Model Weights",
  "id": "basseyjohn/asl-model-weights",
  "licenses": [{"name": "CC0-1.0"}]
}

with open('/content/kaggle_upload/dataset-metadata.json', 'w') as f:
    json.dump(meta, f)

print(f"Success. Folder contains: {os.listdir('/content/kaggle_upload')}")

Copying model weights to upload directory...
Success. Folder contains: ['asl_model_upgraded_best.pth', 'asl_model_upgraded.pth', 'asl_model_best.pth', 'dataset-metadata.json', 'phrase_map.json']


### 🚀 Final Step: Kaggle Upload
Run the cell below to push your weights to Kaggle. Ensure your `kaggle.json` was correctly configured earlier in the notebook.

In [5]:
# Use 'create' for the first time setup
!kaggle datasets create -p /content/kaggle_upload

# Or use 'version' if the dataset already exists on your Kaggle profile
# !kaggle datasets version -p /content/kaggle_upload -m "Updated weights from Colab"

You must authenticate before you can call the Kaggle API.
Follow the instructions to authenticate at: https://github.com/Kaggle/kaggle-cli/blob/main/docs/README.md#authentication


### 🚀 Final Step: Kaggle Upload
Run the cell below to push your weights to Kaggle. Ensure your `kaggle.json` was correctly configured earlier in the notebook.

In [4]:
# Use 'create' for the first time setup
!kaggle datasets create -p /content/kaggle_upload

# Or use 'version' if the dataset already exists on your Kaggle profile
# !kaggle datasets version -p /content/kaggle_upload -m "Updated weights from Colab"

You must authenticate before you can call the Kaggle API.
Follow the instructions to authenticate at: https://github.com/Kaggle/kaggle-cli/blob/main/docs/README.md#authentication


### 🚀 Final Step: Kaggle Upload
Run the cell below to push your weights to Kaggle. Ensure your `kaggle.json` was correctly configured earlier in the notebook.

In [11]:
import os

# Corrected upload logic using standard bash syntax
print("🚀 Starting Kaggle upload...")

# Check if dataset exists to decide between 'create' or 'version'
!kaggle datasets status basseyjohn/asl-model-weights > /dev/null 2>&1 \
&& (echo "Dataset exists. Pushing new version..." && kaggle datasets version -p /content/kaggle_upload -m "Updated weights from Colab") \
|| (echo "Dataset not found. Creating new dataset..." && kaggle datasets create -p /content/kaggle_upload)

🚀 Starting Kaggle upload...
Dataset not found. Creating new dataset...
Starting upload for file asl_model_upgraded.pth
401 Client Error: Unauthorized for url: https://api.kaggle.com/v1/blobs.BlobApiService/StartBlobUpload


In [12]:
import os
import json

# 1. Force refresh environment variables
os.environ['KAGGLE_USERNAME'] = "basseyjohn"
os.environ['KAGGLE_KEY'] = "KGAT_4c2926c448440c3be0187f1fc136663f"

# 2. Re-write the config file
kaggle_dir = os.path.expanduser('~/.kaggle')
os.makedirs(kaggle_dir, exist_ok=True)
with open(os.path.join(kaggle_dir, 'kaggle.json'), 'w') as f:
    json.dump({"username": os.environ['KAGGLE_USERNAME'], "key": os.environ['KAGGLE_KEY']}, f)

!chmod 600 ~/.kaggle/kaggle.json

# 3. Robust upload command with explicit environment export
print("🚀 Retrying upload with explicit auth variables...")
!export KAGGLE_USERNAME="basseyjohn" && export KAGGLE_KEY="KGAT_4c2926c448440c3be0187f1fc136663f" && \
kaggle datasets create -p /content/kaggle_upload || \
kaggle datasets version -p /content/kaggle_upload -m "Retry from Colab"

🚀 Retrying upload with explicit auth variables...
Starting upload for file asl_model_upgraded.pth
401 Client Error: Unauthorized for url: https://api.kaggle.com/v1/blobs.BlobApiService/StartBlobUpload
Starting upload for file asl_model_upgraded.pth
401 Client Error: Unauthorized for url: https://api.kaggle.com/v1/blobs.BlobApiService/StartBlobUpload


In [14]:
import os
import json

# --- 1. PASTE YOUR NEW KEY HERE ---
KAGGLE_USER = "basseyjohn"
KAGGLE_KEY = "PASTE_NEW_KEY_HERE"

# Diagnostic: Check for common formatting issues
KAGGLE_USER = KAGGLE_USER.strip()
KAGGLE_KEY = KAGGLE_KEY.strip()

# 2. Write credentials to the system
kaggle_dir = os.path.expanduser('~/.kaggle')
os.makedirs(kaggle_dir, exist_ok=True)
with open(os.path.join(kaggle_dir, 'kaggle.json'), 'w') as f:
    json.dump({"username": KAGGLE_USER, "key": KAGGLE_KEY}, f)

os.chmod(os.path.join(kaggle_dir, 'kaggle.json'), 0o600)

# 3. Final Upload Attempt
print(f"🚀 Retrying upload with fresh credentials for {KAGGLE_USER}...")
# Corrected the bash command syntax below
!export KAGGLE_USERNAME="{KAGGLE_USER}" && export KAGGLE_KEY="{KAGGLE_KEY}" && \
(kaggle datasets create -p /content/kaggle_upload || \
kaggle datasets version -p /content/kaggle_upload -m 'Retry with fresh token')

🚀 Retrying upload with fresh credentials for basseyjohn...
Starting upload for file asl_model_upgraded.pth
401 Client Error: Unauthorized for url: https://api.kaggle.com/v1/blobs.BlobApiService/StartBlobUpload
Starting upload for file asl_model_upgraded.pth
401 Client Error: Unauthorized for url: https://api.kaggle.com/v1/blobs.BlobApiService/StartBlobUpload


### 🔑 Step 1: Update Credentials
Replace `PASTE_YOUR_NEW_TOKEN_HERE` with the key found in your new `kaggle.json` file.

In [17]:
import os
import json

# --- 1. RE-PASTE YOUR FRESH TOKEN HERE ---
KAGGLE_USERNAME = "basseyjohn"
KAGGLE_KEY = "PASTE_YOUR_NEW_TOKEN_HERE"

# Diagnostic: Ensure no whitespace
KAGGLE_USERNAME = KAGGLE_USERNAME.strip()
KAGGLE_KEY = KAGGLE_KEY.strip()

# 2. Write configuration file
kaggle_dir = os.path.expanduser('~/.kaggle')
!mkdir -p ~/.kaggle
with open(os.path.join(kaggle_dir, 'kaggle.json'), 'w') as f:
    json.dump({"username": KAGGLE_USERNAME, "key": KAGGLE_KEY}, f)

!chmod 600 ~/.kaggle/kaggle.json

# 3. Set environment variables
os.environ['KAGGLE_USERNAME'] = KAGGLE_USERNAME
os.environ['KAGGLE_KEY'] = KAGGLE_KEY

print("✅ Credentials updated. Verifying connectivity...")
!kaggle datasets list -s "iris" | head -n 5

✅ Credentials updated. Verifying connectivity...
ref                                                     title                                                size  lastUpdated                 downloadCount  voteCount  usabilityRating  
------------------------------------------------------  ---------------------------------------------  ----------  --------------------------  -------------  ---------  ---------------  
uciml/iris                                              Iris Species                                         3687  2016-09-27 07:38:05.440000         877217       4773  0.7941176        
himanshunakrani/iris-dataset                            Iris dataset                                         1006  2022-07-20 18:50:06.277000         100485        400  1                
arshid/iris-flower-dataset                              Iris Flower Dataset                                  1010  2018-03-22 15:18:06.097000         247320       1111  0.8235294        


### 🚀 Step 2: Final Upload
If the cell above listed datasets successfully, run this cell to push your weights to Kaggle.

In [18]:
# This command attempts to create the dataset. If it exists, it pushes a new version.
!kaggle datasets create -p /content/kaggle_upload || \
kaggle datasets version -p /content/kaggle_upload -m "Updated weights from Colab"

Starting upload for file asl_model_upgraded.pth
401 Client Error: Unauthorized for url: https://api.kaggle.com/v1/blobs.BlobApiService/StartBlobUpload
Starting upload for file asl_model_upgraded.pth
401 Client Error: Unauthorized for url: https://api.kaggle.com/v1/blobs.BlobApiService/StartBlobUpload


In [26]:
import os
import kagglehub

# 1. Verify credentials are in the environment
os.environ['KAGGLE_USERNAME'] = "basseyjohn"
os.environ['KAGGLE_KEY'] = "KGAT_2d056a06f272c816cbe9ac36d34169d7"

# 2. Define Dataset Handle
handle = "basseyjohn/asl-model-weights"
local_dir = "/content/kaggle_upload"

print(f"🚀 Initializing upload to Kaggle: {handle}")

try:
    # Using local_dir as the correct parameter for kagglehub v1.0.0
    kagglehub.dataset_upload(
        handle=handle,
        local_dir=local_dir,
        version_notes="Upgraded model weights and phrase map"
    )
    print("\n✅ SUCCESS! Your model weights have been uploaded to Kaggle.")
except Exception as e:
    print(f"\n❌ Upload failed: {e}")
    print("Tip: Ensure you have accepted any Kaggle site terms and that your API token is still active.")

🚀 Initializing upload to Kaggle: basseyjohn/asl-model-weights

❌ Upload failed: dataset_upload() got an unexpected keyword argument 'local_dir'
Tip: Ensure you have accepted any Kaggle site terms and that your API token is still active.


In [35]:
import os
import json
import kagglehub

# 1. Setup Credentials from uploaded kaggle.json
KAGGLE_CONFIG = '/content/kaggle.json'

if os.path.exists(KAGGLE_CONFIG):
    with open(KAGGLE_CONFIG, 'r') as f:
        creds = json.load(f)
    os.environ["KAGGLE_USERNAME"] = creds['username']
    os.environ["KAGGLE_KEY"] = creds['key']
    print(f"✅ Credentials loaded for user: {creds['username']}")

# 2. Configuration
handle = "basseyjohn/asl-model-weights"
local_folder = "/content/kaggle_upload"

print(f"  Starting comprehensive transfer to Kaggle: {handle}")

try:
    # Using local_dataset_dir to push the full bundle
    url = kagglehub.dataset_upload(
        handle=handle,
        local_dataset_dir=local_folder,
        version_notes="Complete migration including upgraded best weights and legacy checkpoints"
    )
    print(f"\n✅ SUCCESS! Dataset updated at: {url}")
except Exception as e:
    print(f"\n❌ Transfer Error: {e}")

✅ Credentials loaded for user: basseyjohn
  Starting comprehensive transfer to Kaggle: basseyjohn/asl-model-weights
Uploading Dataset https://api.kaggle.com/datasets/basseyjohn/asl-model-weights ...
Starting upload for file /content/kaggle_upload/asl_model_upgraded_best.pth


Uploading: 100%|██████████| 53.9M/53.9M [00:01<00:00, 47.8MB/s]

Upload successful: /content/kaggle_upload/asl_model_upgraded_best.pth (51MB)
Starting upload for file /content/kaggle_upload/asl_model_upgraded.pth



Uploading: 100%|██████████| 17.9M/17.9M [00:00<00:00, 24.9MB/s]

Upload successful: /content/kaggle_upload/asl_model_upgraded.pth (17MB)
Starting upload for file /content/kaggle_upload/asl_model_best.pth


...


Uploading: 100%|██████████| 103/103 [00:00<00:00, 187B/s]

Upload successful: /content/kaggle_upload/dataset-metadata.json (103B)
Starting upload for file /content/kaggle_upload/phrase_map.json



Uploading: 100%|██████████| 4.97k/4.97k [00:00<00:00, 9.12kB/s]

Upload successful: /content/kaggle_upload/phrase_map.json (5KB)


Your dataset version has been created.
Files are being processed...
See at: https://api.kaggle.com/datasets/basseyjohn/asl-model-weights

✅ SUCCESS! Dataset updated at: None


### ✅ Final Migration Status
Use this cell to confirm the dataset is live on Kaggle.

In [36]:
import kaggle

try:
    # Check dataset status
    status = !kaggle datasets status basseyjohn/asl-model-weights
    print(f"Current Kaggle Status: {status[0] if status else 'Dataset Found'}")
    print(f"Dataset URL: https://www.kaggle.com/datasets/basseyjohn/asl-model-weights")

    # List files to verify completeness
    print("\nLive Files on Kaggle:")
    !kaggle datasets files basseyjohn/asl-model-weights
except Exception as e:
    print(f"Verification failed (it may still be processing): {e}")

Current Kaggle Status: ready
Dataset URL: https://www.kaggle.com/datasets/basseyjohn/asl-model-weights

Live Files on Kaggle:
name                             size  creationDate                
---------------------------  --------  --------------------------  
asl_model_best.pth           43843021  2026-05-13 19:43:39.996000  
asl_model_upgraded.pth       17949837  2026-05-13 19:43:39.594000  
asl_model_upgraded_best.pth  53853623  2026-05-13 19:43:39.959000  
dataset-metadata.json             103  2026-05-13 19:43:39.254000  
phrase_map.json                  4974  2026-05-13 19:43:39.219000  


### 🔍 Verify Files Inside the Dataset
Since we consolidated everything, this cell lists all files currently hosted within the `asl-model-weights` dataset to ensure nothing was missed.

In [37]:
import os

# List all files currently in the Kaggle dataset
print("Contents of 'basseyjohn/asl-model-weights':")
!kaggle datasets files basseyjohn/asl-model-weights

Contents of 'basseyjohn/asl-model-weights':
name                             size  creationDate                
---------------------------  --------  --------------------------  
asl_model_best.pth           43843021  2026-05-13 19:43:39.996000  
asl_model_upgraded.pth       17949837  2026-05-13 19:43:39.594000  
asl_model_upgraded_best.pth  53853623  2026-05-13 19:43:39.959000  
dataset-metadata.json             103  2026-05-13 19:43:39.254000  
phrase_map.json                  4974  2026-05-13 19:43:39.219000  


In [55]:
import json
import os

# Check both potential locations for the phrase map
paths = [
    '/content/drive/MyDrive/phrase_map.json',
    '/kaggle/input/asl-model-weights/phrase_map.json',
    '/content/kaggle_upload/phrase_map.json'
]

for path in paths:
    if os.path.exists(path):
        with open(path, 'r') as f:
            phrase_data = json.load(f)
            # Extract labels based on the JSON structure {labels: [{label: 'sign'}, ...]}
            labels = phrase_data['labels'] if isinstance(phrase_data, dict) else phrase_data
            print(f"--- Phrase Map Found at: {path} ---")
            print(f"Total classes: {len(labels)}")
            print("First 5 labels:", [item['label'] if isinstance(item, dict) else item for item in labels[:5]])
            break
else:
    print("❌ phrase_map.json not found in expected locations.")

--- Phrase Map Found at: /content/drive/MyDrive/phrase_map.json ---
Total classes: 250
First 5 labels: ['TV', 'after', 'airplane', 'all', 'alligator']


### 🚀 Prepare and Upload Weights to Kaggle
This section bundles the model artifacts (`.pth` files and `phrase_map.json`) and uploads them to Kaggle using the API.

In [ ]:
import os
import json
import shutil
import kagglehub

# 1. Configuration
UPLOAD_DIR = '/content/kaggle_bundle'
DRIVE_PATH = '/content/drive/MyDrive'
DATASET_HANDLE = "basseyjohn/asl-model-weights"

# 2. Prepare Directory
if os.path.exists(UPLOAD_DIR): shutil.rmtree(UPLOAD_DIR)
os.makedirs(UPLOAD_DIR, exist_ok=True)

# 3. Copy relevant files from Drive
files_to_copy = [
    'asl_model_upgraded_best.pth',
    'asl_model_upgraded.pth',
    'phrase_map.json'
]

print("📦 Gathering model artifacts...")
for f in files_to_copy:
    src = os.path.join(DRIVE_PATH, f)
    if os.path.exists(src):
        shutil.copy2(src, UPLOAD_DIR)
        print(f"✅ Bundled: {f}")
    else:
        print(f"⚠️ Warning: {f} not found on Drive.")

# 4. Create Metadata
meta = {
  "title": "ASL Model Weights",
  "id": DATASET_HANDLE,
  "licenses": [{"name": "CC0-1.0"}]
}
with open(os.path.join(UPLOAD_DIR, 'dataset-metadata.json'), 'w') as f:
    json.dump(meta, f)

# 5. Upload via KaggleHub
try:
    print(f"🚀 Initializing upload to Kaggle: {DATASET_HANDLE}...")
    url = kagglehub.dataset_upload(
        handle=DATASET_HANDLE,
        local_dataset_dir=UPLOAD_DIR,
        version_notes="Migration from Colab for continued training (Epoch 43+)"
    )
    print(f"\n✅ SUCCESS! Dataset is live at: {url}")
except Exception as e:
    print(f"\n❌ Transfer Error: {e}")
    print("Tip: Ensure you have accepted the competition rules and your Kaggle API key is correctly configured.")

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, IterableDataset
from torch.amp import GradScaler, autocast
import json
import os

# 1. Architecture: LandmarkTransformerUpgraded (4 layers, 256 dim, 8 heads)
class LandmarkTransformerUpgraded(nn.Module):
    def __init__(self, num_classes=250, input_dim=4887, model_dim=256, num_heads=8, num_layers=4, dropout=0.1):
        super().__init__()
        self.input_projection = nn.Linear(input_dim, model_dim)
        self.pos_encoder = nn.Parameter(torch.randn(1, 30, model_dim))
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=model_dim,
            nhead=num_heads,
            dim_feedforward=model_dim * 4,
            dropout=dropout,
            batch_first=True,
            norm_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.classifier = nn.Linear(model_dim, num_classes)

    def forward(self, x):
        x = self.input_projection(x) + self.pos_encoder
        x = self.transformer(x).mean(dim=1)
        return self.classifier(x)

# 2. Preprocessing Logic (0.5x Scaling + Multi-tiered Anchor)
def get_robust_anchor(f_3d):
    anchors = [468, 1, 2, 98] # Multi-tiered check
    for idx in anchors:
        anchor = f_3d[:, idx:idx+1, :]
        if not torch.isnan(anchor).all():
            return anchor
    return torch.zeros((f_3d.size(0), 1, 3)).to(f_3d.device)

class ASLFinalDataset(IterableDataset):
    def __init__(self, file_paths, phrase_map_path, scaling=0.5):
        self.file_paths = file_paths
        self.scaling = scaling
        with open(phrase_map_path, 'r') as f:
            data = json.load(f)
            labels = data['labels'] if isinstance(data, dict) else data
            self.label_map = {item['label']: i for i, item in enumerate(labels)}

    def __iter__(self):
        for file_path in self.file_paths:
            if not os.path.exists(file_path): continue
            with open(file_path, 'r') as f:
                for line in f:
                    try:
                        data = json.loads(line)
                        label = self.label_map[data['label']]
                        frames = torch.tensor(data['frames'], dtype=torch.float32)
                        f_3d = frames.view(30, 543, 3)

                        anchor = get_robust_anchor(f_3d)
                        f_norm = (f_3d - anchor) * self.scaling
                        f_norm = torch.nan_to_num(f_norm, nan=0.0)
                        f_flat = f_norm.view(30, 1629)

                        # Motion Features (Velocity + Acceleration)
                        diff1 = torch.cat([f_flat[1:] - f_flat[:-1], torch.zeros(1, 1629)], dim=0)
                        diff2 = torch.cat([f_flat[2:] - f_flat[:-2], torch.zeros(2, 1629)], dim=0)
                        yield torch.cat([f_flat, diff1, diff2], dim=-1), label
                    except Exception: continue

In [ ]:
# 3. Initialization & Weight Loading
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = LandmarkTransformerUpgraded().to(device)
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)
criterion = nn.CrossEntropyLoss()
scaler = GradScaler('cuda', enabled=torch.cuda.is_available())

weights_path = '/content/drive/MyDrive/asl_model_upgraded_best.pth'
if os.path.exists(weights_path):
    ckpt = torch.load(weights_path, map_location=device)
    model.load_state_dict(ckpt['model_state_dict'] if 'model_state_dict' in ckpt else ckpt)
    print("✅ Loaded 41.12% Baseline Weights. Ready to resume training.")
else:
    print("❌ Weights not found. Ensure Drive is mounted.")

# 4. Resume Training Loop
drive_files = ['/content/drive/MyDrive/asl_train_data.jsonl', '/content/drive/MyDrive/asl_train_data_part2.jsonl']
phrase_map = '/content/drive/MyDrive/phrase_map.json'
asl_loader = DataLoader(ASLFinalDataset(drive_files, phrase_map), batch_size=128, num_workers=2, pin_memory=True)

print("🚀 Starting stabilization run (Epoch 43+)...")
model.train()
for epoch in range(43, 60):
    running_loss, running_acc = 0.0, 0.0
    for batch_idx, (frames, labels) in enumerate(asl_loader):
        frames, labels = frames.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        with autocast('cuda', enabled=torch.cuda.is_available()):
            outputs = model(frames)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        acc = (outputs.argmax(1) == labels).sum().item() / labels.size(0)
        running_loss += loss.item()
        running_acc += acc

        if (batch_idx + 1) % 50 == 0:
            print(f'Epoch [{epoch+1}] Step [{batch_idx+1}] Loss: {running_loss/50:.4f} Acc: {running_acc/50:.4f}')
            running_loss, running_acc = 0.0, 0.0

    torch.save(model.state_dict(), '/content/drive/MyDrive/asl_model_upgraded_latest.pth')

/tmp/ipykernel_13742/1615646965.py:23: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)


✅ Loaded 41.12% Baseline Weights. Ready to resume training.
🚀 Starting stabilization run (Epoch 43+)...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch [44] Step [50] Loss: 4.1246 Acc: 0.2339


In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, IterableDataset
import json
import os

# 1. Standardized Preprocessing with 0.5x Scaling
def get_robust_anchor(frames_3d):
    anchors = [468, 1, 2, 98]
    for idx in anchors:
        anchor = frames_3d[:, idx:idx+1, :]
        if not torch.isnan(anchor).all():
            return anchor
    face = frames_3d[:, 0:468, :]
    mask = ~torch.isnan(face)
    if mask.any():
        return (face * mask).sum(dim=1, keepdim=True) / mask.sum(dim=1, keepdim=True).clamp(min=1)
    return torch.zeros((frames_3d.size(0), 1, 3)).to(frames_3d.device)

class ASLStableDataset(IterableDataset):
    def __init__(self, file_paths, phrase_map_path, scaling_factor=0.5):
        self.file_paths = file_paths
        self.scaling_factor = scaling_factor
        with open(phrase_map_path, 'r') as f:
            data = json.load(f)
            labels = data['labels'] if isinstance(data, dict) else data
            self.label_map = {item['label']: i for i, item in enumerate(labels)}

    def __iter__(self):
        for file_path in self.file_paths:
            if not os.path.exists(file_path): continue
            with open(file_path, 'r') as f:
                for line in f:
                    try:
                        data = json.loads(line)
                        label = self.label_map[data['label']]
                        frames = torch.tensor(data['frames'], dtype=torch.float32)
                        f_3d = frames.view(30, 543, 3)
                        anchor = get_robust_anchor(f_3d)

                        # Apply 0.5x Scaling
                        f_norm = (f_3d - anchor) * self.scaling_factor
                        f_norm = torch.nan_to_num(f_norm, nan=0.0)
                        f_flat = f_norm.view(30, 1629)

                        # Motion Features
                        diff1 = torch.cat([f_flat[1:] - f_flat[:-1], torch.zeros(1, 1629)], dim=0)
                        diff2 = torch.cat([f_flat[2:] - f_flat[:-2], torch.zeros(2, 1629)], dim=0)
                        yield torch.cat([f_flat, diff1, diff2], dim=-1), label
                    except Exception: continue

# 2. Resuming AdamW Training Loop
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = LandmarkTransformerUpgraded().to(device)
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)
criterion = nn.CrossEntropyLoss()
scaler = torch.amp.GradScaler('cuda', enabled=torch.cuda.is_available())

# Load Weights
weights_path = '/content/drive/MyDrive/asl_model_upgraded_best.pth'
if os.path.exists(weights_path):
    ckpt = torch.load(weights_path, map_location=device)
    model.load_state_dict(ckpt['model_state_dict'] if 'model_state_dict' in ckpt else ckpt)
    print("✅ Loaded weights for training resumption.")

drive_files = ['/content/drive/MyDrive/asl_train_data.jsonl', '/content/drive/MyDrive/asl_train_data_part2.jsonl']
phrase_map = '/content/drive/MyDrive/phrase_map.json'
asl_dataset = ASLStableDataset(drive_files, phrase_map)
asl_loader = DataLoader(asl_dataset, batch_size=128, num_workers=2, pin_memory=True)

model.train()
print("🚀 Resuming training with 0.5x Scaling...")
for epoch in range(43, 60):
    running_loss, running_acc = 0.0, 0.0
    for batch_idx, (frames, labels) in enumerate(asl_loader):
        frames, labels = frames.to(device), labels.to(device)
        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()):
            outputs = model(frames)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        acc = (outputs.argmax(1) == labels).sum().item() / labels.size(0)
        running_loss += loss.item()
        running_acc += acc

        if (batch_idx + 1) % 50 == 0:
            print(f'Epoch [{epoch+1}] Step [{batch_idx+1}] Loss: {running_loss/50:.4f} Acc: {running_acc/50:.4f}')
            running_loss, running_acc = 0.0, 0.0

    torch.save(model.state_dict(), '/content/drive/MyDrive/asl_model_upgraded_latest.pth')


NameError: name 'LandmarkTransformerUpgraded' is not defined

In [56]:
import torch
import torch.nn as nn
import os

# 1. Define the architecture used in the checkpoint to ensure weights map correctly
class LandmarkTransformerUpgraded(nn.Module):
    def __init__(self, num_classes=250, input_dim=4887, model_dim=256, num_heads=8, num_layers=4, dropout=0.1):
        super().__init__()
        self.input_projection = nn.Linear(input_dim, model_dim)
        self.pos_encoder = nn.Parameter(torch.randn(1, 30, model_dim))
        encoder_layer = nn.TransformerEncoderLayer(d_model=model_dim, nhead=num_heads, dim_feedforward=model_dim * 4, dropout=dropout, batch_first=True, norm_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.classifier = nn.Linear(model_dim, num_classes)
    def forward(self, x):
        x = self.input_projection(x) + self.pos_encoder
        x = self.transformer(x).mean(dim=1)
        return self.classifier(x)

# 2. Check the weights
weights_path = '/content/drive/MyDrive/asl_model_upgraded_best.pth'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = LandmarkTransformerUpgraded().to(device)

if os.path.exists(weights_path):
    ckpt = torch.load(weights_path, map_location=device)
    state_dict = ckpt['model_state_dict'] if 'model_state_dict' in ckpt else ckpt
    model.load_state_dict(state_dict)
    print(f"✅ Successfully loaded weights. Reported Accuracy in CP: {ckpt.get('accuracy', 'N/A')}")

    # Inspect input projection weight statistics
    weight_mean = model.input_projection.weight.mean().item()
    weight_std = model.input_projection.weight.std().item()
    print(f"Input Projection Weights - Mean: {weight_mean:.6f}, Std: {weight_std:.6f}")
else:
    print("❌ Checkpoint file not found on Drive.")

/tmp/ipykernel_4755/37111461.py:12: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)


✅ Successfully loaded weights. Reported Accuracy in CP: 0.4103125
Input Projection Weights - Mean: -0.000023, Std: 0.035136


In [66]:
import torch
import torch.nn as nn
import json
import os
import numpy as np
from torch.utils.data import DataLoader

# --- Diagnostic Tool v3.6: Moderate Scaling Factor Optimization ---

def get_robust_anchor(frames_3d):
    anchors = [468, 1, 2, 98]
    for idx in anchors:
        anchor = frames_3d[:, idx:idx+1, :]
        if not torch.isnan(anchor).all():
            return anchor
    face = frames_3d[:, 0:468, :]
    mask = ~torch.isnan(face)
    if mask.any():
        return (face * mask).sum(dim=1, keepdim=True) / mask.sum(dim=1, keepdim=True).clamp(min=1)
    return torch.zeros((frames_3d.size(0), 1, 3)).to(frames_3d.device)

def test_scaled_normalization(model, dataset_paths, phrase_map, device):
    model.eval()
    try:
        class ASLDiagnosticDataset(torch.utils.data.IterableDataset):
            def __init__(self, file_paths, phrase_map_path):
                self.file_paths = file_paths
                with open(phrase_map_path, 'r') as f:
                    self.label_map = {item['label']: i for i, item in enumerate(json.load(f)['labels'])}
            def __iter__(self):
                for path in self.file_paths:
                    if not os.path.exists(path): continue
                    with open(path, 'r') as f:
                        for line in f:
                            data = json.loads(line)
                            yield torch.tensor(data['frames'], dtype=torch.float32), self.label_map[data['label']]

        ds = ASLDiagnosticDataset(dataset_paths, phrase_map)
        loader = DataLoader(ds, batch_size=32)
        batch = next(iter(loader))
        raw_frames, labels = batch[0].to(device), batch[1].to(device)

        # v3.6 Strategy: Increase scaling to 0.5x (v3.5 at 0.3x was too aggressive)
        weight_std = model.input_projection.weight.std().item()
        scaling_factor = 0.5

        print(f"--- Preprocessing Diagnostic v3.6 (Moderate Scaling) ---")
        print(f"Target Weight Std: {weight_std:.4f} | Applying Scale: {scaling_factor}")

        with torch.no_grad():
            processed = []
            for f in raw_frames:
                f_3d = f.view(30, 543, 3)
                anchor = get_robust_anchor(f_3d)
                # Apply moderate scaling
                f_norm = (f_3d - anchor) * scaling_factor
                f_norm = torch.nan_to_num(f_norm, nan=0.0)
                f_flat = f_norm.view(30, 1629)

                diff1 = torch.cat([f_flat[1:] - f_flat[:-1], torch.zeros(1, 1629).to(device)], dim=0)
                diff2 = torch.cat([f_flat[2:] - f_flat[:-2], torch.zeros(2, 1629).to(device)], dim=0)
                processed.append(torch.cat([f_flat, diff1, diff2], dim=-1))

            outputs = model(torch.stack(processed))
            acc = (outputs.argmax(1) == labels).sum().item() / labels.size(0)
            print(f"Scaled Preprocessing Accuracy: {acc*100:.2f}%")
            print(f"New Feature Std: {torch.stack(processed).std().item():.4f}")

    except Exception as e:
        print(f"❌ Test failed: {e}")

drive_files = ['/content/drive/MyDrive/asl_train_data.jsonl']
phrase_map = '/content/drive/MyDrive/phrase_map.json'
if os.path.exists(drive_files[0]):
    test_scaled_normalization(model, drive_files, phrase_map, device)
else:
    print("❌ Files not found for testing.")

--- Preprocessing Diagnostic v3.6 (Moderate Scaling) ---
Target Weight Std: 0.0351 | Applying Scale: 0.5
Scaled Preprocessing Accuracy: 18.75%
New Feature Std: 0.0581


In [39]:
import torch
import torch.nn as nn

class LandmarkTransformerUpgraded(nn.Module):
    def __init__(self, num_classes=250, input_dim=4887, model_dim=256, num_heads=8, num_layers=4, dropout=0.1):
        super().__init__()
        self.input_projection = nn.Linear(input_dim, model_dim)
        self.pos_encoder = nn.Parameter(torch.randn(1, 30, model_dim))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=model_dim,
            nhead=num_heads,
            dim_feedforward=model_dim * 4,
            dropout=dropout,
            batch_first=True,
            norm_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.classifier = nn.Linear(model_dim, num_classes)

    def forward(self, x):
        # x shape: (batch_size, 30, 4887)
        x = self.input_projection(x) + self.pos_encoder
        x = self.transformer(x).mean(dim=1)
        return self.classifier(x)

print("✅ LandmarkTransformerUpgraded architecture defined.")

✅ LandmarkTransformerUpgraded architecture defined.


In [43]:
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau

# 1. Hardware Detection
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
gpu_count = torch.cuda.device_count()
print(f'Using device: {device}')
if gpu_count > 0:
    print(f'GPU Name: {torch.cuda.get_device_name(0)}')

# 2. Model Initialization
model = LandmarkTransformerUpgraded().to(device)

# Multi-GPU support (Dual T4 support for Kaggle)
if gpu_count > 1:
    print(f'🚀 Multi-GPU detected! Using {gpu_count} cards.')
    model = nn.DataParallel(model)

# 3. Optimizer, Criterion, and Scheduler
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)
criterion = nn.CrossEntropyLoss()
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

# 4. Load weights from Drive if they exist
weights_path = '/content/drive/MyDrive/asl_model_upgraded.pth'
if os.path.exists(weights_path):
    try:
        state = torch.load(weights_path, map_location=device)
        # If it's a full checkpoint, extract state_dict
        model.load_state_dict(state if 'input_projection.weight' in state else state['model_state_dict'])
        print('✅ Successfully loaded weights from Drive.')
    except Exception as e:
        print(f'⚠️ Error loading weights: {e}')
else:
    print('ℹ️ No checkpoint found. Starting with fresh weights.')

print(f'\nModel is ready for training on {device}.')

Using device: cpu


/tmp/ipykernel_4755/1277387784.py:18: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)


✅ Successfully loaded weights from Drive.

Model is ready for training on cpu.


In [45]:
import os
import torch

# Let's check for both possible filenames
checkpoint_options = [
    '/content/drive/MyDrive/asl_model_upgraded_best.pth',
    '/content/drive/MyDrive/asl_model_upgraded.pth'
]

selected_checkpoint = None
for path in checkpoint_options:
    if os.path.exists(path):
        size = os.path.getsize(path) / (1024**2)
        print(f'✅ Found: {path} ({size:.2f} MB)')
        if selected_checkpoint is None: # Prioritize the first one found (best)
            selected_checkpoint = path
    else:
        print(f'❌ Not found: {path}')

if selected_checkpoint:
    print(f'\nAttempting to load: {selected_checkpoint}')
    state = torch.load(selected_checkpoint, map_location=device)

    # Handle if it's a full checkpoint dict or just a state_dict
    if isinstance(state, dict) and 'model_state_dict' in state:
        model.load_state_dict(state['model_state_dict'])
        start_epoch = state.get('epoch', 50)
    else:
        model.load_state_dict(state)
        start_epoch = 50 # Default if no metadata found

    print(f'✅ Model successfully loaded. Ready to resume from Epoch {start_epoch}.')
else:
    print('❌ Critical Error: No usable checkpoint found in Drive.')

✅ Found: /content/drive/MyDrive/asl_model_upgraded_best.pth (51.36 MB)
✅ Found: /content/drive/MyDrive/asl_model_upgraded.pth (17.12 MB)

Attempting to load: /content/drive/MyDrive/asl_model_upgraded_best.pth
✅ Model successfully loaded. Ready to resume from Epoch 42.


In [46]:
from torch.utils.data import DataLoader

# Define paths for the converted data on Drive
drive_files = [
    '/content/drive/MyDrive/asl_train_data.jsonl',
    '/content/drive/MyDrive/asl_train_data_part2.jsonl'
]
phrase_map_path = '/content/drive/MyDrive/phrase_map.json'

# 1. Verify files exist before initializing
missing_files = [f for f in drive_files + [phrase_map_path] if not os.path.exists(f)]
if missing_files:
    print(f"⚠️ Warning: The following files are missing: {missing_files}")
    print("Ensure Google Drive is mounted and files are in the root of MyDrive.")
else:
    # 2. Initialize the Dataset and Loader
    # Using ASLUpgradedDataset defined previously in the notebook
    asl_dataset = ASLUpgradedDataset(drive_files, phrase_map_path)

    # Assigning to asl_loader (and train_loader for compatibility with your diagnostic)
    asl_loader = DataLoader(asl_dataset, batch_size=128, num_workers=2, pin_memory=True)
    train_loader = asl_loader

    print("✅ asl_loader (train_loader) initialized and ready.")

    # 3. Quick Test
    try:
        for frames, labels in asl_loader:
            print(f"Sample batch received: frames {frames.shape}, labels {labels.shape}")
            break
    except Exception as e:
        print(f"❌ Error testing loader: {e}")

NameError: name 'ASLUpgradedDataset' is not defined

In [44]:
import time

print(f"🚀 Starting training loop from Epoch {start_epoch} on {gpu_count} GPUs...")

# Ensure model is in training mode
model.train()

for epoch in range(start_epoch, END_EPOCH):
    epoch_start_time = time.time()
    running_loss = 0.0
    running_acc = 0.0
    batch_count = 0

    # Use tqdm for progress bar if available
    pbar = tqdm(asl_loader, desc=f'Epoch {epoch}') if 'tqdm' in globals() and tqdm else asl_loader

    for batch_idx, batch in enumerate(pbar):
        # 1. Unpack and prepare data
        inputs, labels = unpack_batch(batch)
        inputs = move_to_device(inputs, device)
        labels = prepare_labels(labels, device)

        # Standardize inputs (handle NaNs)
        inputs = torch.nan_to_num(inputs, nan=0.0)

        # 2. Forward pass with AMP
        optimizer.zero_grad(set_to_none=True)
        with autocast_context():
            outputs = model(inputs)
            loss = criterion(outputs, labels)

        # 3. Backward pass and Optimization
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
        scaler.step(optimizer)
        scaler.update()

        # 4. Metrics
        acc = (outputs.argmax(dim=1) == labels).sum().item() / labels.size(0)
        running_loss += loss.item()
        running_acc += acc
        batch_count += 1

        if (batch_idx + 1) % LOG_EVERY == 0:
            avg_l = running_loss / batch_count
            avg_a = running_acc / batch_count
            if pbar == asl_loader:
                print(f'Epoch [{epoch}] Step [{batch_idx+1}] Loss: {avg_l:.4f} Acc: {avg_a:.4f}')
            else:
                pbar.set_postfix({'loss': f'{avg_l:.3f}', 'acc': f'{avg_a:.3f}'})

    # --- End of Epoch Logic ---
    epoch_loss = running_loss / batch_count
    epoch_acc = running_acc / batch_count

    print(f'\n✨ Epoch {epoch} Summary | Loss: {epoch_loss:.4f} | Acc: {epoch_acc:.4f} | Time: {time.time()-epoch_start_time:.1f}s')

    # Update Scheduler
    scheduler.step(epoch_loss)

    # Save Latest Checkpoint
    save_checkpoint(
        LATEST_PATH, epoch, model, optimizer, scheduler, scaler,
        best_val_acc, best_val_loss, train_loss=epoch_loss, train_acc=epoch_acc
    )

    # Save Best Model
    if epoch_acc > best_val_acc:
        best_val_acc = epoch_acc
        save_checkpoint(
            BEST_PATH, epoch, model, optimizer, scheduler, scaler,
            best_val_acc, best_val_loss, train_acc=epoch_acc
        )

    # Special 45% Milestone Check
    if epoch_acc >= TARGET_VAL_ACC:
        print(f"🎯 MILESTONE REACHED: {epoch_acc:.2%} accuracy!")
        save_checkpoint(
            SUBMISSION_45_PATH, epoch, model, optimizer, scheduler, scaler,
            epoch_acc, best_val_loss
        )
        if STOP_WHEN_45_REACHED:
            print("Stopping training as target accuracy was reached.")
            break

print("\n✅ Training Session Complete.")

NameError: name 'start_epoch' is not defined

In [40]:
import torch

# Check if GPU is available
if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f'✅ GPU detected: {torch.cuda.get_device_name(0)}')
else:
    device = torch.device('cpu')
    print('❌ GPU not detected. Using CPU. Please check your Runtime settings.')

# Moving the model to the selected device
# model = LandmarkTransformerUpgraded().to(device)
# print(f'Model moved to: {device}')

❌ GPU not detected. Using CPU. Please check your Runtime settings.


In [41]:
import torch
import torch.nn as nn

# Check GPU details
gpu_count = torch.cuda.device_count()
print(f'Total GPUs available: {gpu_count}')

if gpu_count > 0:
    for i in range(gpu_count):
        print(f'GPU {i}: {torch.cuda.get_device_name(i)}')

    device = torch.device('cuda')
    # Initialize model
    model = LandmarkTransformerUpgraded().to(device)

    # Enable Multi-GPU support if more than 1 GPU is found
    if gpu_count > 1:
        print(f'🚀 Using {gpu_count} GPUs with DataParallel')
        model = nn.DataParallel(model)

    print(f'✅ Model moved to {device}')
else:
    device = torch.device('cpu')
    print('❌ No GPU detected. Please check Runtime settings.')

Total GPUs available: 0
❌ No GPU detected. Please check Runtime settings.


In [42]:
import torch
import torch.nn as nn

# 1. Identify Hardware Allocation
gpu_count = torch.cuda.device_count()
print(f'Detected GPUs: {gpu_count}')

if gpu_count > 0:
    for i in range(gpu_count):
        print(f' - GPU {i}: {torch.cuda.get_device_name(i)}')

    device = torch.device('cuda')
    # Initialize model architecture
    model = LandmarkTransformerUpgraded().to(device)

    # 2. Configure for Multi-GPU (Dual T4 support)
    if gpu_count > 1:
        print(f'🚀 Multi-GPU detected! Initializing DataParallel for {gpu_count} cards.')
        model = nn.DataParallel(model)
    else:
        print(f'✅ Single GPU ({torch.cuda.get_device_name(0)}) detected. Optimizing for single-stream.')

    print(f'\nModel is ready on: {device}')
else:
    print('❌ No GPU detected. Please go to Runtime -> Change runtime type.')

Detected GPUs: 0
❌ No GPU detected. Please go to Runtime -> Change runtime type.


### 🚀 Kaggle Production Training Script
This cell contains the logic to resume training directly on Kaggle. It assumes you have attached:
1. **Competition Data:** `asl-signs`
2. **Your Weights:** `asl-model-weights` (basseyjohn)

### 🛠️ Kaggle Path & Data Fix
Since the `.jsonl` files were not uploaded, we will configure the notebook to read directly from the Kaggle competition data (`/kaggle/input/asl-signs`).

In [49]:
import os
import pandas as pd
import torch
from torch.utils.data import DataLoader

# 1. Update Global Paths for Kaggle
KAGGLE_DATA_ROOT = '/kaggle/input/asl-signs'
TRAIN_CSV_PATH = os.path.join(KAGGLE_DATA_ROOT, 'train.csv')

# 2. Define a simple Kaggle-native Dataset class (reads Parquet)
class ASLKaggleDataset(torch.utils.data.Dataset):
    def __init__(self, csv_path, root_dir, phrase_map_path):
        self.df = pd.read_csv(csv_path)
        self.root_dir = root_dir
        with open(phrase_map_path, 'r') as f:
            self.phrase_map = json.load(f)['labels']
        self.label_map = {item['label']: i for i, item in enumerate(self.phrase_map)}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = os.path.join(self.root_dir, row['path'])
        # Basic loader logic: reads parquet and pads/trims to 30 frames
        df_feat = pd.read_parquet(path)
        # Simplify to x,y,z landmarks
        frames = []
        for _, frame_df in df_feat.groupby('frame'):
            frames.append(frame_df[['x', 'y', 'z']].values.flatten())
            if len(frames) >= 30: break
        while len(frames) < 30:
            frames.append([0.0] * 1629)

        feat = torch.tensor(frames[:30], dtype=torch.float32)
        label = self.label_map[row['sign']]
        return feat, label

# 3. Initialize the Loader
if os.path.exists(TRAIN_CSV_PATH):
    asl_dataset = ASLKaggleDataset(TRAIN_CSV_PATH, KAGGLE_DATA_ROOT, '/kaggle/input/asl-model-weights/phrase_map.json')
    asl_loader = DataLoader(asl_dataset, batch_size=64, shuffle=True, num_workers=2)
    train_loader = asl_loader
    print(f"✅ train_loader initialized with {len(asl_dataset)} samples.")
else:
    print("❌ Kaggle Competition data not found. Please add 'asl-signs' to your notebook.")

❌ Kaggle Competition data not found. Please add 'asl-signs' to your notebook.


In [54]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import os
import json

class ASLKaggleParquetDataset(Dataset):
    def __init__(self, csv_path, root_dir, phrase_map_path, seq_len=30):
        self.df = pd.read_csv(csv_path)
        self.root_dir = root_dir
        self.seq_len = seq_len

        with open(phrase_map_path, 'r') as f:
            phrase_data = json.load(f)
            # Support both {labels: [...]} and direct list formats
            labels = phrase_data['labels'] if isinstance(phrase_data, dict) else phrase_data
            self.label_map = {item['label']: i for i, item in enumerate(labels)}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        file_path = os.path.join(self.root_dir, row['path'])

        # Load parquet file
        data = pd.read_parquet(file_path)

        # Group by frame and extract x, y, z coordinates
        frames = []
        for _, frame_df in data.groupby('frame'):
            coords = frame_df[['x', 'y', 'z']].values.flatten()
            if len(coords) == 1629: # 543 landmarks * 3 coords
                frames.append(coords)
            if len(frames) >= self.seq_len: break

        # Padding/Trimming
        if len(frames) < self.seq_len:
            padding = [np.zeros(1629) for _ in range(self.seq_len - len(frames))]
            frames.extend(padding)

        frames = np.array(frames[:self.seq_len], dtype=np.float32)
        label = self.label_map[row['sign']]

        return torch.tensor(frames), torch.tensor(label)

# Initialize for Kaggle
KAGGLE_CSV = '/kaggle/input/asl-signs/train.csv'
KAGGLE_ROOT = '/kaggle/input/asl-signs'
KAGGLE_MAP = '/kaggle/input/asl-model-weights/phrase_map.json'

if os.path.exists(KAGGLE_CSV):
    asl_dataset = ASLKaggleParquetDataset(KAGGLE_CSV, KAGGLE_ROOT, KAGGLE_MAP)
    asl_loader = DataLoader(asl_dataset, batch_size=128, shuffle=True, num_workers=4, pin_memory=True)
    print(f"✅ Kaggle Parquet Loader ready: {len(asl_dataset)} samples found.")
else:
    print("⚠️ Kaggle datasets not detected at expected paths.")

⚠️ Kaggle datasets not detected at expected paths.


In [50]:
import os

# The official competition dataset name is 'asl-signs'
COMPETITION_PATH = '/kaggle/input/asl-signs'

if os.path.exists(COMPETITION_PATH):
    print(f"✅ Competition data found at: {COMPETITION_PATH}")
    print("Files available:", os.listdir(COMPETITION_PATH))
else:
    print(f"❌ Competition data NOT found at {COMPETITION_PATH}")
    print("Please ensure you have added the 'asl-signs' dataset to your Kaggle notebook via the '+ Add Data' button.")

❌ Competition data NOT found at /kaggle/input/asl-signs
Please ensure you have added the 'asl-signs' dataset to your Kaggle notebook via the '+ Add Data' button.


In [51]:
import os

print('--- Available Datasets in /kaggle/input ---')
if os.path.exists('/kaggle/input'):
    datasets = os.listdir('/kaggle/input')
    for ds in datasets:
        path = os.path.join('/kaggle/input', ds)
        print(f'📁 {ds}')
        # List first 3 items in each to verify content
        try:
            content = os.listdir(path)
            print(f'   Samples: {content[:3]}')
        except NotADirectoryError:
            print('   (File)')
else:
    print('❌ /kaggle/input not found. Are you running this on Kaggle?')

# Specific check for the competition root
print('\n--- Official Competition Data Check ---')
comp_path = '/kaggle/input/asl-signs'
if os.path.exists(comp_path):
    print(f'✅ Found official asl-signs at {comp_path}')
    if 'train.csv' in os.listdir(comp_path):
        print('✅ train.csv is present.')
else:
    print('❌ Official asl-signs dataset is not attached.')

--- Available Datasets in /kaggle/input ---

--- Official Competition Data Check ---
❌ Official asl-signs dataset is not attached.


### 🎯 Final Training Resumption on Kaggle
Now that we have verified the dataset paths, we will resume training from **Epoch 43** to reach our goal of **80 epochs**.

In [52]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import os

# 1. Configuration
START_EPOCH = 43
END_EPOCH = 80
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 2. Initialize Model
model = LandmarkTransformerUpgraded().to(DEVICE)

# 3. Load Checkpoint from Kaggle Input
WEIGHTS_PATH = '/kaggle/input/asl-model-weights/asl_model_upgraded_best.pth'
if os.path.exists(WEIGHTS_PATH):
    checkpoint = torch.load(WEIGHTS_PATH, map_location=DEVICE)
    # Handle both state_dict and full checkpoint objects
    if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
        model.load_state_dict(checkpoint['model_state_dict'])
        print(f"✅ Full checkpoint loaded. Previous Accuracy: {checkpoint.get('accuracy', 'N/A')}")
    else:
        model.load_state_dict(checkpoint)
        print("✅ Model state_dict loaded successfully.")
else:
    print("❌ Error: Model weights not found in /kaggle/input/asl-model-weights/")

# 4. Setup Optimizer & Criterion
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)
criterion = nn.CrossEntropyLoss()
scaler = torch.amp.GradScaler('cuda', enabled=torch.cuda.is_available())

❌ Error: Model weights not found in /kaggle/input/asl-model-weights/


/tmp/ipykernel_4755/1277387784.py:18: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)


In [53]:
from tqdm.auto import tqdm

print(f"🚀 Starting training from Epoch {START_EPOCH} to {END_EPOCH}...")

model.train()
for epoch in range(START_EPOCH, END_EPOCH):
    running_loss, running_acc = 0.0, 0.0
    pbar = tqdm(asl_loader, desc=f'Epoch {epoch}')

    for batch_idx, (frames, labels) in enumerate(pbar):
        frames, labels = frames.to(DEVICE), labels.to(DEVICE)
        frames = torch.nan_to_num(frames, nan=0.0)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()):
            outputs = model(frames)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()

        # Metrics
        acc = (outputs.argmax(dim=1) == labels).sum().item() / labels.size(0)
        running_loss += loss.item()
        running_acc += acc

        pbar.set_postfix({'loss': f'{running_loss/(batch_idx+1):.4f}', 'acc': f'{running_acc/(batch_idx+1):.4f}'})

    # Save Checkpoint to /kaggle/working
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'accuracy': running_acc / len(asl_loader)
    }, f'/kaggle/working/asl_model_epoch_{epoch}.pth')

print("✅ Training process completed.")

🚀 Starting training from Epoch 43 to 80...


NameError: name 'asl_loader' is not defined

In [38]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

# 1. Path Configuration for Kaggle
DATA_ROOT = '/kaggle/input/asl-signs'
WEIGHTS_DIR = '/kaggle/input/asl-model-weights'

# Determine specific file paths
BEST_WEIGHTS = os.path.join(WEIGHTS_DIR, 'asl_model_upgraded_best.pth')
PHRASE_MAP = os.path.join(WEIGHTS_DIR, 'phrase_map.json')
SAVE_PATH = '/kaggle/working/asl_model_kaggle_refined.pth'

print(f"Weights Found: {os.path.exists(BEST_WEIGHTS)}")
print(f"Data Root Found: {os.path.exists(DATA_ROOT)}")

# 2. Hardware Acceleration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# 3. Model Initialization (LandmarkTransformerUpgraded)
# Note: Ensure the LandmarkTransformerUpgraded class is defined in your Kaggle notebook
model = LandmarkTransformerUpgraded().to(device)

# 4. Resume from Checkpoint
if os.path.exists(BEST_WEIGHTS):
    checkpoint = torch.load(BEST_WEIGHTS, map_location=device)
    # Logic to handle if it's a state_dict or a full metadata dict
    model.load_state_dict(checkpoint if 'input_projection.weight' in checkpoint else checkpoint['model_state_dict'])
    start_epoch = checkpoint.get('epoch', 48) + 1
    print(f"✅ Successfully loaded best weights. Resuming from Epoch {start_epoch}")
else:
    print("❌ Weights not found. Please attach the 'asl-model-weights' dataset in the sidebar.")

Weights Found: False
Data Root Found: False
Using device: cpu


NameError: name 'LandmarkTransformerUpgraded' is not defined

In [33]:
import os
# Verify files in the upload directory
upload_dir = '/content/kaggle_upload'
files = os.listdir(upload_dir)
print(f'Files prepared for upload: {files}')

# Check if there are any other model files on Drive that were missed
print('\n--- Checking for other weights on Drive ---')
!ls -lh /content/drive/MyDrive/asl_model_*

Files prepared for upload: ['asl_model_upgraded.pth', 'dataset-metadata.json', 'phrase_map.json']

--- Checking for other weights on Drive ---
-rw------- 1 root root 42M May  6 17:29 /content/drive/MyDrive/asl_model_best.pth
-rw------- 1 root root 42M May  6 11:04 /content/drive/MyDrive/asl_model_checkpoint.pth
-rw------- 1 root root 52M May 11 07:58 /content/drive/MyDrive/asl_model_upgraded_best.pth
-rw------- 1 root root 18M May 13 19:26 /content/drive/MyDrive/asl_model_upgraded.pth


In [48]:
import os
import shutil

# Option: Include the screenshot if you want it as a reference in the dataset
# Uncomment the lines below if you want to add it
# screenshot_path = '/content/Screenshot 2026-05-13 at 20.49.19.png'
# if os.path.exists(screenshot_path):
#     shutil.copy2(screenshot_path, '/content/kaggle_upload/')

# Final Audit of the Upload Folder
print("--- Final Kaggle Dataset Bundle Contents ---")
upload_contents = os.listdir('/content/kaggle_upload')
for item in upload_contents:
    size_mb = os.path.getsize(f'/content/kaggle_upload/{item}') / (1024**2)
    print(f"📄 {item:<30} | {size_mb:>7.2f} MB")

print(f"\nTotal items: {len(upload_contents)}")

--- Final Kaggle Dataset Bundle Contents ---
📄 asl_model_upgraded_best.pth    |   51.36 MB
📄 asl_model_upgraded.pth         |   17.12 MB
📄 asl_model_best.pth             |   41.81 MB
📄 dataset-metadata.json          |    0.00 MB
📄 phrase_map.json                |    0.00 MB

Total items: 5


In [25]:
import kagglehub
print(f'kagglehub version: {kagglehub.__version__}')

kagglehub version: 1.0.0
